# The Refinery Walkthrough — a 10-step statistical pipeline on crypto tick data

A prototype for the production Glue jobs in this project. It applies the ten-step refinery to
Binance spot trades for BTCUSDT, ETHUSDT and SOLUSDT (January 2025) and demonstrates the one
structural idea the framework is built on:

> **Steps 1–3 are a shared entryway. Steps 4–10 are three different refineries.**

Steps 1–3 are *path-blind* — they run identically no matter what comes later, because they
finish before the pipeline knows what kind of target it is handling. The moment data exits
Step 3 the pipeline reads the geometry of `y` and forks into three path-isolated sub-refineries
that share step *numbers* and almost nothing else. On one path three of the seven steps are
**forbidden**, and those bans are the most interesting thing in the framework: each one exists
because a downstream engine would break if the step ran.

| Part | Contents | Production counterpart |
| --- | --- | --- |
| 1 | Steps 1–3, the shared entryway | `glue-ingest-bars.py` |
| 2 | Path 1 — continuous, next-bar log return | `glue-refinery-path1.py` |
| 3 | Path 2 — categorical, direction `k=3` | `glue-refinery-path2.py` |
| 4 | Path 3 — bandit, Thompson Sampling | `glue-refinery-path3.py` |

### How to run it

Launch Jupyter **from the project root** (`crypto-ticks-refinery-glue-dynamo/`) — every path
below is repo-relative. Needs a Java 11 or 17 JDK and `pyspark==3.5.5`; Spark does not support
Java 21+.

Everything runs on the **committed two-hour sample** in `data/sample/`, so Run All works
straight after a clone with no download and no AWS account. The setup cell below builds the
bars by invoking `glue-ingest-bars.py` itself, so the notebook and the production job cannot
drift apart.

### Reading the numbers

Every figure printed below is computed by the cell that prints it, on the two-hour sample. The
sample is a quiet window (00:00–02:00 UTC on New Year's Day), so its numbers are **not** the
month's. Where a full-month figure matters it is quoted as prose and labelled as measured
separately by `glue-ingest-bars.py` over all 340,971,834 ticks — never presented as something
this notebook computed.

| | this notebook (2 h sample) | full month, measured separately |
| --- | --- | --- |
| ticks | 356,201 | 340,971,834 |
| bars at 5s | 4,320 | 1,607,040 |
| empty bars | 8 | 613 |

In [1]:
# Setup. Nothing here is framework content -- it builds the session and the bars the rest of the
# notebook reads.
import importlib.util
import subprocess
import sys
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as func

# Repo-relative throughout, so nothing this notebook prints carries a local absolute path into
# the committed outputs. It is why Jupyter has to be launched from the project root.
SAMPLE_DIR = "data/sample"
BARS_PATH = "_localrun/bars"
JOB = "glue-ingest-bars.py"

assert Path(SAMPLE_DIR).exists() and Path(JOB).exists(), (
    f"run Jupyter from the project root -- {SAMPLE_DIR!r} not found from {Path.cwd()}")

# verdict() is imported from the production job rather than redefined, so Part 1 demonstrates the
# same function the job ships. importlib is needed because the hyphen in the filename makes
# "import glue-ingest-bars" a syntax error.
_spec = importlib.util.spec_from_file_location("glue_ingest_bars", JOB)
_job = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_job)
verdict = _job.verdict

# The job logs verdicts through logging, which defaults to stderr -- Jupyter renders that as a red
# error stream. Point the job's logger at stdout so the APPLIES / N/A / BANNED ledger appears
# inline, in order, as ordinary cell output.
import logging
_job.LOG.handlers.clear()
_job.LOG.propagate = False
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("%(message)s"))
_job.LOG.addHandler(_handler)
_job.LOG.setLevel(logging.INFO)

spark = (SparkSession.builder
         .appName("RefineryWalkthrough")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
# Not optional and not cosmetic: this box defaults to Africa/Johannesburg, under which every
# hour() and to_date() below would shift two hours and the last bar of January would land in
# February. The job pins this for the same reason.
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Spark", spark.version, "| tz", spark.conf.get("spark.sql.session.timeZone"))

Spark 3.5.5 | tz UTC


In [2]:
# Build the bars by running the production job on the committed sample, unless they already exist.
# _localrun/ is gitignored, so this is what makes Run All work on a fresh clone -- and it means the
# frame every later part reads was produced by glue-ingest-bars.py, not by a copy of its logic.
if not Path(BARS_PATH).exists():
    cmd = [sys.executable, JOB, "--local", "--shuffle-partitions", "8",
           "--input", "data/sample", "--bars-output", "_localrun/bars",
           "--bar-interval", "5s",
           "--calendar-start", "2025-01-01T00:00:00",
           "--calendar-end", "2025-01-01T02:00:00"]
    print("building bars:", " ".join(cmd[1:]))
    subprocess.run(cmd, check=True)

bars = spark.read.parquet(BARS_PATH).cache()
print(f"{bars.count():,} bars x {len(bars.columns)} columns")
bars.printSchema()

building bars: glue-ingest-bars.py --local --shuffle-partitions 8 --input data/sample --bars-output _localrun/bars --bar-interval 5s --calendar-start 2025-01-01T00:00:00 --calendar-end 2025-01-01T02:00:00


4,320 bars x 18 columns
root
 |-- symbol: string (nullable = true)
 |-- bar_us: long (nullable = true)
 |-- bar_open_time_utc: timestamp (nullable = true)
 |-- open: decimal(18,8) (nullable = true)
 |-- high: decimal(18,8) (nullable = true)
 |-- low: decimal(18,8) (nullable = true)
 |-- close: decimal(18,8) (nullable = true)
 |-- volume: decimal(28,8) (nullable = true)
 |-- quote_volume: decimal(28,8) (nullable = true)
 |-- n_ticks: long (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- taker_buy_qty: decimal(28,8) (nullable = true)
 |-- taker_buy_quote_qty: decimal(28,8) (nullable = true)
 |-- all_best_match: boolean (nullable = true)
 |-- is_missing_bar: integer (nullable = true)
 |-- is_first_bar: integer (nullable = true)
 |-- is_last_bar: integer (nullable = true)



# Part 1 — Steps 1-3: The Shared Entryway

Steps 1, 2, 2.5 and 3 are **path-blind**. They run once, identically, for every downstream
path, because they finish before the router has anything to read: the fork branches on the
geometry of a target variable `y`, and at tick grain no bar-level `y` exists yet.

This part re-derives the entryway from the committed two-hour sample rather than trusting the
parquet the setup cell loaded. It is the section that mirrors `glue-ingest-bars.py` operation
for operation, in the same order, so the two can be read side by side:

| notebook stage | job function |
| --- | --- |
| 1. First Look | `SCHEMA`, `step1_ingest` (the read) |
| 2. Step 1 — Ingestion / Deduplication | `step1_ingest`, `step1_dedup_check` |
| 3. Step 2 — Structural Syntax Normalisation | `step2_normalise` |
| 4. Step 2.5 — Row Granularity | `step2_5_aggregate` |
| 5. Step 3 — Missingness Indicator Generation | `step3_missingness`, `assert_bar_invariants` |

Every stage keeps the notebook's find/handle shape and reports through the three-state
`verdict()` the setup cell imported from the job: `APPLIES`, `N/A`, `BANNED`. An `N/A` here is
always backed by a number measured on this run.

The frame this part produces is called `bars` — the one name in the notebook without a section
prefix. The setup cell's `bars` (read from `_localrun/bars`) is there only so the later
path sections can be run on their own; the last cell of this part checks the re-derivation
against it row for row and then replaces it.

## 1. First Look

The Binance monthly trade archives are **headerless** CSV: seven unnamed columns whose meaning
is fixed by position alone. There is nothing to profile until a schema is supplied, so the
schema comes first — it is Step 2's contract, applied at Step 1's read because a headerless
file cannot be read any other way.

In [3]:
from datetime import datetime, timezone
from decimal import Decimal

from pyspark.sql import Window
from pyspark.sql.types import (BooleanType, DecimalType, LongType, StructField,
                               StructType)

# Paths are relative to the repository root, which is where this notebook lives and where
# glue-ingest-bars.py's documented local acceptance run is invoked from.
ENT_SAMPLE_DIR = "data/sample"
ENT_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
ENT_MONTH = "2025-01"
ENT_FILE_TEMPLATE = "{symbol}-trades-{month}.csv*"   # the * matches .csv and the sample's .csv.gz

ENT_INTERVAL_LABEL = "5s"
ENT_INTERVAL_US = 5_000_000

# Every value on the wire is an exact 8-decimal-place decimal string, so decimal(18,8) is the
# wire format rather than an approximation of it. The double alternative is measured in stage 3b.
ENT_MONEY = DecimalType(18, 8)

# The positional contract, and the entirety of Step 2's "sanitise the headers" on this source:
# there are no headers to sanitise, so the step is the SUPPLY of names, not a rename. Column
# ORDER is load-bearing and this is the only place it is written down. inferSchema is not used:
# it costs a second read pass and it would silently re-type a column when the source changes.
ENT_SCHEMA = StructType([
    StructField("trade_id", LongType(), True),        # not Integer: BTC max is 4,495,881,900
    StructField("price", ENT_MONEY, True),
    StructField("qty", ENT_MONEY, True),
    StructField("quote_qty", ENT_MONEY, True),
    # Named _us, never "time". A column called "time" invites the /1000 "convert from millis"
    # reflex, and this one is epoch MICROseconds. See the next cell.
    StructField("event_time_us", LongType(), True),
    # The literal strings in the file are Python-cased "True"/"False", which Spark's CSV reader
    # parses into BooleanType unaided -- do NOT add a when(col == "true", ...) mapping.
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True),
])
ENT_DATA_COLS = [f.name for f in ENT_SCHEMA.fields]

for i, f in enumerate(ENT_SCHEMA.fields):
    print(f"  col {i}  {f.name:<15} {f.dataType.simpleString()}")

  col 0  trade_id        bigint
  col 1  price           decimal(18,8)
  col 2  qty             decimal(18,8)
  col 3  quote_qty       decimal(18,8)
  col 4  event_time_us   bigint
  col 5  is_buyer_maker  boolean
  col 6  is_best_match   boolean


In [4]:
# One symbol first, unmerged: the raw frame exactly as it leaves the reader.
ent_btc_path = f"{ENT_SAMPLE_DIR}/{ENT_FILE_TEMPLATE.format(symbol='BTCUSDT', month=ENT_MONTH)}"

# No .option("header", ...). That is not an omission -- header=true would consume the first
# TRADE as a header row and lose it silently, with no null, no error and no shape change to
# give it away. Priced below rather than asserted in prose.
ent_raw_btc = spark.read.schema(ENT_SCHEMA).csv(ent_btc_path)
ent_raw_btc.show(5, truncate=False)
ent_raw_btc.printSchema()

ent_btc_rows = ent_raw_btc.count()
ent_hdr_rows = (spark.read.schema(ENT_SCHEMA).option("header", "true")
                .csv(ent_btc_path).count())
print(f"headerless read : {ent_btc_rows:,} rows")
print(f"header=true read: {ent_hdr_rows:,} rows  -> {ent_btc_rows - ent_hdr_rows} trade(s) eaten")

+----------+--------------+----------+------------+----------------+--------------+-------------+
|trade_id  |price         |qty       |quote_qty   |event_time_us   |is_buyer_maker|is_best_match|
+----------+--------------+----------+------------+----------------+--------------+-------------+
|4359935386|93576.00000000|0.00136000|127.26336000|1735689600010866|true          |true         |
|4359935387|93576.00000000|0.00212000|198.38112000|1735689600074095|true          |true         |
|4359935388|93576.00000000|0.00154000|144.10704000|1735689600074095|true          |true         |
|4359935389|93576.00000000|0.00241000|225.51816000|1735689600091046|true          |true         |
|4359935390|93576.00000000|0.00545000|509.98920000|1735689600091046|true          |true         |
+----------+--------------+----------+------------+----------------+--------------+-------------+
only showing top 5 rows

root
 |-- trade_id: long (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- q

headerless read : 173,468 rows
header=true read: 173,467 rows  -> 1 trade(s) eaten


In [5]:
# THE MICROSECOND TRAP. event_time_us is a 16-digit epoch MICROsecond count. Both of the
# habitual millisecond readings return a valid-looking timestamp roughly 55,000 years out, with
# no error and no null. Cast to STRING inside Spark: the millis reading is year 56971, which
# is outside Python's datetime range and cannot be collected as an object.
ent_probe_us = ent_raw_btc.select("event_time_us").first()[0]
ent_readings = spark.sql(f"""
    SELECT CAST(timestamp_micros({ent_probe_us})        AS STRING) AS micros,
           CAST(timestamp_millis({ent_probe_us})        AS STRING) AS millis,
           CAST(CAST({ent_probe_us} / 1000 AS TIMESTAMP) AS STRING) AS divided
""").first()

print(f"raw event_time_us            {ent_probe_us}   ({len(str(ent_probe_us))} digits)")
print(f"timestamp_micros(v)       -> {ent_readings['micros']}      <- correct")
print(f"timestamp_millis(v)       -> {ent_readings['millis']}")
print(f"cast(v / 1000 as ts)      -> {ent_readings['divided']}")

# timestamp_micros is SQL-only in Spark 3.3.0 (Glue 4.0) -- the Python wrapper lands in 3.5.0 --
# so the job reaches it through expr(). Same rule as pmod and bool_and further down.
print(f"\nsession: Spark {spark.version}, tz "
      f"{spark.conf.get('spark.sql.session.timeZone')}")

raw event_time_us            1735689600010866   (16 digits)
timestamp_micros(v)       -> 2025-01-01 00:00:00.010866      <- correct
timestamp_millis(v)       -> +56971-10-25 00:00:10.866
cast(v / 1000 as ts)      -> +56971-10-25 00:00:10.86592

session: Spark 3.5.5, tz UTC


## 2. Step 1 — Stateful Relational Ingestion / Deduplication

### a. Find duplicate keys

Three files become one frame. The merge key is `(symbol, trade_id)`, not `trade_id`: a Binance
trade id is a per-symbol sequence, and the three ranges only happen to be disjoint this month.
Two independent checks run — a duplicate count on the key, and a contiguity check on the id
span, which catches the truncated download that duplication cannot see.

In [6]:
# The union carries a discriminator column. Without it this is 341M rows in which a $93,576 BTC
# print and a $187 SOL print are indistinguishable, and every downstream mean averages three
# price scales into a number that describes nothing.
ent_frames = [spark.read.schema(ENT_SCHEMA)
                   .csv(f"{ENT_SAMPLE_DIR}/{ENT_FILE_TEMPLATE.format(symbol=s, month=ENT_MONTH)}")
                   .withColumn("symbol", func.lit(s))
              for s in ENT_SYMBOLS]

ent_ticks = ent_frames[0]
for frame in ent_frames[1:]:
    ent_ticks = ent_ticks.unionByName(frame)

# The sample is 356k rows and is scanned a dozen times below. The job deliberately does NOT
# cache here -- at 341M CSV rows the cache would spill and cost more than the re-reads.
ent_ticks = ent_ticks.cache()

verdict(True, f"Step 1 ingest: {len(ENT_SYMBOLS)} files -> one frame keyed (symbol, trade_id), "
              f"symbols {', '.join(ENT_SYMBOLS)}")

APPLIES  -- Step 1 ingest: 3 files -> one frame keyed (symbol, trade_id), symbols BTCUSDT, ETHUSDT, SOLUSDT


True

In [7]:
# The duplicate count: one shuffle over a two-column projection, kept even though it is expected
# to read N/A. An N/A the job did not measure is a lie, and Step 1's whole claim is that the
# merge is deterministic.
ent_dupes = (ent_ticks.select("symbol", "trade_id")
             .groupBy("symbol", "trade_id").count()
             .filter(func.col("count") > 1)
             .count())

# Contiguity is one 3-row aggregate and catches what dedup structurally cannot: a truncated
# download. Alone it cannot separate "no duplicates" from "equal numbers of duplicates and
# gaps", which is why both checks exist.
ent_profile = (ent_ticks.groupBy("symbol")
               .agg(func.count("*").alias("rows"),
                    func.min("trade_id").alias("tid_min"),
                    func.max("trade_id").alias("tid_max"))
               .orderBy("symbol").collect())

ent_raw_rows = sum(row["rows"] for row in ent_profile)
for row in ent_profile:
    span = row["tid_max"] - row["tid_min"] + 1
    print(f"  {row['symbol']:<8} {row['rows']:>7,} rows   trade_id {row['tid_min']:,} .. "
          f"{row['tid_max']:,}   span {span:,}")
    if span != row["rows"]:
        raise ValueError(
            f"{row['symbol']} trade_id is not contiguous: {row['rows']} rows span {span} "
            f"ids -- the download is truncated or a file split was lost")

# The one place the three-state return value is load-bearing rather than decorative: the handler
# below runs if and only if verdict() said APPLIES, which is also why BANNED returns False.
if verdict(ent_dupes > 0,
           f"Step 1 dedup: {ent_dupes:,} duplicate (symbol, trade_id) keys in "
           f"{ent_raw_rows:,} rows"):
    raise ValueError(f"{ent_dupes} duplicate keys -- a double-ingested file split")

  BTCUSDT  173,468 rows   trade_id 4,359,935,386 .. 4,360,108,853   span 173,468
  ETHUSDT  101,012 rows   trade_id 2,010,047,664 .. 2,010,148,675   span 101,012
  SOLUSDT   81,721 rows   trade_id 916,174,351 .. 916,256,071   span 81,721
N/A      -- Step 1 dedup: 0 duplicate (symbol, trade_id) keys in 356,201 rows


### b. Handle duplicate keys

Nothing to handle: `dropDuplicates` is a no-op here, priced rather than assumed. The step's
second half is a prohibition — the tick-domain version of the framework's "phantom rows from
misaligned timestamps" is joining the three symbols on `event_time` so one row carries BTC,
ETH and SOL side by side. Cross-symbol alignment is deferred to the shared bar grid in stage 4,
where it is a grouping rather than a join.

In [8]:
ent_deduped_rows = ent_ticks.dropDuplicates(["symbol", "trade_id"]).count()
print(f"dropDuplicates(['symbol','trade_id']): {ent_raw_rows:,} -> {ent_deduped_rows:,} rows "
      f"({ent_raw_rows - ent_deduped_rows} removed)")

# Price the banned join instead of asserting it is bad. It fails in two directions at once: at
# the source resolution the timestamps are too fine to ever agree, so the "aligned" frame is
# empty; relax the grain by one order of magnitude to make it match and it fans out
# many-to-many, because most ticks share their timestamp with a neighbour of the same symbol.
ent_shared_us = (ent_raw_rows
                 - ent_ticks.select("symbol", "event_time_us").distinct().count())
print(f"ticks sharing a microsecond with another tick of the SAME symbol: "
      f"{ent_shared_us:,}/{ent_raw_rows:,} ({100.0 * ent_shared_us / ent_raw_rows:.1f}%)")

ent_btc_us = ent_ticks.filter(func.col("symbol") == "BTCUSDT").select("event_time_us")
ent_eth_us = ent_ticks.filter(func.col("symbol") == "ETHUSDT").select("event_time_us")
for ent_div, ent_grain in ((1, "microsecond"), (1_000, "millisecond"), (1_000_000, "second")):
    ent_key = (func.col("event_time_us") / func.lit(ent_div)).cast("long").alias("k")
    ent_fanout = (ent_btc_us.select(ent_key)
                  .join(ent_eth_us.select(ent_key), on="k", how="inner").count())
    print(f"  BTC x ETH inner join at {ent_grain:<11} grain: {ent_fanout:>12,} rows "
          f"from {ent_btc_rows:,} BTC ticks")

# The verdict message is the job's, verbatim, and it is a statement about the full month. On
# this two-hour extract the cross-symbol collision count at microsecond grain is zero instead,
# which is the same finding read from the other end: the join is either empty or fabricated, and
# there is no grain at which it is neither.
verdict(False, "Step 1 temporal join: aligning symbols on event_time fabricates phantom rows -- "
               "BTC/ETH/SOL routinely share a microsecond", banned=True)

dropDuplicates(['symbol','trade_id']): 356,201 -> 356,201 rows (0 removed)


ticks sharing a microsecond with another tick of the SAME symbol: 291,068/356,201 (81.7%)


  BTC x ETH inner join at microsecond grain:            0 rows from 173,468 BTC ticks


  BTC x ETH inner join at millisecond grain:      114,023 rows from 173,468 BTC ticks


  BTC x ETH inner join at second      grain:   10,405,061 rows from 173,468 BTC ticks
BANNED   -- Step 1 temporal join: aligning symbols on event_time fabricates phantom rows -- BTC/ETH/SOL routinely share a microsecond


False

## 3. Step 2 — Structural Syntax Normalisation

### a. Find

The schema already made every typing decision, so this half is a report on what it decided and
a check that nothing arrived unparsed. Spark's CSV reader is `PERMISSIVE` by default: an
unparseable field becomes NULL with no error, which is exactly how a source format change
turns `is_buyer_maker` into a silently 100%-null column.

In [9]:
ent_ticks.printSchema()

verdict(True, "Step 2 schema: 7 headerless columns named and typed by position -- "
              "3 decimal(18,8), 2 bigint, 2 boolean; inferSchema not used")

# Step 2's "trim whitespace, lowercase the headers" half is a measured no-op: after the schema
# is applied there is not one StringType column in the frame. `symbol` is ours, uppercase by
# construction from the filename.
ent_string_cols = [f.name for f in ent_ticks.schema.fields
                   if f.dataType.simpleString() == "string" and f.name != "symbol"]
verdict(bool(ent_string_cols),
        f"Step 2 text: {len(ent_string_cols)} string columns survive the schema -- "
        f"trim/lower has nothing to normalise")

# The PERMISSIVE guard, on every column at once.
ent_nulls = ent_ticks.select([func.count(func.when(func.col(c).isNull(), c)).alias(c)
                              for c in ENT_DATA_COLS]).first().asDict()
ent_bool_mix = (ent_ticks.groupBy("is_buyer_maker").count()
                .orderBy("is_buyer_maker").collect())
print("nulls per column:", ent_nulls)
print("is_buyer_maker parsed from the literal 'True'/'False':",
      {r["is_buyer_maker"]: r["count"] for r in ent_bool_mix})
verdict(sum(ent_nulls.values()) > 0,
        f"Step 2 parse: {sum(ent_nulls.values())} NULLs across {len(ENT_DATA_COLS)} columns "
        f"under PERMISSIVE mode in {ent_raw_rows:,} ticks")

root
 |-- trade_id: long (nullable = true)
 |-- price: decimal(18,8) (nullable = true)
 |-- qty: decimal(18,8) (nullable = true)
 |-- quote_qty: decimal(18,8) (nullable = true)
 |-- event_time_us: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- symbol: string (nullable = false)

APPLIES  -- Step 2 schema: 7 headerless columns named and typed by position -- 3 decimal(18,8), 2 bigint, 2 boolean; inferSchema not used


N/A      -- Step 2 text: 0 string columns survive the schema -- trim/lower has nothing to normalise


nulls per column: {'trade_id': 0, 'price': 0, 'qty': 0, 'quote_qty': 0, 'event_time_us': 0, 'is_buyer_maker': 0, 'is_best_match': 0}
is_buyer_maker parsed from the literal 'True'/'False': {False: 207436, True: 148765}
N/A      -- Step 2 parse: 0 NULLs across 7 columns under PERMISSIVE mode in 356,201 ticks


False

### a. Find — the decimal / double decision

`price`, `qty` and `quote_qty` are `decimal(18,8)` and not `double`. The argument is
determinism, not precision: float addition is not associative, and Spark does not promise a
stable merge order across a shuffle, so the same job over byte-identical input emits different
numbers when the partitioning changes. Below, the same sum is taken at three shuffle widths.

In [10]:
print("sum(quote_qty) over the whole sample, by shuffle width:")
for n in (3, 11, 29):
    ent_d = ent_ticks.repartition(n).agg(
        func.sum(func.col("quote_qty").cast("double"))).first()[0]
    ent_m = ent_ticks.repartition(n).agg(func.sum("quote_qty")).first()[0]
    print(f"  {n:>2} partitions   double {ent_d!r:<22}   decimal {ent_m}")

# The same non-associativity in pure Python, over 5,000 real quote_qty values: summing them
# forwards and backwards is the smallest possible model of two different merge orders.
ent_vals = [r[0] for r in ent_ticks.select("quote_qty").limit(5000).collect()]
ent_f_fwd = 0.0
for v in ent_vals:
    ent_f_fwd += float(v)
ent_f_rev = 0.0
for v in reversed(ent_vals):
    ent_f_rev += float(v)
print(f"\n5,000 real values, float   : forward {ent_f_fwd!r}  reversed {ent_f_rev!r}  "
      f"equal={ent_f_fwd == ent_f_rev}")
print(f"5,000 real values, Decimal : forward {sum(ent_vals, Decimal(0))}  "
      f"reversed {sum(reversed(ent_vals), Decimal(0))}  "
      f"equal={sum(ent_vals, Decimal(0)) == sum(reversed(ent_vals), Decimal(0))}")

# Decimal also has no NaN, and NaN sorts as the LARGEST value in Spark -- one NaN price would
# win max("price") and every argmax below. Cast to double only at the very last step, for log()
# and for pyspark.ml, never for an aggregation.
verdict(True, "Step 2 numeric type: decimal(18,8) on price/qty/quote_qty -- a double pipeline "
              "is not idempotent across shuffle widths")

sum(quote_qty) over the whole sample, by shuffle width:


   3 partitions   double 210231898.95614502       decimal 210231898.95614160


  11 partitions   double 210231898.95614326       decimal 210231898.95614160


  29 partitions   double 210231898.95614254       decimal 210231898.95614160

5,000 real values, float   : forward 2992974.0347169107  reversed 2992974.034716914  equal=False
5,000 real values, Decimal : forward 2992974.03471690  reversed 2992974.03471690  equal=True
APPLIES  -- Step 2 numeric type: decimal(18,8) on price/qty/quote_qty -- a double pipeline is not idempotent across shuffle widths


True

### b. Handle

One derivation (`event_time`), one session-level pin (UTC), and one refusal.

In [11]:
# timestamp_micros through expr(): the Python wrapper does not exist in Spark 3.3.0 (Glue 4.0).
# event_time_us is KEPT alongside event_time -- it is the raw ingested value and it cannot be
# re-derived once dropped.
ent_ticks = ent_ticks.withColumn("event_time", func.expr("timestamp_micros(event_time_us)"))

verdict(True, "Step 2 timestamp: event_time_us (epoch microseconds) -> event_time via "
              "timestamp_micros, session timezone pinned to UTC")

# The UTC pin is unconditional in the job, and not behind --local. Measured on this box: the
# session timezone defaults to Africa/Johannesburg, under which the last bar of January renders
# as 2025-02-01 01:59 and to_date() drops 120 January bars per symbol into a February partition.
# Glue defaults to UTC, so without the pin the same code emits different partitions in the two
# places -- silent, data-corrupting, and invisible in a diff.
ent_ticks.select("symbol", "trade_id", "event_time_us", "event_time",
                 "price", "qty", "is_buyer_maker").show(3, truncate=False)

# is_best_match is a constant on every archive profiled so far. Dropping it here is tempting and
# is forbidden: the framework's own constant column survives to Step 8, which is path-isolated
# and BANNED on Path 3. The entryway runs before the router reads y's geometry, so it cannot
# know whether Step 8 will even execute on this data; dropping the column would enforce a Path 1
# decision on a path where the framework forbids it. The verdict states the RULE, not a count --
# the count is measured in the same shuffle that builds the bars, in stage 5.
verdict(False, "Step 2 prune: is_best_match is a variance question and VarianceThreshold is "
               "Step 8 -- path-isolated, BANNED on Path 3, so the entryway carries the column "
               "through as all_best_match instead of pre-empting the decision", banned=True)

APPLIES  -- Step 2 timestamp: event_time_us (epoch microseconds) -> event_time via timestamp_micros, session timezone pinned to UTC


+-------+----------+----------------+--------------------------+--------------+----------+--------------+
|symbol |trade_id  |event_time_us   |event_time                |price         |qty       |is_buyer_maker|
+-------+----------+----------------+--------------------------+--------------+----------+--------------+
|BTCUSDT|4359935386|1735689600010866|2025-01-01 00:00:00.010866|93576.00000000|0.00136000|true          |
|BTCUSDT|4359935387|1735689600074095|2025-01-01 00:00:00.074095|93576.00000000|0.00212000|true          |
|BTCUSDT|4359935388|1735689600074095|2025-01-01 00:00:00.074095|93576.00000000|0.00154000|true          |
+-------+----------+----------------+--------------------------+--------------+----------+--------------+
only showing top 3 rows

BANNED   -- Step 2 prune: is_best_match is a variance question and VarianceThreshold is Step 8 -- path-isolated, BANNED on Path 3, so the entryway carries the column through as all_best_match instead of pre-empting the decision


False

## 4. Step 2.5 — Row Granularity: Ticks to Bars

The framework does not number this step and pretending otherwise would be dishonest. It sits
between 2 and 3 for two hard reasons: you cannot bucket by time until the microsecond epoch is
a real timestamp and you cannot sum money until `price`/`qty` are `Decimal` (both Step 2), and
Step 3's flag has to describe the row that *exits* the entryway — which is a bar, not a tick —
while the fork reads `y`'s geometry off a bar-level quantity that does not exist at tick grain.

### a. Find — what makes open and close ill-defined

A bar's `open` and `close` are the only two aggregates here that need an *order*, and the two
obvious ways to get one are both wrong.

In [12]:
# The bucket key: pure integer arithmetic, no division. `event_time_us / I` returns a Double in
# Spark and cast() truncates toward zero rather than flooring; pmod is exact for any interval and
# correct for pre-1970 epochs, and it hands back bar_us directly for timestamp_micros. Through
# expr() again -- func.pmod is versionadded 3.4.0 and Glue 4.0 is 3.3.0, so the Python wrapper
# would raise AttributeError on the one line every bar depends on. The `%` operator compiles on
# 3.3.0 but maps to Remainder, which is negative for negative operands: identical for 2025
# epochs, wrong for a pre-1970 backfill.
ent_ticks = ent_ticks.withColumn(
    "bar_us", func.col("event_time_us") - func.expr(f"pmod(event_time_us, {ENT_INTERVAL_US})"))

# Trap 1: event_time is only NON-DECREASING, so ordering by it leaves open/close undefined
# wherever a tie group holds more than one price.
ent_tie = (ent_ticks.groupBy("symbol", "event_time_us")
           .agg(func.count("*").alias("n"), func.countDistinct("price").alias("distinct_prices")))
ent_tie_row = ent_tie.agg(
    func.sum((func.col("n") > 1).cast("long")).alias("tie_groups"),
    func.sum(((func.col("n") > 1) & (func.col("distinct_prices") > 1))
             .cast("long")).alias("ambiguous"),
    func.max("n").alias("max_ticks_in_one_us")).first()
print(f"microsecond tie groups: {ent_tie_row['tie_groups']:,}   "
      f"of which price is not constant: {ent_tie_row['ambiguous']:,}   "
      f"largest tie group: {ent_tie_row['max_ticks_in_one_us']} ticks in one microsecond")

microsecond tie groups: 26,918   of which price is not constant: 9,203   largest tie group: 1331 ticks in one microsecond


In [13]:
# Trap 2: first()/last() over a groupBy return whatever reached the partition first, which
# changes with the partitioning and with a re-run. min_by/max_by on trade_id do not -- trade_id
# is strictly increasing and contiguous (asserted in stage 2a), and a strictly increasing id
# over a non-decreasing clock IS time order, so it is sufficient alone.
def ent_open_close(n_partitions):
    rows = (ent_ticks.repartition(n_partitions).groupBy("symbol", "bar_us").agg(
        func.first("price").alias("first_price"),
        func.last("price").alias("last_price"),
        func.min_by("price", "trade_id").alias("open"),
        func.max_by("price", "trade_id").alias("close")).collect())
    return {(r["symbol"], r["bar_us"]): (r["first_price"], r["last_price"],
                                         r["open"], r["close"]) for r in rows}

ent_a, ent_b = ent_open_close(3), ent_open_close(29)
ent_unstable = sum(1 for k in ent_a if ent_a[k][:2] != ent_b[k][:2])
ent_stable = sum(1 for k in ent_a if ent_a[k][2:] != ent_b[k][2:])
print(f"bars out of {len(ent_a):,} that change between 3 and 29 shuffle partitions:")
print(f"  first()/last()          : {ent_unstable:,}")
print(f"  min_by/max_by(trade_id) : {ent_stable:,}")

bars out of 4,312 that change between 3 and 29 shuffle partitions:
  first()/last()          : 3,343
  min_by/max_by(trade_id) : 0


In [14]:
# The taker side. is_buyer_maker = True means the BUYER was resting on the book, so the SELLER
# crossed the spread: True is an aggressive SELL, and taker_buy is is_buyer_maker == False.
# Reading the field name as "the buyer made the trade happen" gets the sign backwards, and the
# ~50/50 global base rate means no ratio sanity-check would catch it. Evidence, on this sample:
# the next tick's direction, conditioned on the current tick's flag.
ent_w = Window.partitionBy("symbol").orderBy("trade_id")
ent_next = (ent_ticks.withColumn("next_price", func.lead("price").over(ent_w))
            .filter(func.col("next_price").isNotNull()))
for r in (ent_next.groupBy("is_buyer_maker").agg(
        func.count("*").alias("ticks"),
        func.avg((func.col("next_price") > func.col("price")).cast("double")).alias("p_up"),
        func.avg((func.col("next_price") < func.col("price")).cast("double")).alias("p_down"))
        .orderBy("is_buyer_maker").collect()):
    side = "taker BOUGHT" if r["is_buyer_maker"] is False else "taker SOLD  "
    print(f"  is_buyer_maker={str(r['is_buyer_maker']):<5} ({side})  {r['ticks']:>7,} ticks   "
          f"P(next tick up)={r['p_up']:.4f}   P(next tick down)={r['p_down']:.4f}")
# The job records the bar-level version of the same check, measured on 2M ticks:
# corr(taker imbalance, bar return) = +0.5096 with this convention and exactly -0.5096 inverted.

# The notional column is the exchange's own, not a recomputed price*qty. Equal by construction --
# price carries <= 2 significant decimals and qty <= 5, so the product needs <= 7 and the field
# stores 8 -- and equal in fact on every tick of this sample:
ent_notional_diff = ent_ticks.select(
    func.max(func.abs(func.col("price") * func.col("qty")
                      - func.col("quote_qty")))).first()[0]
print(f"\nmax |price * qty - quote_qty| over the sample: {ent_notional_diff}  "
      f"(exactly zero -- the product lands at decimal(38,16), which prints 0 as 0E-16)")

  is_buyer_maker=False (taker BOUGHT)  207,433 ticks   P(next tick up)=0.1678   P(next tick down)=0.0417
  is_buyer_maker=True  (taker SOLD  )  148,765 ticks   P(next tick up)=0.0550   P(next tick down)=0.1830



max |price * qty - quote_qty| over the sample: 0E-16  (exactly zero -- the product lands at decimal(38,16), which prints 0 as 0E-16)


### b. Handle — the aggregate

One `groupBy` on `(symbol, bar_us)`, on a half-open `[bar_us, bar_us + interval)` grid.
Everything the later parts need is produced in this single shuffle, including four guard
columns that exist only to be asserted on and then dropped.

In [15]:
ent_zero = func.lit(0).cast(ENT_MONEY)
ENT_GUARD_COLS = ["_nulls", "_zero_qty", "_zero_price", "_not_best_match"]

ent_agg = (ent_ticks.groupBy("symbol", "bar_us").agg(
    # Order-dependent aggregates, keyed on trade_id for the reasons measured in 4a.
    func.min_by("price", "trade_id").alias("open"),
    func.max("price").alias("high"),
    func.min("price").alias("low"),
    func.max_by("price", "trade_id").alias("close"),
    func.sum("qty").alias("volume"),
    # The exchange's notional. Using it saves a multiply over 341M rows, keeps the sum at
    # decimal(28,8) instead of burning the overflow margin on decimal(38,16), and lets the
    # result reconcile digit-for-digit against Binance klines.
    func.sum("quote_qty").alias("quote_volume"),
    func.count("*").alias("n_ticks"),
    func.min("trade_id").alias("first_trade_id"),
    func.max("trade_id").alias("last_trade_id"),
    # .otherwise(zero) is load-bearing: a bar whose every taker sold is a real observed zero,
    # not a gap, and without it sum(NULL) makes the cell null and poisons the feature.
    func.sum(func.when(~func.col("is_buyer_maker"), func.col("qty"))
             .otherwise(ent_zero)).alias("taker_buy_qty"),
    func.sum(func.when(~func.col("is_buyer_maker"), func.col("quote_qty"))
             .otherwise(ent_zero)).alias("taker_buy_quote_qty"),
    # bool_and has no Python wrapper before 3.5; the SQL name exists in 3.3.0. This is the
    # zero-variance column surviving the granularity change intact.
    func.expr("bool_and(is_best_match)").alias("all_best_match"),
    # Tick-grain guards, folded into this shuffle so they cost no extra pass.
    func.sum(func.greatest(*[func.col(c).isNull().cast("int")
                             for c in ENT_DATA_COLS]).cast("long")).alias("_nulls"),
    func.sum((func.col("qty") == 0).cast("long")).alias("_zero_qty"),
    func.sum((func.col("price") == 0).cast("long")).alias("_zero_price"),
    func.sum((~func.col("is_best_match")).cast("long")).alias("_not_best_match"),
))

verdict(True, f"Step 2.5 granularity: {ent_raw_rows:,} ticks -> OHLCV bars at "
              f"{ENT_INTERVAL_LABEL} (open/close by min_by/max_by on trade_id, never first/last)")

# VWAP is quote_volume / volume and is deliberately NOT stored: it is a ratio of two columns
# that are both present, so storing it would be a feature-layer decision taken in the entryway.
(ent_agg.filter(func.col("symbol") == "BTCUSDT").orderBy("bar_us")
 .select("bar_us", "open", "high", "low", "close", "n_ticks", "volume",
         (func.col("quote_volume") / func.col("volume")).cast(ENT_MONEY).alias("vwap"))
 .show(5, truncate=False))

APPLIES  -- Step 2.5 granularity: 356,201 ticks -> OHLCV bars at 5s (open/close by min_by/max_by on trade_id, never first/last)


+----------------+--------------+--------------+--------------+--------------+-------+----------+--------------+
|bar_us          |open          |high          |low           |close         |n_ticks|volume    |vwap          |
+----------------+--------------+--------------+--------------+--------------+-------+----------+--------------+
|1735689600000000|93576.00000000|93576.01000000|93576.00000000|93576.00000000|76     |0.43243000|93576.00048725|
|1735689605000000|93576.00000000|93576.01000000|93548.16000000|93548.16000000|385    |1.34167000|93561.76224876|
|1735689610000000|93548.17000000|93548.17000000|93548.16000000|93548.17000000|76     |0.76204000|93548.16124705|
|1735689615000000|93548.16000000|93548.17000000|93537.50000000|93548.17000000|405    |0.98736000|93541.57297571|
|1735689620000000|93548.17000000|93548.17000000|93548.16000000|93548.17000000|48     |0.38586000|93548.16130410|
+----------------+--------------+--------------+--------------+--------------+-------+----------

## 5. Step 3 — Missingness Indicator Generation

### a. Find missing bars

Missingness here is **row-shaped, not cell-shaped**. If a bar exists at all then at least one
tick landed in it, so OHLCV are non-null by construction; the only thing that *can* be missing
is a whole bar, and a missing row is invisible until a reference calendar is joined onto it.

The calendar is **declared**, never taken from `min`/`max` of the data — an observed-range
calendar reports zero gaps even when the last three days failed to download. For this two-hour
extract the declared window is narrowed to match, exactly as the job's `--calendar-start` /
`--calendar-end` overrides do. `glue-ingest-bars.py`'s own docstring records why: run the
sample against the month-wide grid and Step 3 truthfully reports 1,606,390 of 1,607,040 bars
missing — arithmetically correct, and a measurement of "the sample is a sample" rather than of
the data. Narrowing the window keeps the check meaningful at both scales while keeping the
calendar declared rather than inferred.

In [16]:
ENT_START_US = int(datetime(2025, 1, 1, 0, 0, tzinfo=timezone.utc).timestamp()) * 1_000_000
ENT_END_US = int(datetime(2025, 1, 1, 2, 0, tzinfo=timezone.utc).timestamp()) * 1_000_000

# Both guards run before Spark does, so a bad interval fails in milliseconds rather than after
# the Step 1 shuffle. A start off the interval boundary would produce slots no tick can land in
# -- every bar_us from pmod IS a multiple of the interval -- so the join would miss on every row
# and Step 3 would report 100% missing with no null and no row-count change to give it away.
assert ENT_START_US % ENT_INTERVAL_US == 0, "calendar start is off the interval boundary"
assert (ENT_END_US - ENT_START_US) % ENT_INTERVAL_US == 0, "interval does not tile the calendar"

ENT_SLOTS = (ENT_END_US - ENT_START_US) // ENT_INTERVAL_US
ent_symbol_frame = spark.createDataFrame([(s,) for s in ENT_SYMBOLS], "symbol string")
ent_grid = (spark.range(0, ENT_SLOTS)
            .select((func.lit(ENT_START_US)
                     + func.col("id") * func.lit(ENT_INTERVAL_US)).alias("bar_us"))
            .crossJoin(func.broadcast(ent_symbol_frame)))

# A dense grid rather than a sparse frame, even when it adds only eight rows: the downstream
# target is a NEXT-bar return, which needs a well-defined successor, and Step 3 needs an actual
# column to flag rather than an absence to infer.
ent_joined = ent_grid.join(ent_agg, on=["symbol", "bar_us"], how="left").cache()

ent_total = ENT_SLOTS * len(ENT_SYMBOLS)
print(f"declared calendar: {ENT_SLOTS:,} slots of {ENT_INTERVAL_LABEL} x "
      f"{len(ENT_SYMBOLS)} symbols = {ent_total:,} rows")
(ent_joined.groupBy("symbol")
 .agg(func.sum(func.col("n_ticks").isNull().cast("int")).alias("empty_bars"),
      func.sum(func.coalesce(func.col("n_ticks"), func.lit(0))).alias("ticks"))
 .orderBy("symbol").show())

ent_missing = ent_joined.filter(func.col("n_ticks").isNull()).count()
verdict(ent_missing > 0,
        f"Step 3 bars: {ent_missing:,}/{ent_total:,} empty at {ENT_INTERVAL_LABEL} "
        f"({100.0 * ent_missing / ent_total:.3f}%)")

declared calendar: 1,440 slots of 5s x 3 symbols = 4,320 rows


+-------+----------+------+
| symbol|empty_bars| ticks|
+-------+----------+------+
|BTCUSDT|         0|173468|
|ETHUSDT|         5|101012|
|SOLUSDT|         3| 81721|
+-------+----------+------+

APPLIES  -- Step 3 bars: 8/4,320 empty at 5s (0.185%)


True

### b. Handle missing bars — flag, never fill

The flag is created here, before any imputation, and that is the whole of Step 3. Filling is
Step 4, Step 4 is *after the fork*, and what it does there depends on which path fired — so
doing any of it in the entryway would enforce one path's decision on all three.

In [17]:
ent_bars = (ent_joined
            .withColumn("is_missing_bar",
                        func.when(func.col("n_ticks").isNull(), 1).otherwise(0))
            # n_ticks is coalesced and NOTHING ELSE IS. A count over an empty bucket is an exact
            # observation -- zero trades were seen -- and it is what makes the flag auditable,
            # because a reader can recompute is_missing_bar from it instead of trusting it.
            # coalesce(volume, 0) is a different claim: it would assert "there was trading
            # interest and it netted zero" over "we observed nothing", in a line that does not
            # look like imputation.
            .withColumn("n_ticks", func.coalesce(func.col("n_ticks"), func.lit(0)))
            # The bar timestamp is the bucket START, matching Binance's kline open_time. The
            # _utc suffix and the explicit "open_time" guard the seam bug: an end-labelled
            # feature table joined to a start-labelled target table on a bare `ts` hands every
            # row the target one bar early -- every key matches, no nulls, no row-count change,
            # and the backtest reads the future.
            .withColumn("bar_open_time_utc", func.expr("timestamp_micros(bar_us)")))

# The window edges. The last bar has no successor and the first no predecessor, so any
# close-to-close return is NULL at both ends. Coalescing that NULL to 0 would hand every month
# seam a free "the market does not move at month end" prior.
ent_last_bar_us = ENT_END_US - ENT_INTERVAL_US
ent_bars = (ent_bars
            .withColumn("is_first_bar",
                        func.when(func.col("bar_us") == func.lit(ENT_START_US), 1).otherwise(0))
            .withColumn("is_last_bar",
                        func.when(func.col("bar_us") == func.lit(ent_last_bar_us), 1)
                        .otherwise(0)))
verdict(True, f"Step 3 window edges: is_first_bar / is_last_bar flagged at {ENT_START_US} and "
              f"{ent_last_bar_us} (epoch microseconds)")

# What an empty bar actually looks like on the way out: flagged, counted, and otherwise NULL.
(ent_bars.filter(func.col("is_missing_bar") == 1)
 .select("symbol", "bar_open_time_utc", "open", "high", "low", "close", "volume",
         "n_ticks", "is_missing_bar")
 .orderBy("symbol", "bar_us").show(truncate=False))

APPLIES  -- Step 3 window edges: is_first_bar / is_last_bar flagged at 1735689600000000 and 1735696795000000 (epoch microseconds)


+-------+-------------------+----+----+----+-----+------+-------+--------------+
|symbol |bar_open_time_utc  |open|high|low |close|volume|n_ticks|is_missing_bar|
+-------+-------------------+----+----+----+-----+------+-------+--------------+
|ETHUSDT|2025-01-01 00:25:30|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|ETHUSDT|2025-01-01 00:30:05|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|ETHUSDT|2025-01-01 01:24:45|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|ETHUSDT|2025-01-01 01:42:25|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|ETHUSDT|2025-01-01 01:59:45|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|SOLUSDT|2025-01-01 01:20:50|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|SOLUSDT|2025-01-01 01:38:10|NULL|NULL|NULL|NULL |NULL  |0      |1             |
|SOLUSDT|2025-01-01 01:43:40|NULL|NULL|NULL|NULL |NULL  |0      |1             |
+-------+-------------------+----+----+----+-----+------+-------+--------------+



In [18]:
# Four standard OHLC gap fixes, all forbidden here, all Step 4 decisions: forward-filling close
# into an empty bar (the standard and standardly wrong fix -- it invents a print), dropping the
# empty rows (that is Path 1's complete-case exclusion and only Path 1's), coalesce(volume, 0),
# and back-filling open from the next bar. Path 3's Step 4 override is explicit that gaps are
# unobserved latent states and that imputing one fabricates a signal that never existed.
verdict(False, "Step 3 fill: forward-fill / coalesce(volume, 0) / row-drop are Step 4 -- "
               "path-isolated, decided after the fork, not here", banned=True)

# lead() over the bar series is likewise not the entryway's to compute: run per-month it bakes a
# permanent NULL into the last bar of every month, so the target is not append-only and
# January's edge would need recomputing when February lands.
verdict(False, "Step 3 target: next-bar return via lead() is a post-fork feature-layer "
               "construct -- computing it per month freezes a NULL at every seam", banned=True)

BANNED   -- Step 3 fill: forward-fill / coalesce(volume, 0) / row-drop are Step 4 -- path-isolated, decided after the fork, not here


BANNED   -- Step 3 target: next-bar return via lead() is a post-fork feature-layer construct -- computing it per month freezes a NULL at every seam


False

### c. Review the entryway output

One aggregate over the bar frame carries every invariant. The first assertion is the row
accounting identity, and it is the best one in the pipeline: it proves the granularity change
neither lost nor invented a tick.

In [19]:
ent_checks = ent_bars.agg(
    func.sum("n_ticks").alias("ticks"),
    func.coalesce(func.sum("_nulls"), func.lit(0)).alias("nulls"),
    func.coalesce(func.sum("_zero_qty"), func.lit(0)).alias("zero_qty"),
    func.coalesce(func.sum("_zero_price"), func.lit(0)).alias("zero_price"),
    func.sum((func.col("low") > func.least("open", "close")).cast("int")).alias("bad_low"),
    func.sum((func.col("high") < func.greatest("open", "close")).cast("int")).alias("bad_high"),
    func.sum((func.col("taker_buy_qty") > func.col("volume")).cast("int")).alias("bad_taker"),
    func.coalesce(func.sum("_not_best_match"), func.lit(0)).alias("not_best_match"),
).collect()[0]

assert ent_checks["ticks"] == ent_raw_rows, (
    f"bars account for {ent_checks['ticks']} ticks but {ent_raw_rows} were read -- ticks fell "
    f"outside the calendar and were dropped by the join, or a file split was lost")
assert ent_checks["nulls"] == 0
assert ent_checks["bad_low"] == 0 and ent_checks["bad_high"] == 0, (
    "an aggregate is wired to the wrong column")
assert ent_checks["bad_taker"] == 0, "the is_buyer_maker predicate is inverted"
print(f"row accounting: {ent_checks['ticks']:,} ticks in the bars == "
      f"{ent_raw_rows:,} ticks read")

# Disguised missingness -- a gap wearing a legal value. A zero qty or a zero price would be one;
# a zero taker_buy_qty would not, and is deliberately not counted here.
verdict(ent_checks["zero_qty"] + ent_checks["zero_price"] > 0,
        f"Step 3 ticks: {ent_checks['nulls']:,} nulls, {ent_checks['zero_qty']:,} zero-qty, "
        f"{ent_checks['zero_price']:,} zero-price in {ent_raw_rows:,} ticks")

# The measured half of the Step 2 prune verdict, re-measured every run rather than trusted. It
# reads N/A precisely because the column has no variance to prune -- which is the finding, not
# an absence of one.
verdict(ent_checks["not_best_match"] > 0,
        f"Step 2 prune evidence: is_best_match is False in {ent_checks['not_best_match']:,}/"
        f"{ent_raw_rows:,} ticks -- kept as all_best_match for Step 8 to prune on the paths "
        f"where Step 8 runs")

row accounting: 356,201 ticks in the bars == 356,201 ticks read
N/A      -- Step 3 ticks: 0 nulls, 0 zero-qty, 0 zero-price in 356,201 ticks


N/A      -- Step 2 prune evidence: is_best_match is False in 0/356,201 ticks -- kept as all_best_match for Step 8 to prune on the paths where Step 8 runs


False

In [20]:
# The 18-column entryway contract, in the job's column order. The guard columns are dropped:
# they existed to be asserted on, and they have been.
ent_out = ent_bars.drop(*ENT_GUARD_COLS).select(
    "symbol", "bar_us", "bar_open_time_utc",
    "open", "high", "low", "close", "volume", "quote_volume",
    "n_ticks", "first_trade_id", "last_trade_id",
    "taker_buy_qty", "taker_buy_quote_qty", "all_best_match",
    "is_missing_bar", "is_first_bar", "is_last_bar")

# The diff, made mechanical: this notebook re-derived the frame from the sample CSVs, and the
# setup cell read the same frame from the parquet glue-ingest-bars.py wrote. exceptAll in both
# directions is zero if and only if the two agree row for row and cell for cell.
ent_only = ent_out.exceptAll(bars).count()
ent_job_only = bars.exceptAll(ent_out).count()
print(f"notebook rows {ent_out.count():,} | job parquet rows {bars.count():,} "
      f"({BARS_PATH})")
print(f"rows in the notebook frame but not the job's: {ent_only}")
print(f"rows in the job's frame but not the notebook's: {ent_job_only}")
assert ent_only == 0 and ent_job_only == 0, "the notebook and the Glue job disagree"

# From here on `bars` is the frame this notebook derived, not the one it loaded. It is the only
# unprefixed name in the notebook, and every path section below starts from it.
bars = ent_out.cache()
bars.printSchema()
print(f"entryway output: {bars.count():,} rows x {len(bars.columns)} columns")

notebook rows 4,320 | job parquet rows 4,320 (_localrun/bars)
rows in the notebook frame but not the job's: 0
rows in the job's frame but not the notebook's: 0
root
 |-- symbol: string (nullable = true)
 |-- bar_us: long (nullable = false)
 |-- bar_open_time_utc: timestamp (nullable = false)
 |-- open: decimal(18,8) (nullable = true)
 |-- high: decimal(18,8) (nullable = true)
 |-- low: decimal(18,8) (nullable = true)
 |-- close: decimal(18,8) (nullable = true)
 |-- volume: decimal(28,8) (nullable = true)
 |-- quote_volume: decimal(28,8) (nullable = true)
 |-- n_ticks: long (nullable = false)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- taker_buy_qty: decimal(28,8) (nullable = true)
 |-- taker_buy_quote_qty: decimal(28,8) (nullable = true)
 |-- all_best_match: boolean (nullable = true)
 |-- is_missing_bar: integer (nullable = false)
 |-- is_first_bar: integer (nullable = false)
 |-- is_last_bar: integer (nullable = false)

entryway output: 

# THE ARCHITECTURAL FORK

> *"The moment data exits Step 3, the pipeline reads the geometry of target variable `y` and
> branches into three path-isolated sub-refineries. This happens **before any imputation or
> spatial modifications** — the path determines how missing data is handled, not the other way
> around."*

Everything above this line ran **once**, for all three paths, and made no decision that any
path could disagree with. That is why the fill in Step 3 is BANNED rather than deferred: the
eight empty bars leave the entryway flagged and unfilled, and the three paths below handle
them in three incompatible ways —

- **Path 1 (Continuous)** — `OVERRIDE`: measure missingness on the target; above 5%, ban
  medians and drop the affected rows as complete cases.
- **Path 2 (Categorical)** — `APPLIES`: median-impute numerics, and give the missing category
  a name of its own rather than discarding the row.
- **Path 3 (Bandit)** — `OVERRIDE`: impute nothing. A gap is an unobserved latent state and
  filling it fabricates a signal that never existed.

All three read the same `is_missing_bar` column, which exists on every path even where the
value it flags is dropped. The pipeline is a **single, non-looping forward pass** — nothing
below revisits anything above, and no path can reach into another.

Everything from here down is **path-isolated**.

---

# 💥 The Architectural Fork

Everything above ran once. Everything below runs three times, differently.

> *"The moment data exits Step 3, the pipeline reads the geometry of target variable `y` and
> branches into three path-isolated sub-refineries. This happens **before any imputation or
> spatial modifications** — the path determines how missing data is handled, not the other way
> around."*

That ordering is the whole design. Missingness *flags* are Step 3 and path-blind, so they exist
on all three paths. Missingness *handling* is Step 4 and path-specific — and the three paths
disagree completely about what to do, which is why the flag has to be created before the router
knows which path it is on. Part 1 flagged 8 empty bars and deliberately filled none of them.

The framework also specifies that the pipeline runs as a **single, non-looping forward pass** —
no step revisits an earlier step's output. Step 8 cannot re-consult Step 6's correlation matrix;
Step 10's scaler choice cannot feed back into Step 8's pruning.

## Steps 4–10 × 3 paths

| Step | Path 1 — Continuous | Path 2 — Categorical | Path 3 — Bandit |
| --- | --- | --- | --- |
| **4** Imputation | 🔶 **OVERRIDE** — if >5% missing: ban medians, complete-case | ✅ median impute; missing categories → `SYSTEM_STATE_UNKNOWN` | 🔶 **OVERRIDE** — maintain native missingness as unobserved latent state |
| **5** Diagnostics | ✅ univariate density profiles; t-SNE sandbox (read-only) | ✅ class-balance; interaction correlation | ✅ stream diagnostics; HF interaction frames |
| **6** Topology | ✅ global cross-correlation matrix | ✅ cross-correlation + cyclical sin/cos coordinates | ✅ cyclical time coordinates |
| **7** Feature Eng | ✅ deterministic cross-products | ✅ cyclical coords + interaction cross-products | ✅ interaction frame preparation |
| **8** Pruning | ✅ VarianceThreshold, bypass categorical dummies | 🔷 **ENFORCE** — one-hot *before* variance pruning | ⛔ **BANNED** — raw tokens needed for association mining |
| **9** Regularisation | ✅ CV Elastic Net | ✅ CV Elastic Net | ⛔ **BANNED** — destroys multi-item relationship detection |
| **10** Scaling | 🔶 **LIMIT** — linear scalers only (StandardScaler) | ✅ quantile search: Standard / Power / Quantile | ⛔ **BANNED** — destroys Bayesian conjugate updating |

21 cells: 15 plain APPLIES, 3 OVERRIDE/LIMIT, 1 ENFORCE, **3 BANNED — all on Path 3**.

## A wider vocabulary below this line

Part 1 used three verdict states, because the entryway only needs three: a step either has work
to do (`APPLIES`), has been checked and has nothing to do (`N/A`), or is forbidden (`BANNED`).
That is exactly `verdict()` as `glue-ingest-bars.py` ships it.

Steps 4–10 need more, because the framework distinguishes three *kinds* of "not the default":

| badge | meaning |
| --- | --- |
| `APPLIES` / `N/A` | as in Part 1 — measured, with a number behind it |
| 🔶 `OVERRIDE` | the default operation is replaced by a different one on this path |
| 🔶 `LIMIT` | the operation runs, but a narrower version of it |
| 🔷 `ENFORCE` | the operation runs, but its **ordering** relative to another step is mandated |
| ⛔ `BANNED` | forbidden, and the message is the reason |

`OVERRIDE`, `LIMIT` and `ENFORCE` are all narrowings of an `APPLIES` cell, so the code below
reports them as `APPLIES` with the framework's badge named in the message. Only `BANNED` changes
control flow — `verdict()` returns `False` for it, so a banned step can never reach its handler
even if someone later wraps it in `if verdict(...)`.

## Why Path 3's three bans exist

Not style. Each one names an engine that breaks:

- **Step 8** — Apriori/ECLAT association mining consumes raw categorical *tokens*. One-hot
  encoding dissolves a token into indicator columns, and there is no itemset left to mine.
- **Step 9** — Elastic Net's L1 term zeroes low-variance columns. A rare-but-high-lift itemset
  is exactly a low-variance column, so regularisation deletes the discoveries the path exists
  to make.
- **Step 10** — Thompson Sampling's Beta posterior updates as `α ← γα + x`, `β ← γβ + (1−x)`
  with `x ∈ {0,1}`. α and β are *counts*. Scaling turns them into something that is not a count
  and the conjugate arithmetic stops meaning anything.

Part 4 demonstrates each breakage rather than asserting it.

---

# Part 2 — Path 1: the continuous value stream

The fork reads the geometry of `y` and routes. Path 1 is selected when `y` is a real-valued
continuous quantity — in the framework's own worked example, `order_total_usd`. Here the
nominated target is the **next-bar log return**, `log(lead(close) / close)` over a window
partitioned by `symbol` and ordered by `bar_us`. It is a real-valued continuous double, so
Path 1 fires.

**The framework offers no guidance on manufacturing a target.** NexusMart arrived with three
genuine experiments and three genuine targets; a bar file arrives with none. Nominating the
next-bar return is this notebook's decision, not the framework's, and every Path 1 verdict
below is conditional on it. Flagged here once rather than implied downstream.

Steps 4–10 below are path-isolated: nothing in this Part is shared with Parts 3 or 4, and the
pipeline is a single non-looping forward pass — Step 8 cannot consult Step 6's matrix, and
Step 10's scaler cannot feed back into Step 8's pruning.

In [21]:
import math

from pyspark.sql import Window
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (Imputer, StandardScaler, VarianceThresholdSelector,
                                VectorAssembler)
# A Spark ML vector is a VectorUDT struct, NOT an array -- getItem/[i] raise
# INVALID_EXTRACT_BASE_FIELD_TYPE on it.
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import LinearRegression
from pyspark.ml.stat import Correlation
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# The fork's own verdict. It reads y's TYPE, not y's content -- missingness is Step 4's problem
# and the framework is explicit that the path is chosen before any imputation, never after.
verdict(True, "Fork: target geometry is a real-valued continuous double (next-bar log return) "
              "-> Path 1. Steps 4-10 below are Path 1's and no other path's")

# The post-fork windowed pass. lead() for the target, lag() for the one backward-looking
# feature, and the per-bar scale-free ratios, all in ONE pass so that Step 4 -- which runs next
# and never runs again -- can see every null the construction creates. Doing the lag in Step 7
# instead would manufacture nulls AFTER the only step allowed to handle them.
p1_win = Window.partitionBy("symbol").orderBy("bar_us")

# Decimal arithmetic all the way to the log. close is decimal(18,8); decimal/decimal division is
# exact and partition-order independent, so the gross return is byte-identical under any shuffle
# width. Casting close to double FIRST and dividing there is the version that gives three
# different answers at 3 / 11 / 29 partitions. log() has no decimal implementation in Spark, so
# the cast happens on the finished ratio and nowhere earlier.
p1 = (bars
      .withColumn("p1_close_next", func.lead("close").over(p1_win))
      .withColumn("p1_close_prev", func.lag("close").over(p1_win))
      # THE TARGET. Empty bars carry NULL close (Step 3 flagged and deliberately did not fill),
      # so both the empty bar and the bar BEFORE it get a null target -- the null propagates
      # backwards through lead() as well as forwards, which is why Step 4's rate below is not
      # simply the empty-bar rate.
      .withColumn("p1_y", func.log((func.col("p1_close_next") / func.col("close")).cast("double")))
      .withColumn("p1_ret1", func.log((func.col("close") / func.col("p1_close_prev")).cast("double")))
      # Scale-free per-bar features. BTC prints near $94k and SOL near $190 in this window, so a
      # raw price or a raw dollar range is not comparable across the three symbols in one matrix;
      # a basis-point ratio is. Kept in decimal -- the *10000 is exact.
      .withColumn("p1_range_bp",
                  (func.col("high") - func.col("low")) / func.col("close") * func.lit(10000))
      .withColumn("p1_body_bp",
                  (func.col("close") - func.col("open")) / func.col("close") * func.lit(10000))
      # taker_buy = (is_buyer_maker == False) -- is_buyer_maker True means the BUYER was resting
      # on the book, so the trade is an aggressive SELL. The entryway measured
      # corr(imbalance, contemporaneous return) = +0.5096 with this convention and exactly
      # -0.5096 inverted; the ~50/50 base rate means no ratio sanity-check would catch the flip.
      .withColumn("p1_imbalance", func.col("taker_buy_quote_qty") / func.col("quote_volume"))
      .withColumn("p1_log_qv", func.log(func.col("quote_volume").cast("double")))
      .drop("p1_close_next", "p1_close_prev"))

p1.select("symbol", "bar_us", "close", "p1_y", "p1_ret1", "p1_range_bp",
          "p1_imbalance").show(4, truncate=False)
print(f"post-fork frame: {p1.count():,} rows x {len(p1.columns)} columns")

APPLIES  -- Fork: target geometry is a real-valued continuous double (next-bar log return) -> Path 1. Steps 4-10 below are Path 1's and no other path's


+-------+----------------+--------------+---------------------+---------------------+---------------+------------+
|symbol |bar_us          |close         |p1_y                 |p1_ret1              |p1_range_bp    |p1_imbalance|
+-------+----------------+--------------+---------------------+---------------------+---------------+------------+
|BTCUSDT|1735689600000000|93576.00000000|-2.975564481401788E-4|NULL                 |0.0010686500812|0.0487246540|
|BTCUSDT|1735689605000000|93548.16000000|1.0689680546712642E-7|-2.975564481401788E-4|2.9770761926263|0.1544117390|
|BTCUSDT|1735689610000000|93548.17000000|0.0                  |1.0689680546712642E-7|0.0010689679980|0.1247047516|
|BTCUSDT|1735689615000000|93548.17000000|0.0                  |0.0                  |1.1405888538493|0.6880501786|
+-------+----------------+--------------+---------------------+---------------------+---------------+------------+
only showing top 4 rows

post-fork frame: 4,320 rows x 24 columns


## 4. Imputation — Path 1 (OVERRIDE)

> *"If >5% missingness: ban medians, pass raw for complete-case analysis"*

The trigger is a strict inequality in both sources — the grid says `>5%`, the LaTeX says
*"exceeds the 5% firewall threshold"* — so at exactly 5.0% the override does **not** fire and
the default (median impute) stands. That boundary is never demonstrated by the framework and
is hard-coded as a named constant below so the choice is visible rather than accidental.

### a. Measure missingness on the target

In [22]:
# Hard-coded, named, and tested strictly. Source wording: ">5% missingness" (step grid) and
# "exceeds the 5% firewall threshold" (LaTeX). Both strict; 5.0% exactly lands on the default.
P1_MISSINGNESS_BAN_THRESHOLD = 0.05

# The framework evaluates the rate as a mean over the flag Step 3 manufactured
# (mean(is_missing_order_total) > 0.05). That does not transfer unchanged here: Step 3's flag
# describes ONE BAR, and the target is a TWO-BAR construct, so the flag under-counts the
# target's nulls. Both numbers are printed and the target's own rate is the one the firewall
# reads -- the LaTeX says "target missingness", so the test is on y, not per-column across X.
p1_n = p1.count()
p1_rates = p1.select(
    func.avg(func.col("p1_y").isNull().cast("double")).alias("y_null_rate"),
    func.avg(func.col("is_missing_bar").cast("double")).alias("step3_flag_rate"),
    func.sum(func.col("p1_y").isNull().cast("int")).alias("y_nulls"),
    func.sum((func.col("p1_y").isNull() & (func.col("is_last_bar") == 1)).cast("int")).alias("tail"),
    func.sum((func.col("p1_y").isNull() & (func.col("is_missing_bar") == 1)).cast("int")).alias("own"),
).first()

print(f"rows                                  {p1_n:,}")
print(f"target nulls                          {p1_rates['y_nulls']:,}  "
      f"({100 * p1_rates['y_null_rate']:.3f}%)")
print(f"  of which last bar of a symbol       {p1_rates['tail']:,}  (structural: no successor)")
print(f"  of which the bar itself is empty    {p1_rates['own']:,}  (NULL close, Step 3 flagged)")
print(f"  of which the NEXT bar is empty      "
      f"{p1_rates['y_nulls'] - p1_rates['tail'] - p1_rates['own']:,}  (lead() pulls the null back)")
print(f"Step 3 is_missing_bar rate            {100 * p1_rates['step3_flag_rate']:.3f}%  "
      f"(under-counts the target: one-bar flag, two-bar target)")
print()

# The structural tail is a mechanical artefact of the window, not a data defect, and the
# framework never says whether it should be inside or outside the rate. Both are reported; the
# firewall is read on the raw rate because that is the quantity the framework names, and the
# gap between the two is 3 rows here so the branch does not turn on the choice.
p1_rate_ex_tail = (p1_rates["y_nulls"] - p1_rates["tail"]) / p1_n
print(f"rate excluding the structural tail    {100 * p1_rate_ex_tail:.3f}%")

p1_override = verdict(
    p1_rates["y_null_rate"] > P1_MISSINGNESS_BAN_THRESHOLD,
    f"Step 4 firewall: target missingness {100 * p1_rates['y_null_rate']:.3f}% vs the "
    f"{100 * P1_MISSINGNESS_BAN_THRESHOLD:.0f}% threshold -- the OVERRIDE does NOT fire, so the "
    f"framework's default (median impute) is the branch that stands")

rows                                  4,320
target nulls                          19  (0.440%)
  of which last bar of a symbol       3  (structural: no successor)
  of which the bar itself is empty    8  (NULL close, Step 3 flagged)
  of which the NEXT bar is empty      8  (lead() pulls the null back)
Step 3 is_missing_bar rate            0.185%  (under-counts the target: one-bar flag, two-bar target)

rate excluding the structural tail    0.370%
N/A      -- Step 4 firewall: target missingness 0.440% vs the 5% threshold -- the OVERRIDE does NOT fire, so the framework's default (median impute) is the branch that stands


### b. Handle it — the branch that actually fires

The measured rate is well under the firewall, so this run exercises the branch the framework
never demonstrates: median imputation is **permitted**. It is applied to the feature columns.

It is **not** applied to the target, and that is a deviation the framework does not authorise.
Its reasoning for complete-case at 8.3% is distributional (*"filling these 66 rows with a
median … would artificially compress the distribution's tails"*); the reason here is stronger
and different in kind. A null target on the last bar of a symbol is not an unobserved value —
there is no successor bar in the universe to observe. Filling it with a median return
fabricates a future. The last bar of each symbol is therefore dropped, not filled.

In [23]:
# THE CAST BOUNDARY. Everything above is decimal(18,8) and exact; pyspark.ml is Double/Float
# only -- Imputer, VectorAssembler, VarianceThresholdSelector and LinearRegression all reject
# DecimalType or silently widen it. So the cast happens HERE, once, at the entrance to the ml
# package, and never in an aggregation. p1_ret1, p1_y and p1_log_qv crossed early because log()
# has no decimal form, and each of those computed its argument in decimal first.
P1_BASE_X = ["open", "high", "low", "close",
             "volume", "quote_volume", "n_ticks",
             "taker_buy_qty", "taker_buy_quote_qty",
             "all_best_match", "is_missing_bar",
             "p1_ret1", "p1_range_bp", "p1_body_bp", "p1_imbalance", "p1_log_qv"]

# first_trade_id / last_trade_id / bar_us are identifiers and are excluded, not pruned: an id is
# not a low-variance column, it is a column whose variance is meaningless. `symbol` is a
# categorical string and Path 1 deletes those outright (see Step 8).
p1_dbl = p1.select("symbol", "bar_us", "is_last_bar",
                   *[func.col(c).cast("double").alias(c) for c in P1_BASE_X],
                   func.col("p1_y").cast("double").alias("p1_y"))

# Complete-case on the TARGET only. na.drop(subset=["p1_y"]) is the framework's operation; the
# row leaves the matrix entirely rather than being blanked or flagged-and-kept.
p1_tail = p1_dbl.filter(func.col("p1_y").isNull() & (func.col("is_last_bar") == 1))
print("last bar of each symbol -- NULL target, dropped rather than filled:")
p1_tail.groupBy("symbol").agg(func.count("*").alias("rows"),
                              func.max("bar_us").alias("bar_us")).orderBy("symbol").show()

p1_cc = p1_dbl.na.drop(subset=["p1_y"]).cache()
p1_cc_n = p1_cc.count()
print(f"complete-case on the target: {p1_n:,} -> {p1_cc_n:,} rows "
      f"({p1_n - p1_cc_n} evacuated)")
print()

# Now the default branch, on the features. The residual X nulls are p1_ret1 on the FIRST bar of
# each symbol (no predecessor) and on every bar whose predecessor was empty.
p1_x_nulls = p1_cc.select([func.sum(func.col(c).isNull().cast("int")).alias(c)
                           for c in P1_BASE_X]).first().asDict()
p1_dirty = {c: n for c, n in p1_x_nulls.items() if n}
print(f"feature-column nulls surviving the complete-case cut: {p1_dirty}")

# This is the transformer the OVERRIDE would have banned. It runs because the rate is under the
# firewall -- and it is fitted, not merely named, so the medians it writes are real.
p1_imputer = Imputer(strategy="median", inputCols=P1_BASE_X, outputCols=P1_BASE_X)
p1_imputer_model = p1_imputer.fit(p1_cc)
p1_step4 = p1_imputer_model.transform(p1_cc).cache()

p1_medians = dict(zip(P1_BASE_X, p1_imputer_model.surrogateDF.first()))
for c, n in p1_dirty.items():
    # Scientific notation, because a log return's median is small enough that fixed notation
    # would hide whether it is a small number or an exact zero. It prints as exactly
    # +0.000000e+00, and that is not a rounding artefact: the modal 5-second bar in this window
    # closes where it opened, so the median one-bar return IS zero. Worth saying out loud --
    # a zero written into a return column is indistinguishable from an observed flat bar, so
    # this particular median fill is unrecoverable after the fact.
    print(f"   {c:<14} {n:>3} cells <- median {p1_medians[c]:+.6e}")
print(f"cells filled: {sum(p1_dirty.values())} of {p1_cc_n * len(P1_BASE_X):,}")
print()

verdict(True, f"Step 4 Path 1: complete-case on y ({p1_n:,} -> {p1_cc_n:,} rows), median impute "
              f"on X ({sum(p1_dirty.values())} cells) -- the sub-5% branch the framework states "
              f"but never demonstrates")
verdict(False, "Step 4 flag pairing: Step 3's is_missing_bar marks the EMPTY bar, but the rows "
               "the median fill touches are that bar's SUCCESSORS -- the framework's "
               "flag-then-impute pairing does not cover a lagged feature", banned=True)

last bar of each symbol -- NULL target, dropped rather than filled:


+-------+----+----------------+
| symbol|rows|          bar_us|
+-------+----+----------------+
|BTCUSDT|   1|1735696795000000|
|ETHUSDT|   1|1735696795000000|
|SOLUSDT|   1|1735696795000000|
+-------+----+----------------+



complete-case on the target: 4,320 -> 4,301 rows (19 evacuated)



feature-column nulls surviving the complete-case cut: {'p1_ret1': 11}


   p1_ret1         11 cells <- median +0.000000e+00
cells filled: 11 of 68,816

APPLIES  -- Step 4 Path 1: complete-case on y (4,320 -> 4,301 rows), median impute on X (11 cells) -- the sub-5% branch the framework states but never demonstrates


BANNED   -- Step 4 flag pairing: Step 3's is_missing_bar marks the EMPTY bar, but the rows the median fill touches are that bar's SUCCESSORS -- the framework's flag-then-impute pairing does not cover a lagged feature


False

## 5. Diagnostics — Path 1 (APPLIES)

> *"Univariate topological density profiles for operator audit. t-SNE sandbox (read-only, async)"*

Read-only. The LaTeX is explicit: *"What changed: Nothing inside the table."* The output is a
log line, not a matrix change, and the row count is asserted identical on the way out.

In [24]:
p1_before_rows, p1_before_cols = p1_step4.count(), len(p1_step4.columns)

# Univariate density profile of the target. Skewness and kurtosis are the two shape statistics
# Spark ships natively; approxQuantile gives the tails without a full sort.
p1_shape = p1_step4.select(
    func.count("p1_y").alias("n"),
    func.avg("p1_y").alias("mean"),
    func.stddev("p1_y").alias("sd"),
    func.skewness("p1_y").alias("skew"),
    func.kurtosis("p1_y").alias("kurt"),
    func.min("p1_y").alias("min"),
    func.max("p1_y").alias("max"),
).first()

print("target: next-bar log return, all symbols pooled")
print(f"   n {p1_shape['n']:,} | mean {1e4 * p1_shape['mean']:+.4f} bp | "
      f"sd {1e4 * p1_shape['sd']:.4f} bp")
print(f"   skewness {p1_shape['skew']:+.4f} | excess kurtosis {p1_shape['kurt']:+.4f}")
print(f"   range [{1e4 * p1_shape['min']:+.2f}, {1e4 * p1_shape['max']:+.2f}] bp")
print()

p1_q = p1_step4.approxQuantile("p1_y", [0.01, 0.25, 0.5, 0.75, 0.99], 0.0)
print("   percentiles (bp)  " + "  ".join(
    f"p{int(p * 100):02d} {1e4 * v:+.3f}" for p, v in zip([0.01, 0.25, 0.5, 0.75, 0.99], p1_q)))
print()

# Per-symbol, because a pooled skewness over three price scales is an average of three different
# distributions. The flat share is the mass at exactly zero -- the fraction of bars whose close
# equals the next bar's close.
p1_by_symbol = (p1_step4.groupBy("symbol").agg(
    func.count("*").alias("n"),
    (func.avg((func.col("p1_y") == 0).cast("double")) * 100).alias("flat_pct"),
    (func.stddev("p1_y") * 1e4).alias("sd_bp"),
    func.skewness("p1_y").alias("skew"),
    func.kurtosis("p1_y").alias("kurt"),
).orderBy("symbol"))
p1_by_symbol.show(truncate=False)

# The operator-audit line the framework asks for: name the extreme observation rather than
# smoothing it away. Its NexusMart equivalent is "High-value outlier detected at session_884."
p1_extreme = p1_step4.orderBy(func.abs(func.col("p1_y")).desc()).first()
print(f"Target Skewness Flagged: extreme move {1e4 * p1_extreme['p1_y']:+.2f} bp on "
      f"{p1_extreme['symbol']} at bar_us {p1_extreme['bar_us']} "
      f"({abs(p1_extreme['p1_y'] / p1_shape['sd']):.1f} sd)")
print()

# The flat mass is a property of THIS two-hour window, not of the month. 00:00-02:00 on 1 Jan is
# the quietest stretch of the year; the full-month pass over all 340,971,834 ticks put the flat
# class at 5s at 14.37% (SOL) / 13.25% (ETH) / 20.04% (BTC). Quoting the sample as if it were
# the month is the exact error the calendar override in the entryway exists to prevent.
verdict(True, "Step 5 diagnostics: read-only univariate density profile emitted for the target; "
              "the flat share above is this quiet two-hour window, NOT the month")
verdict(False, "Step 5 t-SNE sandbox: no t-SNE in pyspark.ml at all, and the framework "
               "hard-quarantines it to an async read-only operator plot -- it never feeds "
               "K-Means, Hierarchical or DBSCAN, so nothing downstream here may consume it",
        banned=True)

assert (p1_step4.count(), len(p1_step4.columns)) == (p1_before_rows, p1_before_cols)
print(f"read-only confirmed: {p1_before_rows:,} rows x {p1_before_cols} columns, unchanged")

target: next-bar log return, all symbols pooled
   n 4,301 | mean +0.0259 bp | sd 1.6641 bp
   skewness +0.2727 | excess kurtosis +5.4650
   range [-9.17, +14.41] bp

   percentiles (bp)  p01 -4.702  p25 -0.523  p50 +0.000  p75 +0.523  p99 +4.714



+-------+----+------------------+------------------+------------------+------------------+
|symbol |n   |flat_pct          |sd_bp             |skew              |kurt              |
+-------+----+------------------+------------------+------------------+------------------+
|BTCUSDT|1439|27.58860319666435 |1.1824311985988742|0.19624325529282  |11.13956624123837 |
|ETHUSDT|1429|19.034289713086075|1.414491812482836 |0.8564143254454857|8.979869452065905 |
|SOLUSDT|1433|18.84159106769016 |2.2168567124114062|0.0673553662493199|1.9433205037526298|
+-------+----+------------------+------------------+------------------+------------------+

Target Skewness Flagged: extreme move +14.41 bp on ETHUSDT at bar_us 1735691480000000 (8.7 sd)

APPLIES  -- Step 5 diagnostics: read-only univariate density profile emitted for the target; the flat share above is this quiet two-hour window, NOT the month


BANNED   -- Step 5 t-SNE sandbox: no t-SNE in pyspark.ml at all, and the framework hard-quarantines it to an async read-only operator plot -- it never feeds K-Means, Hierarchical or DBSCAN, so nothing downstream here may consume it


read-only confirmed: 4,301 rows x 20 columns, unchanged


## 6. Topology — Path 1 (APPLIES)

> *"Global cross-correlation matrix — preserves multi-feature relationships"*

Also read-only. *"No value changes. The system exports an internal mathematical dependency
schema to guide downstream feature regularizers."* The schema is a log line here, not a Python
object consumed later — the forward-pass rule forbids Step 8 or Step 9 from reading it back.

In [25]:
# Correlation.corr wants one assembled vector. y rides along as the last element so the same
# single pass gives both the X-X dependency structure and every feature's marginal relationship
# with the target.
p1_corr_cols = P1_BASE_X + ["p1_y"]
p1_corr_asm = VectorAssembler(inputCols=p1_corr_cols, outputCol="p1_corr_vec",
                              handleInvalid="error")
p1_corr_m = Correlation.corr(p1_corr_asm.transform(p1_step4), "p1_corr_vec", "pearson")
p1_corr_a = p1_corr_m.collect()[0][0].toArray()

# The dependency schema, half one: the collinear cluster. A raw OHLC block on a 5-second grid is
# four measurements of the same price seconds apart, so this is expected -- and it is precisely
# what an L1 penalty resolves arbitrarily, keeping whichever of four near-identical columns the
# solver reaches first. Step 9 is told about it here and cannot read it back.
print("|r| >= 0.90 among X (the collinear cluster Step 9 will have to break):")
for i in range(len(P1_BASE_X)):
    for j in range(i + 1, len(P1_BASE_X)):
        r = p1_corr_a[i][j]
        if not math.isnan(r) and abs(r) >= 0.90:
            print(f"   {P1_BASE_X[i]:<20} {P1_BASE_X[j]:<20} {r:+.4f}")
print()

# Half two: marginal correlation with the target. NaN appears for the two constant columns --
# a zero-variance column has an undefined correlation, which is Step 8's finding arriving one
# step early and by accident.
print("marginal corr with the next-bar target:")
p1_y_corr = sorted(((P1_BASE_X[i], p1_corr_a[i][-1]) for i in range(len(P1_BASE_X))),
                   key=lambda kv: -abs(kv[1]) if not math.isnan(kv[1]) else 1)
for name, r in p1_y_corr:
    print(f"   {name:<20} {'nan (zero variance)' if math.isnan(r) else f'{r:+.4f}'}")
print()

# The imbalance sign check, re-measured on this frame. The entryway verified
# corr(imbalance, CONTEMPORANEOUS bar return) = +0.5096 on 2M ticks. Against the NEXT bar it is
# near zero and that is the correct, boring answer -- order-flow imbalance is a same-bar
# identity, not a forecast. A large positive number here would mean the target is leaking.
p1_same_bar = p1_step4.select(func.corr("p1_imbalance", "p1_body_bp")).first()[0]
print(f"corr(imbalance, SAME-bar body_bp)  {p1_same_bar:+.4f}   (entryway: +0.5096 on 2M ticks)")
print(f"corr(imbalance, NEXT-bar target)   "
      f"{p1_corr_a[P1_BASE_X.index('p1_imbalance')][-1]:+.4f}   (near zero is the honest answer)")

verdict(True, f"Step 6 topology: {len(p1_corr_cols)}x{len(p1_corr_cols)} Pearson matrix exported "
              f"as a dependency schema; no value in the matrix changed")

|r| >= 0.90 among X (the collinear cluster Step 9 will have to break):
   open                 high                 +1.0000
   open                 low                  +1.0000
   open                 close                +1.0000
   high                 low                  +1.0000
   high                 close                +1.0000
   low                  close                +1.0000
   quote_volume         taker_buy_quote_qty  +0.9176
   p1_ret1              p1_body_bp           +0.9905

marginal corr with the next-bar target:
   p1_imbalance         +0.0930
   p1_body_bp           +0.0747
   p1_ret1              +0.0738
   taker_buy_qty        +0.0360
   volume               +0.0307
   p1_range_bp          +0.0303
   taker_buy_quote_qty  +0.0213
   n_ticks              +0.0184
   p1_log_qv            +0.0131
   open                 -0.0103
   low                  -0.0103
   high                 -0.0103
   close                -0.0103
   quote_volume         +0.0089
   all_best_matc

True

## 7. Feature Engineering — Path 1 (APPLIES)

> *"Deterministic mathematical cross-products"*

The word **deterministic** is load-bearing: no learned, sampled or stochastic construction is
permitted on Path 1, because Path 1's output has to stay auditable in real units. Four named
products of columns that already exist. `pyspark.ml.feature.Interaction` and
`PolynomialExpansion` would produce the same thing; explicit `withColumn` is used so each
column has a name a reviewer can read, which is the whole point of the constraint.

In [26]:
p1_step7 = (p1_step4
            # order-flow pressure conditional on direction
            .withColumn("p1_x_imb_ret", func.col("p1_imbalance") * func.col("p1_ret1"))
            # volatility conditional on participation
            .withColumn("p1_x_range_qv", func.col("p1_range_bp") * func.col("p1_log_qv"))
            # pressure conditional on volatility
            .withColumn("p1_x_imb_range", func.col("p1_imbalance") * func.col("p1_range_bp"))
            # momentum conditional on trade count
            .withColumn("p1_x_ret_ticks", func.col("p1_ret1") * func.col("n_ticks")))

P1_CROSS = ["p1_x_imb_ret", "p1_x_range_qv", "p1_x_imb_range", "p1_x_ret_ticks"]
P1_X = P1_BASE_X + P1_CROSS

p1_step7.select(*P1_CROSS).summary("mean", "stddev", "min", "max").show(truncate=False)
verdict(True, f"Step 7 Path 1: column dimensions expand {len(P1_BASE_X)} -> {len(P1_X)}; four "
              f"deterministic products appended, nothing learned or sampled")
verdict(False, "Step 7 ceiling: interactions deeper than a manual pairwise product are a "
               "MODELLING escalation (the framework's ANN clause), not more Step 7 -- adding a "
               "learned interaction here would break the deterministic constraint", banned=True)

+-------+---------------------+------------------+------------------+--------------------+
|summary|p1_x_imb_ret         |p1_x_range_qv     |p1_x_imb_range    |p1_x_ret_ticks      |
+-------+---------------------+------------------+------------------+--------------------+
|mean   |3.349749024593404E-5 |12.381728452465499|0.6660987791272279|0.003221296064348492|
|stddev |1.0965781600383526E-4|17.42896647687578 |1.137135997391308 |0.06663201339790166 |
|min    |-4.674163373588316E-4|0.0               |0.0               |-1.3332343093384387 |
|max    |0.0014374462521556258|244.69541846861273|16.95143714947022 |2.4124735760709144  |
+-------+---------------------+------------------+------------------+--------------------+

APPLIES  -- Step 7 Path 1: column dimensions expand 16 -> 20; four deterministic products appended, nothing learned or sampled


BANNED   -- Step 7 ceiling: interactions deeper than a manual pairwise product are a MODELLING escalation (the framework's ANN clause), not more Step 7 -- adding a learned interaction here would break the deterministic constraint


False

## 8. Pruning — Path 1 (APPLIES)

> *"VarianceThreshold filter — bypass categorical dummy strings"*

Two operations in one step. The variance filter runs over the numerics; the categorical
strings are handled by exclusion. The two sources disagree on that second half — the HTML says
*"bypass"*, the LaTeX says *"marketing text string variables are entirely deleted"* and its
matrix trace shows `ad_creative_id` gone from the After-Step-8 table. **Delete is taken as the
reading**, so `symbol` does not enter the matrix and is not one-hot encoded.

The threshold is set to exactly `0.0`. On NexusMart's toy matrix every column is O(1)–O(400)
and any threshold behaves; here `quote_volume` is O(10⁵) and `p1_y` is O(10⁻⁴), so a positive
raw-variance threshold deletes every return column before it deletes anything uninformative.
The framework forbids fixing this by scaling first — Step 10 comes after Step 8 and the pass
does not loop. `0.0` keeps the step to what the LaTeX actually demonstrates: removing columns
that are genuinely constant.

In [27]:
p1_asm = VectorAssembler(inputCols=P1_X, outputCol="p1_features", handleInvalid="error")
p1_assembled = p1_asm.transform(p1_step7)

# varianceThreshold=0.0 removes features whose sample variance is <= 0, i.e. exactly the
# constants. Anything higher would be a scale judgement the framework does not license.
p1_vts = VarianceThresholdSelector(featuresCol="p1_features", outputCol="p1_selected",
                                   varianceThreshold=0.0)
p1_vts_model = p1_vts.fit(p1_assembled)
p1_step8 = p1_vts_model.transform(p1_assembled).cache()

p1_kept = [P1_X[i] for i in p1_vts_model.selectedFeatures]
p1_dropped = [c for c in P1_X if c not in p1_kept]
print(f"kept    {len(p1_kept)}/{len(P1_X)}")
print(f"dropped {p1_dropped}")
print()

# Why each one went, measured rather than asserted.
for c in p1_dropped:
    stats = p1_step7.select(func.var_samp(c).alias("v"),
                            func.countDistinct(c).alias("d"),
                            func.first(c).alias("f")).first()
    print(f"   {c:<16} var {stats['v']:.1f}  distinct {stats['d']}  value {stats['f']}")
print()

# all_best_match is the framework's constant_metric, verbatim: is_best_match was True in all
# 340,971,834 ticks of the month, so bool_and is True in every bar. The entryway carried it
# through on purpose, because pruning is Step 8 and Step 8 is path-isolated -- it is BANNED on
# Path 3, so the shared entryway had no right to make this decision on Path 3's behalf.
#
# is_missing_bar is more interesting, and it is an internal inconsistency in the framework worth
# stating out loud. Step 3's missingness flag is supposed to survive Step 4 and "become a
# predictive feature" on Path 1. But every row it flags has a NULL close, so every row it flags
# was evacuated by Step 4's complete-case cut, so the column is a flat 0 by the time Step 8
# measures it -- and Step 8 deletes it. The LaTeX trace quietly loses the same column between
# Step 7 and Step 8 without comment. Path 1's Step 4 and Path 1's Step 3 flag cannot both stand.
verdict(True, f"Step 8 Path 1: VarianceThreshold(0.0) removed {len(p1_dropped)} zero-variance "
              f"columns {p1_dropped}; `symbol` deleted as a categorical string, never encoded")
verdict(False, "Step 8 scale ordering: a positive raw-variance threshold would delete every "
               "return column (O(1e-4)) before touching quote_volume (O(1e5)); Step 10 fixes "
               "the scales and runs AFTER, and the pass does not loop", banned=True)

kept    18/20
dropped ['all_best_match', 'is_missing_bar']



   all_best_match   var 0.0  distinct 1  value 1.0


   is_missing_bar   var 0.0  distinct 1  value 0.0

APPLIES  -- Step 8 Path 1: VarianceThreshold(0.0) removed 2 zero-variance columns ['all_best_match', 'is_missing_bar']; `symbol` deleted as a categorical string, never encoded


BANNED   -- Step 8 scale ordering: a positive raw-variance threshold would delete every return column (O(1e-4)) before touching quote_volume (O(1e5)); Step 10 fixes the scales and runs AFTER, and the pass does not loop


False

## 9. Regularisation — Path 1 (APPLIES)

> *"Cross-validated Elastic Net — discards noise columns"*

Step 8 was unsupervised and target-blind — it only ever looked at variance. Step 9 is
supervised: columns whose coefficient the L1 penalty drives to exactly zero are the ones that
vary but do not predict.

**The framework's cross-validation is unsafe here and its silence is the problem.** Spark's
`CrossValidator` uses random folds; on ordered bars that trains on the future and tests on the
past, and bars adjacent in time are near-duplicates, so a random split puts a row and its own
neighbours on both sides of the boundary. NexusMart's rows are exchangeable sessions and bars
are not. `foldCol` is used instead, with folds cut as contiguous time blocks keyed on `bar_us`
so all three symbols of one instant stay together. That is an improvement, not a fix: block
fold 0 is still validated against a model trained on folds 1–4, which are later. A true
expanding window needs a manual loop and is outside both the framework and this cell.

In [28]:
P1_FOLDS = 5

# Folds cut on bar_us, not by ntile over rows: keying on the timestamp keeps the three symbols
# of a single instant in the same fold. Three correlated rows split across the boundary is the
# same leak as a random fold, just smaller.
p1_span = p1_step8.select(func.min("bar_us").alias("lo"), func.max("bar_us").alias("hi")).first()
p1_width = (p1_span["hi"] - p1_span["lo"] + 1) / P1_FOLDS
p1_cv_in = p1_step8.withColumn(
    "p1_fold",
    func.least(func.lit(P1_FOLDS - 1),
               func.floor((func.col("bar_us") - func.lit(p1_span["lo"])) / func.lit(p1_width))
               ).cast("int"))
p1_cv_in.groupBy("p1_fold").agg(func.count("*").alias("rows"),
                                func.min("bar_us").alias("from_us")).orderBy("p1_fold").show()

# standardization=True is Spark's DEFAULT and it is set explicitly because it silently
# neutralises the Step 8/Step 10 ordering complaint above: Spark standardises internally for the
# optimiser so the L1 penalty is applied on a common scale, then reports coefficients back in
# raw units. The framework's forward pass would have left the penalty scale-dependent; the
# implementation quietly does not. Stating it beats inheriting it.
p1_lr = LinearRegression(featuresCol="p1_selected", labelCol="p1_y",
                         standardization=True, maxIter=100)
p1_grid = (ParamGridBuilder()
           .addGrid(p1_lr.regParam, [1e-8, 1e-6, 1e-4])
           .addGrid(p1_lr.elasticNetParam, [0.25, 0.5, 1.0])   # 1.0 is pure L1, 0.0 pure ridge
           .build())
p1_cv = CrossValidator(estimator=p1_lr, estimatorParamMaps=p1_grid,
                       evaluator=RegressionEvaluator(labelCol="p1_y", metricName="rmse"),
                       # numFolds is NOT implied by foldCol -- it defaults to 3 and the fit dies
                       # with "Fold number must be in range [0, 3), but got 3" only after the
                       # folds have already been materialised.
                       numFolds=P1_FOLDS, foldCol="p1_fold", parallelism=1, seed=42)
p1_cv_model = p1_cv.fit(p1_cv_in)
p1_best = p1_cv_model.bestModel

print(f"selected  regParam {p1_best.getRegParam():g} | "
      f"elasticNetParam {p1_best.getElasticNetParam()} "
      f"({'pure L1' if p1_best.getElasticNetParam() == 1.0 else 'L1/L2 mix'})")
print(f"CV RMSE   {min(p1_cv_model.avgMetrics):.8f}  (target sd "
      f"{p1_shape['sd']:.8f}) over {P1_FOLDS} time-blocked folds")
print()

p1_coefs = list(p1_best.coefficients)
p1_zero = [n for n, c in zip(p1_kept, p1_coefs) if c == 0.0]
p1_live = sorted(((n, c) for n, c in zip(p1_kept, p1_coefs) if c != 0.0),
                 key=lambda kv: -abs(kv[1]))
print(f"non-zero coefficients: {len(p1_live)}/{len(p1_kept)}")
print(f"shrunk to exactly 0.0: {p1_zero or 'none'}")
print()
print("top features by |coefficient| (raw units, y in log-return):")
for name, c in p1_live[:6]:
    print(f"   {name:<20} {c:+.6e}")

verdict(len(p1_zero) > 0,
        f"Step 9 Path 1: CV Elastic Net kept {len(p1_live)}/{len(p1_kept)} columns; "
        f"{len(p1_zero)} shrunk to exactly zero")

+-------+----+----------------+
|p1_fold|rows|         from_us|
+-------+----+----------------+
|      0| 864|1735689600000000|
|      1| 860|1735691040000000|
|      2| 861|1735692480000000|
|      3| 860|1735693915000000|
|      4| 856|1735695355000000|
+-------+----+----------------+



2026-09-03 23:58:29,701 INFO py4j.clientserver - Closing down clientserver connection


2026-09-03 23:58:29,703 INFO py4j.clientserver - Closing down clientserver connection


selected  regParam 1e-06 | elasticNetParam 1.0 (pure L1)
CV RMSE   0.00016487  (target sd 0.00016641) over 5 time-blocked folds

non-zero coefficients: 6/18
shrunk to exactly 0.0: ['open', 'high', 'low', 'close', 'quote_volume', 'n_ticks', 'taker_buy_qty', 'taker_buy_quote_qty', 'p1_ret1', 'p1_range_bp', 'p1_log_qv', 'p1_x_imb_range']

top features by |coefficient| (raw units, y in log-return):
   p1_x_imb_ret         +1.319601e-02
   p1_imbalance         +3.151237e-05
   p1_x_ret_ticks       -1.119101e-05
   p1_body_bp           +2.685824e-06
   p1_x_range_qv        +1.394276e-07
   volume               +1.359882e-08
APPLIES  -- Step 9 Path 1: CV Elastic Net kept 6/18 columns; 12 shrunk to exactly zero


True

## 10. Scaling — Path 1 (LIMIT)

> *"Linear scalers only (StandardScaler) — keeps regression coefficients interpretable"*

`StandardScaler` centres and divides by σ, which is affine, so the spacing of the data is
preserved and a coefficient on a scaled feature multiplies straight back into real units.
`QuantileTransformer` would turn a feature value into a percentile rank and the coefficient
would stop meaning anything measurable. On Path 1 the unit is basis points of next-bar return,
and a reviewer has to be able to read the sentence.

The LIMIT costs nothing to obey in Spark — it ships neither `PowerTransformer` nor
`QuantileTransformer`, so the blacklisted transforms are not available to apply by accident.
That is luck, not compliance, and Path 2's Step 10 is where the same gap becomes a real
problem.

In [29]:
# Step 9 SELECTED; it did not shrink the vector. A LinearRegressionModel reports a zero
# coefficient, it does not delete a column, so the survivors are re-assembled here -- the
# framework is explicit that the zeroed column "is dropped", and carrying 12 dead columns into
# the scaler would leave Part II a matrix whose width disagrees with the Step 9 ledger line.
P1_FINAL = [c for c in p1_kept if c in dict(p1_live)]
p1_final_asm = VectorAssembler(inputCols=P1_FINAL, outputCol="p1_final_features",
                               handleInvalid="error")
p1_step9 = p1_final_asm.transform(p1_cv_in)

p1_scaler = StandardScaler(inputCol="p1_final_features", outputCol="p1_scaled",
                           withMean=True, withStd=True)
p1_scaler_model = p1_scaler.fit(p1_step9)
p1_step10 = p1_scaler_model.transform(p1_step9)

# withMean=True is NOT the default and is the half that makes the centring real. It densifies
# sparse vectors, which would matter on a wide one-hot matrix -- Path 2's problem, not Path 1's,
# because Path 1 deleted its categoricals at Step 8.
p1_mu = list(p1_scaler_model.mean)
p1_sigma = list(p1_scaler_model.std)

print(f"{'feature':<20} {'mu':>16} {'sigma':>16}")
for name, mu, sd in zip(P1_FINAL, p1_mu, p1_sigma):
    print(f"{name:<20} {mu:>16.6g} {sd:>16.6g}")
print()

# THE POINT OF THE LIMIT. sigma is retained, so a Step 9 coefficient can be back-translated into
# the unit a non-technical reviewer reads. y is a log return, so 1e4 * y is basis points.
print("back-translation (this is what a non-linear scaler would destroy):")
for name, c in p1_live:
    sd = p1_sigma[P1_FINAL.index(name)]
    print(f"   +1 sd of {name} (sd = {sd:.6g}) -> {1e4 * c * sd:+.4f} bp of next-bar return")
print()

p1_scaled_0 = vector_to_array("p1_scaled")[0]
p1_check = p1_step10.select(func.avg(p1_scaled_0).alias("m"),
                            func.stddev(p1_scaled_0).alias("s")).first()
print(f"scaled {P1_FINAL[0]}: mean {p1_check['m']:+.9f}, sd {p1_check['s']:.6f}  "
      f"(0 and 1 by construction)")

verdict(True, f"Step 10 Path 1 LIMIT: StandardScaler(withMean=True, withStd=True) over "
              f"{len(P1_FINAL)} columns; mu and sigma retained so coefficients stay in bp")
verdict(False, "Step 10 non-linear transforms: QuantileTransformer / PowerTransformer are "
               "blacklisted on Path 1 -- a percentile rank has no basis-point meaning and the "
               "model must stay explainable to a non-technical reviewer", banned=True)

feature                            mu            sigma
volume                        44.7167          190.095
p1_body_bp                  0.0254038          1.63675
p1_imbalance                 0.563427         0.379399
p1_x_imb_ret              3.34975e-05      0.000109658
p1_x_range_qv                 12.3817           17.429
p1_x_ret_ticks              0.0032213         0.066632

back-translation (this is what a non-linear scaler would destroy):
   +1 sd of p1_x_imb_ret (sd = 0.000109658) -> +0.0145 bp of next-bar return
   +1 sd of p1_imbalance (sd = 0.379399) -> +0.1196 bp of next-bar return
   +1 sd of p1_x_ret_ticks (sd = 0.066632) -> -0.0075 bp of next-bar return
   +1 sd of p1_body_bp (sd = 1.63675) -> +0.0440 bp of next-bar return
   +1 sd of p1_x_range_qv (sd = 17.429) -> +0.0243 bp of next-bar return
   +1 sd of volume (sd = 190.095) -> +0.0259 bp of next-bar return



scaled volume: mean -0.000000000, sd 1.000000  (0 and 1 by construction)
APPLIES  -- Step 10 Path 1 LIMIT: StandardScaler(withMean=True, withStd=True) over 6 columns; mu and sigma retained so coefficients stay in bp


BANNED   -- Step 10 non-linear transforms: QuantileTransformer / PowerTransformer are blacklisted on Path 1 -- a percentile rank has no basis-point meaning and the model must stay explainable to a non-technical reviewer


False

### Path 1 ledger

In [30]:
print(f"{'stage':<34} {'rows':>7} {'X cols':>7}")
print(f"{'entryway output (Steps 1-3)':<34} {p1_n:>7,} {'-':>7}")
print(f"{'post-fork, y + base features':<34} {p1_n:>7,} {len(P1_BASE_X):>7}")
print(f"{'4  complete-case on y':<34} {p1_cc_n:>7,} {len(P1_BASE_X):>7}")
print(f"{'5  diagnostics (read-only)':<34} {p1_cc_n:>7,} {len(P1_BASE_X):>7}")
print(f"{'6  topology (read-only)':<34} {p1_cc_n:>7,} {len(P1_BASE_X):>7}")
print(f"{'7  cross-products':<34} {p1_cc_n:>7,} {len(P1_X):>7}")
print(f"{'8  VarianceThreshold(0.0)':<34} {p1_cc_n:>7,} {len(p1_kept):>7}")
print(f"{'9  CV Elastic Net':<34} {p1_cc_n:>7,} {len(P1_FINAL):>7}")
print(f"{'10 StandardScaler (LIMIT)':<34} {p1_cc_n:>7,} {len(P1_FINAL):>7}")
print()
print("Path 1 hands its scaled matrix to Part II: UMAP/autoencoder -> cohort labels -> CUPAC")
print("-> the Difference Engine, and then all four gates. Gate 1 will see n = "
      f"{p1_cc_n:,}, not {p1_n:,} -- a refinery decision with a gate-level consequence.")

stage                                 rows  X cols
entryway output (Steps 1-3)          4,320       -
post-fork, y + base features         4,320      16
4  complete-case on y                4,301      16
5  diagnostics (read-only)           4,301      16
6  topology (read-only)              4,301      16
7  cross-products                    4,301      20
8  VarianceThreshold(0.0)            4,301      18
9  CV Elastic Net                    4,301       6
10 StandardScaler (LIMIT)            4,301       6

Path 1 hands its scaled matrix to Part II: UMAP/autoencoder -> cohort labels -> CUPAC
-> the Difference Engine, and then all four gates. Gate 1 will see n = 4,301, not 4,320 -- a refinery decision with a gate-level consequence.


# Part 3 — Path 2: The Categorical Class Stream

The fork fires the moment data exits Step 3. It reads the geometry of the target variable and
routes: a real-valued continuous decimal goes to Path 1, a class flag goes to Path 2, an
un-batched real-time stream goes to Path 3. This part follows the Path 2 lane, Steps 4 to 10.

**The target.** Bars have no experiment and no assigned `y` — the framework offers no guidance
on manufacturing one, so nominating it is this notebook's decision and not the framework's. The
Path 2 target here is the **direction class of the next bar's close-to-close change**, k=3:

```
delta = close[t+1] - close[t]      up: delta > 0     flat: delta == 0     down: delta < 0
```

`flat` is the whole reason this part exists. A two-class up/down target is available at any bar
width; a genuine three-class target only survives while a material share of consecutive bars
close at exactly the same price, and that share is a function of the bar interval. It is why 5s
was chosen as the interval in the entryway, and section 0b below measures the collapse.

**What Path 2 gets, per the framework's step grid:**

| Step | Verdict | Operation |
| --- | --- | --- |
| 4 Imputation | APPLIES | median impute numerics; missing categories to `SYSTEM_STATE_UNKNOWN` |
| 5 Diagnostics | APPLIES | class-balance diagnostics; interaction correlation matrices |
| 6 Topology | APPLIES | cross-correlation matrix + cyclical sine/cosine coordinate spaces |
| 7 Feature eng | APPLIES | cyclical coordinate engineering; interaction cross-products |
| 8 Pruning | **ENFORCE** | One-Hot encoding **before** variance pruning |
| 9 Regularisation | APPLIES | cross-validated Elastic Net regularisation loop |
| 10 Scaling | APPLIES | quantile search across Standard / Power / Quantile transforms |

Step 5 changes nothing. It is read-only in the framework's own trace ("Nothing inside the
table"), and the honest demonstration of a read-only step is an identical before/after frame
beside a diagnostics log. Step 6 is read-only on Path 1 ("No value changes") but **not** on
Path 2, where the same cell also mandates cyclical coordinate spaces — those are columns, and
the grid asks for them here rather than at Step 7.

## 0a. The Fork — reading y's geometry

`y` is built here, before Step 4, because the framework is explicit that the fork fires
"immediately after Step 3" and "before any imputation or spatial modifications" — the path
decides how missing data is handled, so the target cannot be a product of imputation. Every
label below therefore comes from an **observed** close on both sides of the transition.

In [31]:
import math

from pyspark.sql import Window
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (OneHotEncoder, StandardScaler, StringIndexer,
                                VarianceThresholdSelector, VectorAssembler)
from pyspark.ml.functions import vector_to_array
from pyspark.ml.stat import Correlation

# One bar series per symbol. partitionBy("symbol") is not cosmetic: without it lead() would run
# a single unpartitioned window over all 4,320 rows and hand BTC's last bar ETH's first close.
P2_BAR_ORDER = Window.partitionBy("symbol").orderBy("bar_us")

# The subtraction stays in decimal(18,8). Casting close to double first would make the flat class
# an artefact of binary rounding rather than a measurement: two identical decimal prices are
# equal exactly, two doubles built from them are equal only most of the time, and the entire
# k=3 target hangs on that equality test.
p2_forked = (bars
             .withColumn("p2_next_close", func.lead("close").over(P2_BAR_ORDER))
             .withColumn("p2_delta", func.col("p2_next_close") - func.col("close"))
             .withColumn("p2_class",
                         func.when(func.col("p2_delta") > 0, "up")
                             .when(func.col("p2_delta") < 0, "down")
                             .when(func.col("p2_delta") == 0, "flat"))
             .cache())

p2_levels = sorted(r[0] for r in
                   p2_forked.select("p2_class").distinct().filter(func.col("p2_class").isNotNull()).collect())
p2_labelled = p2_forked.filter(func.col("p2_class").isNotNull()).count()
p2_unlabelled = p2_forked.count() - p2_labelled

print(f"y geometry: {p2_forked.schema['p2_class'].dataType.simpleString()}, "
      f"{len(p2_levels)} distinct levels {p2_levels}")
print(f"labelled rows: {p2_labelled:,} of {p2_forked.count():,}   unlabelled: {p2_unlabelled}")

# Where the 19 unlabelled rows come from, so the number is auditable rather than asserted.
p2_edges = p2_forked.filter(func.col("p2_class").isNull()).agg(
    func.sum("is_last_bar").alias("last_bar"),
    func.sum("is_missing_bar").alias("empty_bar")).first()
print(f"   window edge (is_last_bar):        {p2_edges['last_bar']}")
print(f"   empty bar, own close missing:     {p2_edges['empty_bar']}")
print(f"   predecessor of an empty bar:      "
      f"{p2_unlabelled - p2_edges['last_bar'] - p2_edges['empty_bar']}")

# A real-valued decimal would have routed to Path 1; a stream of un-batched impressions to
# Path 3. Three discrete levels is the Button geometry, so Path 2 it is.
verdict(True, "Fork: y is a 3-level discrete class label (up/flat/down) -> Path 2, categorical")

y geometry: string, 3 distinct levels ['down', 'flat', 'up']
labelled rows: 4,301 of 4,320   unlabelled: 19
   window edge (is_last_bar):        3
   empty bar, own close missing:     8
   predecessor of an empty bar:      8
APPLIES  -- Fork: y is a 3-level discrete class label (up/flat/down) -> Path 2, categorical


True

### The one thing the fork does NOT decide

The framework's Step 4 on Path 2 says "median impute numerics; assign missing categories to
`SYSTEM_STATE_UNKNOWN`". Both halves are about **X**. Nothing in the framework says what to do
with a row whose **y** was never observed, and the two available readings are both wrong:
median-imputing a class label is undefined, and assigning it `SYSTEM_STATE_UNKNOWN` invents a
fourth class the fork did not route on.

**Judgement call, not the framework's:** an unlabelled row cannot train a classifier, so the 19
rows above leave at Step 9, which is the first *supervised* step. They stay in the frame for
Steps 4 to 8, which are target-blind — Step 8's variance filter in particular is unsupervised
by the framework's own division of labour, so it is entitled to see all 4,320 rows.

## 0b. Why 5s — the flat class collapses at coarser intervals

The bar interval is not a display preference on Path 2, it is what decides whether the target
has three classes or two. Coarser bars are built here by taking the last observed close in each
coarse slot from the 5s bars already in hand, so this costs one shuffle over 4,320 rows and no
second pass over ticks.

In [32]:
P2_INTERVALS = [("5s", 5_000_000), ("15s", 15_000_000), ("1m", 60_000_000), ("5m", 300_000_000)]

p2_sweep = {}
for p2_label, p2_us in P2_INTERVALS:
    # max_by through expr(), not func.max_by: the Python wrapper lands in 3.3.0 and the SQL name
    # is older, which is the same rule the entryway follows for timestamp_micros and pmod.
    # Nulls are filtered rather than carried: the coarse close is "the last price actually
    # printed in this slot", which is an observation, where coalescing would be a Step 4 fill.
    # One consequence, stated because it makes the 5s row here disagree slightly with the target
    # built at the fork: dropping the empty bars STITCHES the transition across a gap, so ETH and
    # SOL get 1,434 / 1,436 transitions at 5s where the fork's stricter both-ends-observed rule
    # gave 1,429 / 1,433. One definition is used across the whole sweep so the intervals are
    # comparable to each other; BTC has no empty bars, so its 5s row reproduces the fork exactly.
    p2_coarse = (p2_forked
                 .filter(func.col("close").isNotNull())
                 .withColumn("p2_slot", func.col("bar_us") - func.expr(f"pmod(bar_us, {p2_us})"))
                 .groupBy("symbol", "p2_slot")
                 .agg(func.expr("max_by(close, bar_us)").alias("p2_close")))

    p2_w = Window.partitionBy("symbol").orderBy("p2_slot")
    p2_coarse = (p2_coarse
                 .withColumn("p2_d", func.lead("p2_close").over(p2_w) - func.col("p2_close"))
                 .filter(func.col("p2_d").isNotNull()))

    for row in (p2_coarse.groupBy("symbol")
                .agg(func.count("*").alias("n"),
                     func.avg((func.col("p2_d") == 0).cast("double")).alias("flat"))
                .collect()):
        p2_sweep[(p2_label, row["symbol"])] = (row["n"], row["flat"])

print(f"{'interval':<10}" + "".join(f"{s:>20}" for s in ["BTCUSDT", "ETHUSDT", "SOLUSDT"]))
for p2_label, _ in P2_INTERVALS:
    cells = ""
    for sym in ["BTCUSDT", "ETHUSDT", "SOLUSDT"]:
        n, flat = p2_sweep[(p2_label, sym)]
        cells += f"{100 * flat:>12.2f}%  n={n:<5}"
    print(f"{p2_label:<10}{cells}")

verdict(True, "Step 2.5 interval: the flat class is material at 5s and collapses by 5m -- "
              "a 3-class target exists only at the fine end of this grid")

interval               BTCUSDT             ETHUSDT             SOLUSDT
5s               27.59%  n=1439        19.18%  n=1434        18.87%  n=1436 
15s              13.57%  n=479          6.47%  n=479          8.56%  n=479  
1m                2.52%  n=119          3.36%  n=119          2.52%  n=119  
5m                0.00%  n=23           0.00%  n=23           4.35%  n=23   
APPLIES  -- Step 2.5 interval: the flat class is material at 5s and collapses by 5m -- a 3-class target exists only at the fine end of this grid


True

The two-hour sample is a quiet window (00:00–02:00 UTC on 1 January) and the flat shares above
are correspondingly generous. Over the full month — 340,971,834 ticks, measured by a separate
full-scale run and **not** by this notebook — the 5s flat class is 20.04% on BTC, 13.25% on ETH
and 14.37% on SOL, and at 1m it collapses to 1.67 / 0.64 / 2.31%. The shape of the collapse is
the same; the sample simply sits at the generous end of it. Nothing below should be read as a
measurement of January.

## 4. Imputation — Path 2

### a. Find what is missing

Missingness on bars is **row-shaped**: a bar either had ticks in it or it did not, so every
nullable column is null on exactly the same rows. Step 3 already flagged them and deliberately
did not fill them, because the fill decision is path-isolated and belongs here.

In [33]:
P2_MONEY = ["open", "high", "low", "close", "volume", "quote_volume",
            "taker_buy_qty", "taker_buy_quote_qty"]
P2_CAT_SOURCE = "all_best_match"

p2_nulls = p2_forked.select([func.count(func.when(func.col(c).isNull(), c)).alias(c)
                             for c in P2_MONEY + [P2_CAT_SOURCE]]).first().asDict()
p2_rows = p2_forked.count()
for c, n in p2_nulls.items():
    print(f"   {c:<22} {n:>4} null  ({n / p2_rows:.3%})")

# Path 1 would evaluate a 5% firewall here and ban medians above it. Path 2 has no such rule --
# the framework runs the Path 2 branch on device_os at 3.1% missing without any threshold test at
# all -- so the rate below is reported, not acted on. Quoting Path 1's threshold on this lane
# would be borrowing an authority the framework did not grant.
p2_rate = p2_forked.select(func.avg(func.col("is_missing_bar").cast("double"))).first()[0]
verdict(True, f"Step 4 missingness: {p2_rate:.3%} of bars empty -- Path 2 imputes regardless of "
              f"rate; the >5% median ban is a Path 1 rule and is not evaluated here")

   open                      8 null  (0.185%)
   high                      8 null  (0.185%)
   low                       8 null  (0.185%)
   close                     8 null  (0.185%)
   volume                    8 null  (0.185%)
   quote_volume              8 null  (0.185%)
   taker_buy_qty             8 null  (0.185%)
   taker_buy_quote_qty       8 null  (0.185%)
   all_best_match            8 null  (0.185%)
APPLIES  -- Step 4 missingness: 0.185% of bars empty -- Path 2 imputes regardless of rate; the >5% median ban is a Path 1 rule and is not evaluated here


True

### b. Handle — i. median impute the numerics

The trap is the grouping. A single median over the frame blends three price scales (BTC ≈ $94k,
ETH ≈ $3.3k, SOL ≈ $190) and would hand an empty ETH bar a five-figure close. The framework's
NexusMart matrix has one cohort per run and never has to state this; bar data does.

In [34]:
# Cast to double HERE and not earlier. Decimal is what made the entryway's sums reproducible
# across shuffle partitionings, and that work is finished: from this line on the values feed
# percentile_approx, log(), and pyspark.ml, none of which accept decimal without coercing anyway.
p2_imputed = p2_forked
for c in P2_MONEY:
    p2_imputed = p2_imputed.withColumn(c, func.col(c).cast("double"))

# percentile_approx through expr() for the same 3.3.0-wrapper reason as max_by above.
p2_medians = (p2_imputed.groupBy("symbol")
              .agg(*[func.expr(f"percentile_approx({c}, 0.5)").alias(f"p2_med_{c}")
                     for c in P2_MONEY]))
p2_imputed = p2_imputed.join(func.broadcast(p2_medians), on="symbol", how="left")
for c in P2_MONEY:
    p2_imputed = p2_imputed.withColumn(c, func.coalesce(func.col(c), func.col(f"p2_med_{c}")))

print("per-symbol medians used as the fill value:")
for row in p2_medians.orderBy("symbol").collect():
    print(f"   {row['symbol']:<9} close {row['p2_med_close']:>12,.2f}   "
          f"volume {row['p2_med_volume']:>12,.4f}")

p2_imputed = p2_imputed.drop(*[f"p2_med_{c}" for c in P2_MONEY]).cache()
p2_left = p2_imputed.select([func.count(func.when(func.col(c).isNull(), c)).alias(c)
                             for c in P2_MONEY]).first().asDict()
# The verdict is gated on whether there was anything to impute, not on whether anything is left:
# "0 nulls remain" is the success condition, and hanging an N/A on it would report a completed
# step as a skipped one.
verdict(sum(p2_nulls[c] for c in P2_MONEY) > 0,
        f"Step 4 numerics: {sum(p2_nulls[c] for c in P2_MONEY)} nulls across {len(P2_MONEY)} "
        f"columns median-imputed per symbol, {sum(p2_left.values())} remain")

per-symbol medians used as the fill value:


   BTCUSDT   close    93,898.13   volume       0.1821
   ETHUSDT   close     3,354.62   volume       3.6017
   SOLUSDT   close       191.19   volume      44.7970


APPLIES  -- Step 4 numerics: 64 nulls across 8 columns median-imputed per symbol, 0 remain


True

### b. Handle — ii. name the missing category

`all_best_match` is the column the entryway refused to prune. Step 2's BANNED verdict there
said the constancy question is Step 8's, and Step 8 is path-isolated. On Path 2 the column
arrives as the categorical block's only member besides `symbol`, and it is null on exactly the
empty bars — so it is the framework's `device_os` case, verbatim: a missing category gets a
named level rather than a dropped row.

In [35]:
# na.fill on a string column rather than StringIndexer(handleInvalid="keep"). Both route the
# nulls somewhere; only na.fill makes the destination a NAMED, inspectable level, which is what
# the framework asks for -- "assign missing categories to SYSTEM_STATE_UNKNOWN", not "route them
# to whatever index the encoder assigns last".
P2_UNKNOWN = "SYSTEM_STATE_UNKNOWN"
p2_imputed = (p2_imputed
              .withColumn("p2_best_match_state", func.col(P2_CAT_SOURCE).cast("string"))
              .na.fill(P2_UNKNOWN, subset=["p2_best_match_state"]))

p2_imputed.groupBy("p2_best_match_state").count().orderBy("p2_best_match_state").show(truncate=False)

# The finding the framework's e-commerce example cannot produce, and it matters at Step 9: on bar
# data the unknown categorical state and the Step 3 missingness flag are the SAME rows, so the
# named level and is_missing_bar are perfectly collinear by construction.
p2_alias = p2_imputed.filter(
    (func.col("p2_best_match_state") == P2_UNKNOWN) != (func.col("is_missing_bar") == 1)).count()
verdict(True, f"Step 4 categoricals: {P2_UNKNOWN} assigned to "
              f"{p2_nulls[P2_CAT_SOURCE]} rows, 0 dropped; disagreements with the Step 3 flag: "
              f"{p2_alias} -- the level and is_missing_bar are perfectly collinear here")

+--------------------+-----+
|p2_best_match_state |count|
+--------------------+-----+
|SYSTEM_STATE_UNKNOWN|8    |
|true                |4312 |
+--------------------+-----+

APPLIES  -- Step 4 categoricals: SYSTEM_STATE_UNKNOWN assigned to 8 rows, 0 dropped; disagreements with the Step 3 flag: 0 -- the level and is_missing_bar are perfectly collinear here


True

## 5. Diagnostics — Path 2

Read-only. Nothing in the matrix changes; the step emits a log. Path 2's mandate is
class-balance diagnostics plus interaction correlation matrices — the class balance is here,
and the correlation matrix is taken up under Step 6, where the grid also names it.

### a. Class balance

In [36]:
# Captured before the first diagnostic runs, so section 5c can compare against a number rather
# than against the same expression evaluated twice.
P2_SHAPE_IN = (p2_imputed.count(), len(p2_imputed.columns))

P2_CLASSES = ["up", "flat", "down"]
p2_balance = {(r["symbol"], r["p2_class"]): r["n"] for r in
              (p2_forked.filter(func.col("p2_class").isNotNull())
               .groupBy("symbol", "p2_class").agg(func.count("*").alias("n")).collect())}

print(f"{'symbol':<10}{'n':>7}" + "".join(f"{c:>18}" for c in P2_CLASSES))
for sym in ["BTCUSDT", "ETHUSDT", "SOLUSDT"]:
    n = sum(p2_balance[(sym, c)] for c in P2_CLASSES)
    cells = "".join(f"{p2_balance[(sym, c)]:>8,} {100 * p2_balance[(sym, c)] / n:>7.2f}%"
                    for c in P2_CLASSES)
    print(f"{sym:<10}{n:>7,}{cells}")

# The denominator is not 1,440. It is the count of transitions with an OBSERVED close at both
# ends: 1,439 on BTC (one window edge) and 10 / 6 fewer on ETH / SOL, because each empty bar
# kills both the transition into it and the transition out of it. Reporting these shares against
# 1,440 would quietly credit the empty bars to whichever class the arithmetic favoured.
p2_expected = {"BTCUSDT": (36.34, 27.59, 36.07), "ETHUSDT": (40.24, 19.03, 40.73),
               "SOLUSDT": (41.10, 18.84, 40.06)}
for sym, expect in p2_expected.items():
    n = sum(p2_balance[(sym, c)] for c in P2_CLASSES)
    got = tuple(round(100 * p2_balance[(sym, c)] / n, 2) for c in P2_CLASSES)
    assert got == expect, f"{sym}: measured {got}, committed {expect}"
print("\nmeasured shares match the committed sample figures to 2dp on all three symbols")

p2_pooled_flat = sum(p2_balance[(s, "flat")] for s in ["BTCUSDT", "ETHUSDT", "SOLUSDT"])
p2_min_class = min(p2_balance.values())
verdict(True, f"Step 5 class balance: k=3, pooled flat share "
              f"{100 * p2_pooled_flat / p2_labelled:.2f}% of {p2_labelled:,} labelled rows, "
              f"smallest per-symbol cell n={p2_min_class:,} -- every class clears any np>=10 "
              f"style adequacy floor by three orders of magnitude")

symbol          n                up              flat              down
BTCUSDT     1,439     523   36.34%     397   27.59%     519   36.07%
ETHUSDT     1,429     575   40.24%     272   19.03%     582   40.73%
SOLUSDT     1,433     589   41.10%     270   18.84%     574   40.06%

measured shares match the committed sample figures to 2dp on all three symbols
APPLIES  -- Step 5 class balance: k=3, pooled flat share 21.83% of 4,301 labelled rows, smallest per-symbol cell n=270 -- every class clears any np>=10 style adequacy floor by three orders of magnitude


True

### b. What the balance is a measurement of

The `flat` share above (27.59 / 19.03 / 18.84%) is a property of **this two-hour window**, not
of January. 00:00–02:00 UTC on New Year's Day is a quiet tape, and a quiet tape prints more
repeated closes. The full-month figures quoted in section 0b are 13–20% at 5s. The sample
overstates `flat`; it does not manufacture it.

### c. Read-only check

In [37]:
# Step 5's whole contract is that it observes and does not act. The check is the boring one --
# same rows, same columns, in and out -- and it is worth printing precisely because a notebook
# naturally wants every cell to transform something.
p2_shape_out = (p2_imputed.count(), len(p2_imputed.columns))
print(f"rows   {P2_SHAPE_IN[0]:,} -> {p2_shape_out[0]:,}")
print(f"columns {P2_SHAPE_IN[1]} -> {p2_shape_out[1]}")
assert P2_SHAPE_IN == p2_shape_out, "Step 5 modified the matrix"
verdict(False, "Step 5 mutation: diagnostics are read-only on every path -- the matrix that "
               "leaves this step is the matrix that entered it")

rows   4,320 -> 4,320
columns 22 -> 22
N/A      -- Step 5 mutation: diagnostics are read-only on every path -- the matrix that leaves this step is the matrix that entered it


False

## 6. Topology — Path 2

### a. Cross-correlation matrix

The framework's reason for putting this before pruning is sequencing, not curiosity: Step 6
"exports an internal mathematical dependency schema to guide downstream feature regularizers",
and the pipeline is a single non-looping forward pass, so Step 9 can only consult a matrix that
was built while the columns still existed.

In [38]:
P2_TOPO_COLS = ["close", "volume", "quote_volume", "n_ticks", "taker_buy_qty", "is_missing_bar"]

def p2_corr(frame, cols):
    """Pearson matrix over an assembled vector -- Correlation.corr is the only native route."""
    vec = VectorAssembler(inputCols=cols, outputCol="p2_topo_vec").transform(
        frame.select([func.col(c).cast("double").alias(c) for c in cols]))
    return Correlation.corr(vec, "p2_topo_vec", "pearson").collect()[0][0].toArray()

p2_pooled = p2_corr(p2_imputed, P2_TOPO_COLS)
p2_btc = p2_corr(p2_imputed.filter(func.col("symbol") == "BTCUSDT"), P2_TOPO_COLS)

p2_fmt = lambda v: "  undefined" if v != v else f"{v:>11.3f}"
print(f"{'pair':<34}{'pooled':>11}{'BTC only':>11}")
p2_flips = 0
for i in range(len(P2_TOPO_COLS)):
    for j in range(i + 1, len(P2_TOPO_COLS)):
        a, b = p2_pooled[i][j], p2_btc[i][j]
        p2_flips += (b == b) and (a * b < 0)
        print(f"{P2_TOPO_COLS[i] + ' x ' + P2_TOPO_COLS[j]:<34}{p2_fmt(a)}{p2_fmt(b)}")

# Two traps in one table.
# 1. A "global" matrix over a frame holding three symbols is largely a matrix of BETWEEN-SYMBOL
#    scale differences: close and volume separate BTC from SOL far more strongly than either
#    moves within a symbol, so the pooled figure and the per-symbol figure disagree -- on some
#    pairs in sign. Step 9's regulariser inherits whichever one it is handed.
# 2. "undefined" is not a rendering accident. BTC has zero empty bars, so is_missing_bar is
#    CONSTANT within that group and its correlation has a zero denominator. The pooled matrix
#    hides that by borrowing ETH's and SOL's variance to fill BTC's column.
verdict(True, f"Step 6 topology: cross-correlation computed pooled and per symbol -- "
              f"{p2_flips} pairs flip sign between the two, so 'global across all X features' "
              f"needs a stated grouping on bar data")

pair                                   pooled   BTC only
close x volume                         -0.170      0.104
close x quote_volume                    0.115      0.104
close x n_ticks                         0.204      0.151
close x taker_buy_qty                  -0.164      0.140
close x is_missing_bar                 -0.030  undefined
volume x quote_volume                   0.150      1.000
volume x n_ticks                        0.173      0.599
volume x taker_buy_qty                  0.672      0.928
volume x is_missing_bar                -0.006  undefined
quote_volume x n_ticks                  0.584      0.599
quote_volume x taker_buy_qty            0.107      0.928
quote_volume x is_missing_bar          -0.007  undefined
n_ticks x taker_buy_qty                 0.128      0.562
n_ticks x is_missing_bar               -0.027  undefined
taker_buy_qty x is_missing_bar         -0.007  undefined
APPLIES  -- Step 6 topology: cross-correlation computed pooled and per symbol -- 2 pairs

True

### b. Cyclical coordinate spaces

The framework names the technique — "cyclical coordinate spaces (sine/cosine encoding)" — and
never names a period. **Judgement call:** crypto trades 24/7, so there is no session open to
anchor on, and the two-hour sample admits only the fast periods. Minute-of-hour (period 60) is
encoded because it varies fully inside the sample; hour-of-day (period 24) is encoded as well,
and is deliberately left near-degenerate here — it takes exactly two values over two hours, and
watching Step 8 decide what to do with a near-constant column is more useful than hiding it.

The session timezone is pinned to UTC in the setup cell. Without that pin this box defaults to
Africa/Johannesburg and every `hour()` below is shifted by two, silently.

In [39]:
def p2_cyclical(frame, col, period, name):
    """sin/cos pair for a bounded integer clock component. No native transformer exists."""
    angle = 2 * math.pi * func.col(col) / func.lit(period)
    return (frame.withColumn(f"p2_{name}_sin", func.sin(angle))
                 .withColumn(f"p2_{name}_cos", func.cos(angle)))

p2_topo = (p2_imputed
           .withColumn("p2_minute", func.minute("bar_open_time_utc"))
           .withColumn("p2_hour", func.hour("bar_open_time_utc")))
p2_topo = p2_cyclical(p2_topo, "p2_minute", 60, "minute")
p2_topo = p2_cyclical(p2_topo, "p2_hour", 24, "hour")

# The point of the encoding, in one number: minute 59 and minute 0 are adjacent on a clock and 59
# apart as integers. Euclidean distance in the (sin, cos) plane restores the adjacency.
p2_pair = {r["p2_minute"]: (r["s"], r["c"]) for r in
           (p2_topo.filter(func.col("p2_minute").isin(0, 1, 30, 59))
            .groupBy("p2_minute").agg(func.first("p2_minute_sin").alias("s"),
                                      func.first("p2_minute_cos").alias("c")).collect())}
dist = lambda a, b: math.dist(p2_pair[a], p2_pair[b])
print(f"raw integer distance  |59 - 0| = 59      |1 - 0| = 1       |30 - 0| = 30")
print(f"cyclical distance     59 -> 0  = {dist(59, 0):.4f}   1 -> 0  = {dist(1, 0):.4f}  "
      f"30 -> 0  = {dist(30, 0):.4f}")

p2_hour_var = p2_topo.select(func.var_samp("p2_hour_sin")).first()[0]
verdict(True, f"Step 6 cyclical: minute-of-hour and hour-of-day encoded as sin/cos; hour_sin "
              f"variance is {p2_hour_var:.4f} on a 2-hour sample -- near-degenerate by "
              f"construction, and left in for Step 8 to rule on")

raw integer distance  |59 - 0| = 59      |1 - 0| = 1       |30 - 0| = 30
cyclical distance     59 -> 0  = 0.1047   1 -> 0  = 0.1047  30 -> 0  = 2.0000


APPLIES  -- Step 6 cyclical: minute-of-hour and hour-of-day encoded as sin/cos; hour_sin variance is 0.0168 on a 2-hour sample -- near-degenerate by construction, and left in for Step 8 to rule on


True

## 7. Feature Engineering — Path 2

"Cyclical coordinate engineering; interaction cross-products." Deterministic products only —
nothing learned, nothing stochastic. The framework is explicit that this step's ceiling is
shallow hand-built interactions, and that anything deeper is a modelling escalation to the ANN
tier rather than more Step 7.

Every feature below is computed from the **current** bar. No lag, no rolling window: those
carry structural nulls at the series head, which Step 4 has already closed and which the 5%
firewall on the sibling path would misread as a data defect.

In [40]:
p2_eng = (p2_topo
          # Signed taker imbalance in [-1, +1]. The sign convention is the entryway's, and it is
          # the one that was measured rather than assumed: is_buyer_maker=True means the BUYER
          # was the maker, so the trade is an aggressive SELL, and taker_buy is the complement.
          # Inverted, corr(imbalance, return) flips from +0.51 to -0.51 on the full month.
          .withColumn("p2_imbalance",
                      (2 * func.col("taker_buy_qty") - func.col("volume")) / func.col("volume"))
          # log(high/low) rather than (high-low)/low: scale-free across three price levels, and
          # exactly 0 on a zero-range bar instead of a denominator choice.
          .withColumn("p2_log_range", func.log(func.col("high") / func.col("low")))
          .withColumn("p2_body", (func.col("close") - func.col("open")) / func.col("open"))
          .withColumn("p2_ticks", func.col("n_ticks").cast("double"))
          # Deterministic cross-products: the framework's depth x freq, in bar terms. Each is a
          # product of two columns that already exist, so it is reproducible from the frame and
          # auditable -- which is the property "deterministic" is protecting.
          .withColumn("p2_x_imb_body", func.col("p2_imbalance") * func.col("p2_body"))
          .withColumn("p2_x_ticks_range", func.col("p2_ticks") * func.col("p2_log_range"))
          .withColumn("p2_x_imb_minute", func.col("p2_imbalance") * func.col("p2_minute_sin"))
          # The categorical cross-product. The framework's headline Path 2 finding was a
          # three-way device_os x referral_channel x cohort interaction, so a categorical x
          # categorical product is in scope for this step -- and it is what gives Step 8 a
          # multi-level column with genuinely rare levels to rule on.
          .withColumn("p2_symbol_state",
                      func.concat_ws("__", func.col("symbol"), func.col("p2_best_match_state")))
          .cache())

P2_NUMERIC = ["p2_imbalance", "p2_log_range", "p2_body", "p2_ticks",
              "p2_minute_sin", "p2_minute_cos", "p2_hour_sin", "p2_hour_cos",
              "p2_x_imb_body", "p2_x_ticks_range", "p2_x_imb_minute",
              "is_missing_bar", "is_first_bar", "is_last_bar"]
P2_CATEGORICAL = ["symbol", "p2_best_match_state", "p2_symbol_state"]

p2_eng.groupBy("p2_symbol_state").count().orderBy("p2_symbol_state").show(truncate=False)
p2_eng_nulls = p2_eng.select([func.count(func.when(func.col(c).isNull(), c)).alias(c)
                              for c in P2_NUMERIC]).first().asDict()
verdict(True, f"Step 7 features: {len(P2_NUMERIC)} numeric + {len(P2_CATEGORICAL)} categorical "
              f"columns, {sum(p2_eng_nulls.values())} nulls -- no lag or rolling term, so no "
              f"structural null at the series edges")

+-----------------------------+-----+
|p2_symbol_state              |count|
+-----------------------------+-----+
|BTCUSDT__true                |1440 |
|ETHUSDT__SYSTEM_STATE_UNKNOWN|5    |
|ETHUSDT__true                |1435 |
|SOLUSDT__SYSTEM_STATE_UNKNOWN|3    |
|SOLUSDT__true                |1437 |
+-----------------------------+-----+



APPLIES  -- Step 7 features: 14 numeric + 3 categorical columns, 0 nulls -- no lag or rolling term, so no structural null at the series edges


True

## 8. Pruning — Path 2 **ENFORCE**

> "One-Hot encoding before variance pruning."

The mandate is on the **order**. Both operations happen either way. What changes is the
granularity at which the variance filter can act: encode-first prunes at **level** granularity
(`p2_symbol_state = ETHUSDT__SYSTEM_STATE_UNKNOWN` lives or dies on its own), prune-first
prunes at **column** granularity (`p2_symbol_state` survives whole or dies whole).

### a. The mandated order — index, encode, assemble, prune

In [41]:
# Spark forces half the discipline for you: OneHotEncoder takes numeric indices only, so
# StringIndexer is mandatory first and there is no way to encode a raw string column.
# dropLast=False is deliberate and not the default. The default drops a reference level, and a
# dropped level is a level the variance filter can never measure -- which is exactly the
# granularity the ENFORCE exists to protect.
p2_indexers = [StringIndexer(inputCol=c, outputCol=f"p2_idx_{c}").fit(p2_eng)
               for c in P2_CATEGORICAL]
p2_encoded = p2_eng
for idx in p2_indexers:
    p2_encoded = idx.transform(p2_encoded)

p2_ohe = OneHotEncoder(inputCols=[f"p2_idx_{c}" for c in P2_CATEGORICAL],
                       outputCols=[f"p2_ohe_{c}" for c in P2_CATEGORICAL],
                       dropLast=False).fit(p2_encoded)
p2_encoded = p2_ohe.transform(p2_encoded)

# Explode the dummy vectors back into named 0/1 columns. Not decoration: the whole ENFORCE
# argument is that a level has an independent existence, and a level buried inside a VectorUDT
# cannot be named in a coefficient table or a variance report.
P2_DUMMIES = []
for cat, idx in zip(P2_CATEGORICAL, p2_indexers):
    arr = vector_to_array(func.col(f"p2_ohe_{cat}"))
    for i, level in enumerate(idx.labels):
        name = f"oh__{cat}__{level}"
        p2_encoded = p2_encoded.withColumn(name, arr.getItem(i))
        P2_DUMMIES.append(name)

P2_ASSEMBLED = P2_NUMERIC + P2_DUMMIES
p2_encoded = VectorAssembler(inputCols=[c for c in P2_ASSEMBLED], outputCol="p2_features_raw") \
    .transform(p2_encoded.select(
        *[func.col(c).cast("double").alias(c) for c in P2_ASSEMBLED],
        "symbol", "bar_us", "p2_class", "p2_symbol_state")).cache()

print(f"{len(P2_NUMERIC)} numeric + {len(P2_DUMMIES)} one-hot levels = "
      f"{len(P2_ASSEMBLED)} columns into the variance filter")
print("one-hot levels:")
for d in P2_DUMMIES:
    print(f"   {d}")

14 numeric + 10 one-hot levels = 24 columns into the variance filter
one-hot levels:
   oh__symbol__BTCUSDT
   oh__symbol__ETHUSDT
   oh__symbol__SOLUSDT
   oh__p2_best_match_state__true
   oh__p2_best_match_state__SYSTEM_STATE_UNKNOWN
   oh__p2_symbol_state__BTCUSDT__true
   oh__p2_symbol_state__SOLUSDT__true
   oh__p2_symbol_state__ETHUSDT__true
   oh__p2_symbol_state__ETHUSDT__SYSTEM_STATE_UNKNOWN
   oh__p2_symbol_state__SOLUSDT__SYSTEM_STATE_UNKNOWN


### b. Choosing the threshold, and saying why

The framework never states one. **Judgement call:** on bar data a raw-magnitude variance
threshold is not comparable across columns. The table below spans twelve orders of magnitude —
`p2_ticks` at 1.8 × 10⁴ against `p2_x_imb_body` at 1 × 10⁻⁸ — so *any* threshold that keeps the
tick count deletes every return-shaped column, and any threshold that keeps the returns keeps
everything. The forward-pass rule forbids fixing this by scaling first, because scaling is
Step 10. The threshold is therefore set to **0.0**, which is the framework's own demonstration
(`constant_metric` is a flat 1.0 and nothing else goes): it removes genuinely constant columns
and nothing else. Section 8c prices what any larger value would cost.

In [42]:
p2_var = p2_encoded.select([func.var_samp(func.col(c)).alias(c) for c in P2_ASSEMBLED]).first().asDict()
print(f"{'column':<52}{'sample variance':>18}")
for c in sorted(P2_ASSEMBLED, key=lambda k: p2_var[k]):
    print(f"{c:<52}{p2_var[c]:>18.8f}")

p2_selector = VarianceThresholdSelector(featuresCol="p2_features_raw", outputCol="p2_features",
                                        varianceThreshold=0.0).fit(p2_encoded)
p2_kept = [P2_ASSEMBLED[i] for i in p2_selector.selectedFeatures]
p2_dropped = [c for c in P2_ASSEMBLED if c not in p2_kept]
p2_encoded = p2_selector.transform(p2_encoded)

verdict(bool(p2_dropped),
        f"Step 8 prune at threshold 0.0: {len(p2_dropped)} of {len(P2_ASSEMBLED)} columns removed "
        f"{p2_dropped or '-- nothing in this frame is constant'}")

column                                                 sample variance
p2_x_imb_body                                               0.00000001
p2_log_range                                                0.00000002
p2_body                                                     0.00000003
p2_hour_cos                                                 0.00029033
is_last_bar                                                 0.00069412
is_first_bar                                                0.00069412
oh__p2_symbol_state__SOLUSDT__SYSTEM_STATE_UNKNOWN          0.00069412
oh__p2_symbol_state__ETHUSDT__SYSTEM_STATE_UNKNOWN          0.00115634
is_missing_bar                                              0.00184885
oh__p2_best_match_state__SYSTEM_STATE_UNKNOWN               0.00184885
oh__p2_best_match_state__true                               0.00184885
p2_x_ticks_range                                            0.00651693
p2_hour_sin                                                 0.01675070
oh__p2

N/A      -- Step 8 prune at threshold 0.0: 0 of 24 columns removed -- nothing in this frame is constant


False

### c. What the other order does — measured, not asserted

Prune-first has exactly two possible behaviours and both are wrong. If the pruner skips string
columns, Path 2's pruning stage protects nothing — the categoricals pass through unscreened and
get encoded downstream anyway. If a label encoder ran first, the pruner measures the variance
of **index codes**, which is a property of how many levels there are and how they happened to
be numbered.

In [43]:
# Same column, two indexer orderings, nothing else changed.
p2_alt = StringIndexer(inputCol="p2_symbol_state", outputCol="p2_idx_alt",
                       stringOrderType="alphabetAsc").fit(p2_eng).transform(p2_eng)
p2_codes = (p2_indexers[2].transform(p2_eng)
            .select(func.var_samp("p2_idx_p2_symbol_state").alias("v")).first()["v"])
p2_codes_alt = p2_alt.select(func.var_samp("p2_idx_alt").alias("v")).first()["v"]

print("PRUNE-FIRST -- variance of the index codes for p2_symbol_state (5 levels):")
print(f"   stringOrderType='frequencyDesc' (default) : {p2_codes:.6f}")
print(f"   stringOrderType='alphabetAsc'             : {p2_codes_alt:.6f}")
print(f"   the data is identical; the difference is the numbering\n")

print("ENCODE-FIRST -- variance of each level of the same column:")
for d in [c for c in P2_DUMMIES if c.startswith("oh__p2_symbol_state__")]:
    print(f"   {d:<52}{p2_var[d]:>12.6f}")

# The threshold sweep. Column granularity vs level granularity, priced.
print(f"\n{'threshold':>11}{'encode-first kept':>20}{'UNKNOWN levels kept':>22}"
      f"{'prune-first: symbol_state':>28}")
for t in [0.0, 0.0005, 0.001, 0.01, 0.5, 1.0]:
    kept = [c for c in P2_ASSEMBLED if p2_var[c] > t]
    unknown = [c for c in kept if P2_UNKNOWN in c]
    whole = "kept (as 1 ordinal column)" if p2_codes > t else "DELETED (all 5 levels)"
    print(f"{t:>11.4f}{len(kept):>20}{len(unknown):>22}{whole:>28}")

verdict(True, "Step 8 ENFORCE: prune-first can only keep or delete p2_symbol_state whole, and "
              "what it measures to decide is a numbering artefact; encode-first measures each "
              "level on the thing that carries the signal")

PRUNE-FIRST -- variance of the index codes for p2_symbol_state (5 levels):
   stringOrderType='frequencyDesc' (default) : 0.676535
   stringOrderType='alphabetAsc'             : 2.666355
   the data is identical; the difference is the numbering

ENCODE-FIRST -- variance of each level of the same column:
   oh__p2_symbol_state__BTCUSDT__true                      0.222274
   oh__p2_symbol_state__SOLUSDT__true                      0.222042
   oh__p2_symbol_state__ETHUSDT__true                      0.221886
   oh__p2_symbol_state__ETHUSDT__SYSTEM_STATE_UNKNOWN      0.001156
   oh__p2_symbol_state__SOLUSDT__SYSTEM_STATE_UNKNOWN      0.000694

  threshold   encode-first kept   UNKNOWN levels kept   prune-first: symbol_state
     0.0000                  24                     3  kept (as 1 ordinal column)
     0.0005                  20                     3  kept (as 1 ordinal column)
     0.0010                  17                     2  kept (as 1 ordinal column)
     0.0100               

True

### d. What that costs on this data, concretely

Three things break under prune-first, and all three are visible above.

1. **Step 4's work is annulled.** `SYSTEM_STATE_UNKNOWN` is a *level*, not a column. It has no
   independent existence until one-hot creates a column for it. Prune-first decides the fate of
   `p2_symbol_state` in aggregate, so the named state the framework went to the trouble of
   creating one step earlier is not something the filter can even see.
2. **The keep/drop decision stops being a function of the data.** The two index variances above
   are computed over the same 4,320 rows. Rename a symbol, or switch the indexer's ordering,
   and the number changes — so pruning becomes non-deterministic across data refreshes with no
   code change at all.
3. **The ordinal fiction propagates.** A variance over codes `0,1,2,3,4` asserts that
   `SOLUSDT__SYSTEM_STATE_UNKNOWN` is four times `BTCUSDT__true`. Every statistic downstream of
   that number inherits the fiction, and the framework's ANN escalation — whose stated
   prerequisites are exactly Step 8's one-hot and Step 10's quantile transform — would have to
   re-run Step 8 to undo it, which the single-forward-pass rule forbids.

The honest bookend: on **this** data the preserved level cannot pay off. Section 0a showed that
the empty bars are precisely the rows whose target is unobservable, so the
`SYSTEM_STATE_UNKNOWN` dummies are all-zero on the labelled frame Step 9 trains against. The
ENFORCE is still right — it is what keeps the level addressable, and on a month of data with
feed gaps that are not also target gaps it would pay — but the notebook should not claim a
subgroup finding it did not measure.

In [44]:
p2_unknown_labelled = (p2_encoded.filter(func.col("p2_class").isNotNull())
                       .filter(func.col("p2_symbol_state").contains(P2_UNKNOWN)).count())
verdict(p2_unknown_labelled > 0,
        f"Step 8 -> Step 9 handover: {p2_unknown_labelled} labelled rows carry a "
        f"{P2_UNKNOWN} level -- the level survived pruning but is constant on the supervised "
        f"frame, so Elastic Net cannot score it")

N/A      -- Step 8 -> Step 9 handover: 0 labelled rows carry a SYSTEM_STATE_UNKNOWN level -- the level survived pruning but is constant on the supervised frame, so Elastic Net cannot score it


False

## 9. Regularisation — Path 2

"Cross-validated Elastic Net regularisation loop", with a class target instead of a continuous
one: multinomial logistic regression with `elasticNetParam`, k=3.

### a. The framework's most dangerous silence

The framework says "cross-validated" and never mentions temporal ordering, because NexusMart's
rows are exchangeable sessions. Bars are not exchangeable. Spark's `CrossValidator` builds
random folds, so on ordered data it trains on the future and validates on the past. Both
schemes are run below and the gap is measured, because the size of the optimism is the argument.

In [45]:
p2_model_df = p2_encoded.filter(func.col("p2_class").isNotNull()).withColumn(
    "p2_label", func.when(func.col("p2_class") == "down", 0.0)
                    .when(func.col("p2_class") == "flat", 1.0).otherwise(2.0)).cache()

# Chronological cuts on bar_us, which is a uniform 5s grid, so a fraction of the range is a
# fraction of the slots. Folds are EXPANDING: each validates on a block that is strictly later
# than everything it trained on.
p2_span = p2_model_df.agg(func.min("bar_us").alias("lo"), func.max("bar_us").alias("hi")).first()
p2_cut = lambda f: p2_span["lo"] + int(f * (p2_span["hi"] - p2_span["lo"]))
P2_FOLDS = [(0.50, 0.60), (0.60, 0.70), (0.70, 0.80)]
p2_holdout = p2_model_df.filter(func.col("bar_us") >= p2_cut(0.80))
p2_dev = p2_model_df.filter(func.col("bar_us") < p2_cut(0.80))

p2_eval = MulticlassClassificationEvaluator(labelCol="p2_label", predictionCol="prediction",
                                            metricName="accuracy")
P2_GRID = [(reg, en) for reg in [0.01, 0.1] for en in [0.5, 1.0]]

# The number every accuracy below has to be read against: predict the class that was most common
# on dev, on every holdout row. Without it "0.40" is unanchored.
p2_majority = (p2_dev.groupBy("p2_label").count().orderBy(func.desc("count")).first()["p2_label"])
P2_BASELINE = p2_holdout.select(func.avg((func.col("p2_label") == p2_majority)
                                         .cast("double"))).first()[0]
print(f"majority-class baseline on the holdout: {P2_BASELINE:.4f}   "
      f"(dev rows {p2_dev.count():,} | holdout rows {p2_holdout.count():,})\n")

def p2_fit(train, reg, en):
    return LogisticRegression(featuresCol="p2_features", labelCol="p2_label", maxIter=50,
                              regParam=reg, elasticNetParam=en).fit(train)

print(f"{'regParam':>10}{'elasticNet':>12}{'time-ordered CV':>18}{'random-fold CV':>17}{'gap':>9}")
p2_scores = {}
for reg, en in P2_GRID:
    ordered = []
    for lo, hi in P2_FOLDS:
        tr = p2_model_df.filter(func.col("bar_us") < p2_cut(lo))
        va = p2_model_df.filter((func.col("bar_us") >= p2_cut(lo)) &
                                (func.col("bar_us") < p2_cut(hi)))
        ordered.append(p2_eval.evaluate(p2_fit(tr, reg, en).transform(va)))
    # The leaky comparator: random folds over the same dev window.
    random_folds = []
    for seed in [11, 22, 33]:
        tr, va = p2_dev.randomSplit([0.8, 0.2], seed=seed)
        random_folds.append(p2_eval.evaluate(p2_fit(tr, reg, en).transform(va)))
    o, r = sum(ordered) / len(ordered), sum(random_folds) / len(random_folds)
    p2_scores[(reg, en)] = o
    print(f"{reg:>10}{en:>12}{o:>18.4f}{r:>17.4f}{r - o:>+9.4f}")

p2_best = max(p2_scores, key=p2_scores.get)
verdict(True, f"Step 9 CV: time-ordered expanding-window folds select regParam={p2_best[0]}, "
              f"elasticNetParam={p2_best[1]}; random folds score higher on every grid point, "
              f"which is the leak, not a better model")

majority-class baseline on the holdout: 0.3540   (dev rows 3,445 | holdout rows 856)

  regParam  elasticNet   time-ordered CV   random-fold CV      gap


      0.01         0.5            0.4102           0.4448  +0.0346


      0.01         1.0            0.4079           0.4433  +0.0355


       0.1         0.5            0.3598           0.4058  +0.0460


       0.1         1.0            0.3598           0.4058  +0.0460
APPLIES  -- Step 9 CV: time-ordered expanding-window folds select regParam=0.01, elasticNetParam=0.5; random folds score higher on every grid point, which is the leak, not a better model


True

### b. What Elastic Net discarded

The framework's own statement of the Step 8 / Step 9 division of labour: Step 8 is unsupervised
and target-blind (variance only), Step 9 is supervised and target-aware. A column whose numbers
vary — so it passed Step 8 — but which has no relationship to the class label has its
coefficient shrunk to exactly zero here.

In [46]:
p2_final = p2_fit(p2_dev, *p2_best)
p2_coef = p2_final.coefficientMatrix.toArray()          # k x p, one row per class
p2_weight = {name: max(abs(p2_coef[k][i]) for k in range(p2_coef.shape[0]))
             for i, name in enumerate(p2_kept)}

print(f"{'feature':<52}{'max |coef| across the 3 classes':>34}")
for name in sorted(p2_weight, key=lambda k: -p2_weight[k]):
    print(f"{name:<52}{p2_weight[name]:>34.6f}")

p2_zeroed = [n for n, w in p2_weight.items() if w == 0.0]
verdict(bool(p2_zeroed),
        f"Step 9 Elastic Net: {len(p2_zeroed)} of {len(p2_kept)} surviving columns shrunk to "
        f"exactly zero -- {p2_zeroed}")

feature                                                max |coef| across the 3 classes
p2_log_range                                                               1020.169276
p2_body                                                                     329.603709
p2_x_imb_body                                                               181.200841
p2_hour_cos                                                                   3.164202
is_first_bar                                                                  1.814542
p2_hour_sin                                                                   0.416575
oh__symbol__BTCUSDT                                                           0.147360
oh__p2_symbol_state__BTCUSDT__true                                            0.147360
p2_imbalance                                                                  0.122090
p2_minute_sin                                                                 0.093634
p2_minute_cos                              

True

### c. Two things to read out of that table

**The identical pair.** `oh__symbol__BTCUSDT` and `oh__p2_symbol_state__BTCUSDT__true` carry
exactly the same coefficient. They differ only on the empty bars, and the empty bars are not in
the supervised frame, so on these 4,301 rows Step 7's categorical cross-product is a perfect
duplicate of the column it was built from. Elastic Net at `elasticNetParam=0.5` split the
weight evenly between them rather than dropping one — that is the L2 half of the penalty doing
what L2 does with collinear columns. At `elasticNetParam=1.0` (pure L1) one of the pair would
have been zeroed arbitrarily, which is worse for reading the model and no better for fit.

**The magnitudes.** `p2_log_range` carries a coefficient near 10³ and `p2_ticks` one near
10⁻³, because the L1 penalty `λ|w|` is charged in **coefficient** units and the coefficient
absorbs the column's scale. A feature whose natural units are large buys its contribution to
the fit cheaply; one whose units are small — every return-shaped column on a bar frame — is
charged heavily for exactly the same contribution and is zeroed first.

This is Step 8's variance-threshold problem wearing different clothes, and it has the same
cause: the framework's Step 10 scaling runs **after** Step 9, and the single-forward-pass rule
forbids reordering them. The framework never hits it because every NexusMart column is O(1) to
O(400). It is stated here rather than fixed, because fixing it means leaving the framework.

Note also what the regulariser did with the Step 3 flag: `is_missing_bar` is one of the ten
columns shrunk to exactly zero, and so are both `SYSTEM_STATE_UNKNOWN` levels. All three are
constant on the supervised frame for the reason section 8d gave, so this is Elastic Net
correctly finding nothing rather than Elastic Net destroying something.

## 10. Scaling — Path 2

> "Quantile search across StandardScaler, PowerTransformer, QuantileTransformer."

Path 2 has no interpretability constraint — a classifier emits a probability, not a dollar
figure — so unlike Path 1's LIMIT it is free to take a non-linear transform if the measurement
says so. The measurement is a *search*, and the criterion has to be stated: **held-out accuracy
on the last 20% of the window**, which is the only block no candidate was fitted on.

**Substitution, labelled as one.** Spark ships neither `PowerTransformer` (no Yeo-Johnson, no
Box-Cox) nor `QuantileTransformer` (no rank-to-uniform map). `QuantileDiscretizer` is the
nearest shipped thing and is a different transform — it bins to ordinal buckets. Two of the
three legs below are therefore hand-rolled stand-ins, and the framework's search cannot be run
natively in Spark as written.

In [47]:
# percent_rank over an unpartitioned window is the rank-to-uniform map. It forces every row onto
# one executor -- acceptable at 4,320 rows and nowhere near it at 1.6M, which is the honest
# ceiling on this substitution.
def p2_quantile(frame, cols):
    for c in cols:
        frame = frame.withColumn(c, func.percent_rank().over(Window.orderBy(func.col(c))))
    return frame

def p2_power(frame, cols):
    # Signed log1p: the monotone, sign-preserving stand-in for Yeo-Johnson at lambda=0. It is not
    # Yeo-Johnson -- there is no lambda search here -- and calling it one would be a lie.
    for c in cols:
        frame = frame.withColumn(c, func.signum(c) * func.log1p(func.abs(func.col(c))))
    return frame

def p2_score(train, test, features):
    tr = VectorAssembler(inputCols=features, outputCol="p2_scaled_vec").transform(train)
    te = VectorAssembler(inputCols=features, outputCol="p2_scaled_vec").transform(test)
    model = LogisticRegression(featuresCol="p2_scaled_vec", labelCol="p2_label", maxIter=50,
                               regParam=p2_best[0], elasticNetParam=p2_best[1]).fit(tr)
    return p2_eval.evaluate(model.transform(te))

p2_results = {}

# Leg 1 -- StandardScaler, the only one of the three that Spark actually ships. withMean=True is
# not the default and it densifies the vector; at this width that is free, on a wide one-hot
# matrix it is the memory blow-up to watch for. Fitted on dev only: fitting on the full frame
# would leak the holdout's mean and standard deviation into training.
p2_asm = VectorAssembler(inputCols=p2_kept, outputCol="p2_v")
p2_dev_v, p2_hold_v = p2_asm.transform(p2_dev), p2_asm.transform(p2_holdout)
p2_std = StandardScaler(inputCol="p2_v", outputCol="p2_scaled_vec",
                        withMean=True, withStd=True).fit(p2_dev_v)
p2_results["StandardScaler"] = p2_eval.evaluate(
    LogisticRegression(featuresCol="p2_scaled_vec", labelCol="p2_label", maxIter=50,
                       regParam=p2_best[0], elasticNetParam=p2_best[1])
    .fit(p2_std.transform(p2_dev_v)).transform(p2_std.transform(p2_hold_v)))

# Legs 2 and 3. Both are fitted on dev and applied to the holdout by re-deriving the transform on
# each frame -- a real QuantileTransformer would carry the train quantiles across, and this
# stand-in cannot, which is a second thing the substitution loses.
p2_results["PowerTransformer (signed log1p)"] = p2_score(
    p2_power(p2_dev, p2_kept), p2_power(p2_holdout, p2_kept), p2_kept)
p2_results["QuantileTransformer (percent_rank)"] = p2_score(
    p2_quantile(p2_dev, p2_kept), p2_quantile(p2_holdout, p2_kept), p2_kept)

print(f"{'candidate':<40}{'holdout accuracy':>18}{'vs baseline':>14}")
for name, score in sorted(p2_results.items(), key=lambda kv: -kv[1]):
    print(f"{name:<40}{score:>18.4f}{score - P2_BASELINE:>+14.4f}")
print(f"{'(majority-class baseline)':<40}{P2_BASELINE:>18.4f}")

p2_winner = max(p2_results, key=p2_results.get)
verdict(True, f"Step 10 quantile search: {p2_winner} wins on held-out accuracy "
              f"({p2_results[p2_winner]:.4f}) over {len(p2_results)} candidates, chosen on a "
              f"measured criterion rather than on the framework's NexusMart precedent")

candidate                                 holdout accuracy   vs baseline
PowerTransformer (signed log1p)                     0.4042       +0.0502
StandardScaler                                      0.4007       +0.0467
QuantileTransformer (percent_rank)                  0.3972       +0.0432
(majority-class baseline)                           0.3540
APPLIES  -- Step 10 quantile search: PowerTransformer (signed log1p) wins on held-out accuracy (0.4042) over 3 candidates, chosen on a measured criterion rather than on the framework's NexusMart precedent


True

## Path 2 — what left the refinery, and what is owed

The Path 2 lane ends here and hands its matrix to Part II's clustering block, then to the
Proportion Engine and the four gates. Nothing below Step 10 is this notebook's business.

**The judgement calls this section made, none of them the framework's:**

| Decision | Why the framework could not settle it |
| --- | --- |
| y = next-bar close-to-close direction, k=3 | bar data has no experiment and no assigned target |
| 5s bar interval | the flat class, and therefore k=3, only exists at the fine end of the grid |
| unlabelled rows leave at Step 9, not Step 4 | Step 4's rules are about X; a missing y has no stated handling |
| variance threshold = 0.0 | raw magnitudes are not comparable across bar features, and the forward-pass rule forbids scaling first |
| expanding-window CV folds | "cross-validated" is stated; temporal ordering never is, and random folds leak |
| minute-of-hour and hour-of-day periods | the technique is named, the period never is; crypto has no session boundary to anchor on |
| signed log1p and percent_rank | Spark ships neither PowerTransformer nor QuantileTransformer |
| held-out accuracy as the search criterion | "quantile search" is mandated, the criterion never is |

**Two ordering problems the forward-pass rule makes unfixable from inside.** Step 8 prunes on
raw variance and Step 10 scales, in that order, so the pruner compares columns whose variances
span twelve orders of magnitude. Step 9 penalises coefficients and Step 10 scales, so the
regulariser charges scale-dependent prices for the same predictive contribution. Both are
stated above and neither is repaired here: "no step revisits earlier output" is the framework's
rule, and quietly reordering the steps would make this a different pipeline wearing its name.

**What the ENFORCE actually bought on this data:** the `SYSTEM_STATE_UNKNOWN` level stayed
addressable through pruning, which is the framework's whole claim for the order. It could not
be scored at Step 9 because on bar data the unobserved feature state and the unobservable
target are the same rows — a coincidence of this dataset, not a flaw in the mandate, and one a
month of data with genuine feed outages would break.

## Part 4 — Path 3: Bandit (Ads)

The fork routed here on the geometry of `y`. Paths 1 and 2 read a column; Path 3 reads the
absence of one. There is no per-row target to regress or classify — there is a *decision*
taken at each instant, a binary reward observed only for the option that was taken, and no
fixed sample size. Steps 4–10 below are the Path 3 sub-refinery. Three of the seven are
forbidden, and the bans are the substance of this part, not its footnotes.

**The mapping onto this dataset, stated before anything is computed.** The framework's
Path 3 example is a live ad campaign; nothing here is an ad. The correspondence is the
notebook's invention and the framework offers no guidance on manufacturing it
(judgement call: *"Crypto bars are a batch file. There are no arms and no click."*):

| Framework | Here |
|---|---|
| Arm (ad creative) | Symbol — `BTCUSDT`, `ETHUSDT`, `SOLUSDT` |
| Impression | One 5-second bar slot |
| Reward `x ∈ {0,1}` | The bar closed up (`close > previous close`) |
| Engine | Beta-Bernoulli Thompson Sampling, `α ← γα + x`, `β ← γβ + (1−x)`, γ = 0.97 |
| Tokens for association mining | symbol, hour bucket, maker/taker regime — kept as raw strings |

**This is a replay of a monthly archive in timestamp order. It is not a live stream.**
Every bar in `bars` was written to disk before this notebook opened. The engine is driven by
reading rows in ascending `bar_us`, which imitates arrival order and imitates nothing else:
there is no latency, no partial day, no arm added mid-flight, and — the part that matters —
the file holds the reward of *every* arm at *every* slot, including the arms the bandit did
not pull. A live bandit never sees a counterfactual. The replay below deliberately hides the
unpulled arms' rewards from the update, but the notebook could cheat and a live system could
not, so no result here is evidence that the engine would behave this way in production.
The framework never sanctions this substitution: Gate 3's Peeking Breaker exists precisely to
stop a stream being treated as a batch, and this is that trade run backwards.

**Four framework badges, three verdict states.** The grid uses `APPLIES`, `OVERRIDE`,
`ENFORCE` and `BANNED`; `verdict()` from `glue-ingest-bars.py` prints `APPLIES` / `N/A` /
`BANNED`. Step 4's `OVERRIDE` is reported below as two calls — an `APPLIES` for the work the
override *does* (carry the gaps forward as latent states) and a `BANNED` for the work it
*forbids* (any fill). Nothing is relabelled to make it fit; the split is stated where it
happens.

In [48]:
# Path 3 constants. Every one of them is a decision the framework did not make for us, so each
# is named, hard-coded once, and commented with the reasoning rather than left inline.
import math

import numpy as np
from pyspark.sql import Window
from pyspark.ml.feature import VarianceThresholdSelector, VectorAssembler
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.regression import LinearRegression

# gamma is the framework's, but its UNIT is not. "data from 50 impressions ago retains only
# 0.97^50 = 21%" counts IMPRESSIONS, and one impression here is one 5s slot, so the ~33-pull
# effective memory (1/(1-gamma)) is about 2m45s of wall clock. Copy 0.97 onto 1h bars and the
# same constant means a day and a half. It stays a named constant for that reason.
P3_GAMMA = 0.97
P3_ARMS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

# Beta(1,1) -- "Every ad boots up with zero tracking history, initialized to a flat uniform
# prior state of Beta(1,1)". Uniform on [0,1]: no arm is favoured before the first pull.
P3_PRIOR_A, P3_PRIOR_B = 1.0, 1.0

# The framework's own Apriori floor, kept verbatim (min_support=0.003) so the support cut is
# the document's and not ours. It bites on this extract and that is reported, not hidden.
P3_MIN_SUPPORT = 0.003
P3_MIN_CONFIDENCE = 0.50

# Maker/taker regime cut points on taker_buy_qty / volume. Fixed and stated rather than fitted
# to quantiles of this extract: a quantile-derived band is a different token on every month,
# which makes support figures incomparable across runs.
P3_TAKER_EXTREME, P3_TAKER_HEAVY = 0.90, 0.60
P3_MAKER_HEAVY, P3_MAKER_EXTREME = 0.40, 0.10

# Thompson sampling is stochastic: the answer moves with the seed. One seeded run is shown in
# full and then the whole replay is repeated over many seeds, because a single trajectory of a
# discounted bandit is a sample, not a result.
P3_SEED = 20250101
P3_SEED_REPLICATES = 200

p3_slots = bars.select("bar_us").distinct().count()
print(f"Path 3 input: {bars.count():,} bars | {p3_slots:,} slots x {len(P3_ARMS)} arms")
print(f"gamma={P3_GAMMA} (per pull, ~{1 / (1 - P3_GAMMA):.0f} pulls of memory "
      f"= ~{5 / (1 - P3_GAMMA) / 60:.1f} minutes at 5s bars)")

Path 3 input: 4,320 bars | 1,440 slots x 3 arms
gamma=0.97 (per pull, ~33 pulls of memory = ~2.8 minutes at 5s bars)


### 4. Imputation — OVERRIDE

> "Maintain native missingness — treat gaps as unobserved latent states."

Step 3 flagged the empty bars and deliberately did not fill them, because filling is a Step 4
decision and Step 4 is post-fork. Here the fork has fired, and Path 3's answer is that there
is nothing to decide: no median, no forward fill, no `coalesce(volume, 0)`, no row drop. The
nulls travel to the engines intact.

The reason this is safe rather than lazy is structural. A transaction on Path 3 is a
*variable-length set of tokens that were observed*, not a row with slots. An empty bar is not
a defective row with holes in it — it is a shorter basket. There is no cell to fill because
the representation has no cells. The same bar still contributes the tokens that *were*
observed (its symbol, its hour), so it participates in the frame as itself rather than as an
imputed impostor.

One distinction the framework leaves open and this data forces (judgement call: *"a bar with
zero trades and a bar whose feed dropped are not the same thing"*). Step 3 measured
`n_ticks = 0` against a calendar declared independently of the data, so an empty bar here is
an **observed absence of trading**, not a tracking failure. It therefore earns a token of its
own, `<SYM>_NO_TRADES`, while the quantities that cannot exist without a trade — price,
volume, maker/taker ratio — contribute no token at all.

In [49]:
# The rate test is a mean over the flag Step 3 already built, not a fresh null scan. Path 1
# compares this against its 5% firewall; Path 3 has no threshold at all -- the override is
# unconditional, so the number below is reported for the record and gates nothing.
p3_missing = (bars.groupBy("symbol")
                  .agg(func.sum("is_missing_bar").alias("empty_bars"),
                       func.count("*").alias("slots"),
                       func.avg(func.col("is_missing_bar").cast("double")).alias("rate"))
                  .orderBy("symbol").collect())
for row in p3_missing:
    print(f"   {row['symbol']}  {row['empty_bars']:>3} empty / {row['slots']:,} slots "
          f"({100 * row['rate']:.3f}%)")

p3_empty_total = sum(r["empty_bars"] for r in p3_missing)
verdict(p3_empty_total > 0,
        f"Step 4 native missingness: {p3_empty_total} empty bars carried forward as unobserved "
        f"latent states -- no imputation, no row drop, row count unchanged at {bars.count():,}")

# Named individually because each is a real temptation with real precedent in OHLCV code, and
# each would be invisible in a diff: none of them changes the row count or leaves a null behind.
verdict(False, "Step 4 impute: forward-fill close, coalesce(volume, 0), back-fill open from the "
               "next bar, median fill, complete-case drop -- all fabricate a print that never "
               "happened; Path 3 preserves the gap as the honest representation", banned=True)

   BTCUSDT    0 empty / 1,440 slots (0.000%)
   ETHUSDT    5 empty / 1,440 slots (0.347%)
   SOLUSDT    3 empty / 1,440 slots (0.208%)


APPLIES  -- Step 4 native missingness: 8 empty bars carried forward as unobserved latent states -- no imputation, no row drop, row count unchanged at 4,320


BANNED   -- Step 4 impute: forward-fill close, coalesce(volume, 0), back-fill open from the next bar, median fill, complete-case drop -- all fabricate a print that never happened; Path 3 preserves the gap as the honest representation


False

In [50]:
# What a missing bar looks like on the way through, and what it does to its SUCCESSOR. An empty
# bar costs TWO rewards, not one: its own (no close) and the next bar's (no previous close).
# That second cost is the one that gets forgotten, and it is why the unobserved count below is
# roughly double the empty-bar count rather than equal to it.
p3_win = Window.partitionBy("symbol").orderBy("bar_us")

# lag(), NOT last(..., ignorenulls=True). The ignorenulls form reaches back over the gap and
# silently compares against a close from 10 seconds earlier -- forward-fill wearing a window
# function's clothes, and exactly what the override forbids.
p3 = bars.withColumn("prev_close", func.lag("close").over(p3_win))

(p3.filter(func.col("symbol") == "ETHUSDT")
   .filter((func.col("close").isNull()) | (func.col("prev_close").isNull()))
   .select("symbol", "bar_open_time_utc", "n_ticks", "is_missing_bar",
           "close", "prev_close", "volume", "taker_buy_qty")
   .orderBy("bar_us").show(6, truncate=False))

+-------+-------------------+-------+--------------+-------------+-------------+-----------+-------------+
|symbol |bar_open_time_utc  |n_ticks|is_missing_bar|close        |prev_close   |volume     |taker_buy_qty|
+-------+-------------------+-------+--------------+-------------+-------------+-----------+-------------+
|ETHUSDT|2025-01-01 00:00:00|39     |0             |3337.42000000|NULL         |1.79470000 |0.12920000   |
|ETHUSDT|2025-01-01 00:25:30|0      |1             |NULL         |3349.75000000|NULL       |NULL         |
|ETHUSDT|2025-01-01 00:25:35|1      |0             |3349.76000000|NULL         |0.02990000 |0.02990000   |
|ETHUSDT|2025-01-01 00:30:05|0      |1             |NULL         |3350.00000000|NULL       |NULL         |
|ETHUSDT|2025-01-01 00:30:10|53     |0             |3349.35000000|NULL         |11.35160000|1.11780000   |
|ETHUSDT|2025-01-01 01:24:45|0      |1             |NULL         |3353.79000000|NULL       |NULL         |
+-------+-------------------+-------+

### 5. Diagnostics — APPLIES

> "Stream diagnostics; high-frequency interaction frame assembly."

The only Step 5 of the three paths that builds something rather than only observing. Two
jobs: measure the replay as if it were an arrival process, and decide what the reward is.

#### a. Stream diagnostics

In [51]:
# Arrival-process view of the replay: how much evidence each pull can carry, and how often a
# pull returns nothing at all. n_ticks is the per-slot "impression volume"; the unobserved
# count is the number of slots where the reward is genuinely absent rather than zero.
p3 = p3.withColumn(
    "reward_strict",                      # close > prev close
    func.when(func.col("close").isNull() | func.col("prev_close").isNull(), None)
        .otherwise((func.col("close") > func.col("prev_close")).cast("int"))
).withColumn(
    "reward_loose",                       # close >= prev close  (the alternative, kept for 5b)
    func.when(func.col("close").isNull() | func.col("prev_close").isNull(), None)
        .otherwise((func.col("close") >= func.col("prev_close")).cast("int"))
).withColumn(
    "direction",
    func.when(func.col("close").isNull() | func.col("prev_close").isNull(), None)
        .when(func.col("close") > func.col("prev_close"), "UP")
        .when(func.col("close") < func.col("prev_close"), "DOWN")
        .otherwise("FLAT")
).cache()

(p3.groupBy("symbol").agg(
    func.expr("percentile_approx(n_ticks, 0.5)").alias("median_ticks"),
    func.min("n_ticks").alias("min_ticks"),
    func.max("n_ticks").alias("max_ticks"),
    func.sum(func.col("reward_strict").isNull().cast("int")).alias("reward_unobserved"),
    func.sum("is_missing_bar").alias("empty_bars"),
 ).orderBy("symbol").show())

# Replay order is a property of the key, so it is asserted rather than assumed: bar_us is the
# bucket start in epoch microseconds and sorting on it IS chronological order. Sorting on
# bar_open_time_utc would be identical only while the session timezone stays UTC.
p3_ordered = [r[0] for r in p3.filter(func.col("symbol") == "BTCUSDT")
                              .select("bar_us").orderBy("bar_us").limit(3).collect()]
print("first three slot keys:", p3_ordered, "| step:", p3_ordered[1] - p3_ordered[0], "us")

verdict(True, "Step 5 stream diagnostics: replay is ordered by bar_us (5,000,000us step); "
              f"{p3.filter(func.col('reward_strict').isNull()).count()} of {bars.count():,} "
              f"arm-slots have an unobserved reward and will decay without incrementing")

+-------+------------+---------+---------+-----------------+----------+
| symbol|median_ticks|min_ticks|max_ticks|reward_unobserved|empty_bars|
+-------+------------+---------+---------+-----------------+----------+
|BTCUSDT|          26|        2|     2508|                1|         0|
|ETHUSDT|          34|        0|      862|               11|         5|
|SOLUSDT|          41|        0|      766|                7|         3|
+-------+------------+---------+---------+-----------------+----------+

first three slot keys: [1735689600000000, 1735689605000000, 1735689610000000] | step: 5000000 us
APPLIES  -- Step 5 stream diagnostics: replay is ordered by bar_us (5,000,000us step); 19 of 4,320 arm-slots have an unobserved reward and will decay without incrementing


True

#### b. Reward definition — and what it costs

The reward is not read off the data; it is chosen, and the choice changes the winner. Below,
the same 4,320 bars are scored two ways: `close > prev` (a strict up-tick) and
`close >= prev` (up *or unchanged*). Both are defensible sentences in English. They rank the
arms in opposite orders.

The mechanism is the flat class. At 5-second resolution a large share of bars close exactly
where the last one did, and the two definitions differ only in who gets credit for those. Up
and down are close to symmetric on every symbol, so up-share ≈ (1 − flat)/2 — which means a
difference in *flat share* of d points shows up as a difference of about d/2 in the strict
reward, with the sign flipped. The apparent edge is the flat class splitting, not direction.

In [52]:
p3_dir = (p3.filter(func.col("direction").isNotNull())
            .groupBy("symbol").pivot("direction", ["UP", "FLAT", "DOWN"]).count()
            .orderBy("symbol").collect())

print(f"{'arm':<9} {'n':>6} {'up%':>7} {'flat%':>7} {'down%':>7} | "
      f"{'>  (strict)':>12} {'>= (loose)':>12}")
p3_rates = {}
for row in p3_dir:
    n = row["UP"] + row["FLAT"] + row["DOWN"]
    up, flat, down = 100 * row["UP"] / n, 100 * row["FLAT"] / n, 100 * row["DOWN"] / n
    p3_rates[row["symbol"]] = (up, flat, down)
    print(f"{row['symbol']:<9} {n:>6,} {up:>7.2f} {flat:>7.2f} {down:>7.2f} | "
          f"{up:>11.2f}% {up + flat:>11.2f}%")

p3_best_strict = max(p3_rates, key=lambda s: p3_rates[s][0])
p3_best_loose = max(p3_rates, key=lambda s: p3_rates[s][0] + p3_rates[s][1])
print(f"\nbest arm under '>' : {p3_best_strict}   best arm under '>=': {p3_best_loose}")
print("up/down asymmetry (pp): " + ", ".join(
    f"{s} {abs(v[0] - v[2]):.2f}" for s, v in sorted(p3_rates.items())))

# The arithmetic identity, computed rather than asserted: with up ~ down, the strict-reward gap
# between two arms is about half their flat-share gap, opposite in sign.
p3_a, p3_b = "BTCUSDT", "ETHUSDT"
p3_flat_gap = p3_rates[p3_a][1] - p3_rates[p3_b][1]
p3_up_gap = p3_rates[p3_b][0] - p3_rates[p3_a][0]
print(f"{p3_b} - {p3_a}: strict-reward gap {p3_up_gap:.2f}pp vs half the flat-share gap "
      f"{p3_flat_gap / 2:.2f}pp")

verdict(True, f"Step 5 reward: '>' picks {p3_best_strict}, '>=' picks {p3_best_loose} -- the "
              f"ranking inverts on the definition, so the reward is a modelling choice and is "
              f"declared here, not discovered downstream")

arm            n     up%   flat%   down% |  >  (strict)   >= (loose)
BTCUSDT    1,439   36.34   27.59   36.07 |       36.34%       63.93%
ETHUSDT    1,429   40.24   19.03   40.73 |       40.24%       59.27%
SOLUSDT    1,433   41.10   18.84   40.06 |       41.10%       59.94%

best arm under '>' : SOLUSDT   best arm under '>=': BTCUSDT
up/down asymmetry (pp): BTCUSDT 0.28, ETHUSDT 0.49, SOLUSDT 1.05
ETHUSDT - BTCUSDT: strict-reward gap 3.89pp vs half the flat-share gap 4.28pp
APPLIES  -- Step 5 reward: '>' picks SOLUSDT, '>=' picks BTCUSDT -- the ranking inverts on the definition, so the reward is a modelling choice and is declared here, not discovered downstream


True

The full month tells the same story at a larger flat share and with the two leaders swapped;
those figures were measured by the ingest job over all 340,971,834 ticks and are **not**
computed in this notebook — they are quoted here only so the two-hour extract is not mistaken
for the month. On the month at 5s the flat class runs 14.37% (SOL) / 13.25% (ETH) /
20.04% (BTC), against 18–28% in this quiet 00:00–02:00 window on New Year's Day. The
extract is not representative of the month, and this section says so rather than generalising
from it.

**The definition taken forward is the strict one, `close > prev`.** It at least requires a
print above the previous one, whereas `>=` pays an arm for standing still. That choice does
not rescue the exercise: under either definition the arm ordering is essentially an ordering
of tick-flatness, and the winner is the symbol whose 5-second bars move most often — a
property of tick size and quote density, not of an exploitable edge. Nothing below should be
read as a trading result.

### 6. Topology — APPLIES

> "Cyclical time coordinates; high-frequency interaction frames."

Path 3 gets the cyclical encoding but not the cross-correlation matrix that Paths 1 and 2
build — consistent, since Step 6's dependency schema exists to guide Step 9's regulariser and
Step 9 is banned here.

The framework names the technique and never names a period (judgement call). Crypto trades
24/7, so there is no session open to anchor on the way an equity feed would force. Two
periods are emitted: **hour-of-day (24)** as the production coordinate, and
**minute-of-hour (60)** as the one this two-hour extract actually exercises. The hour-of-day
pair occupies exactly 2 of its 24 positions here, which is a fact about the extract and is
printed rather than glossed.

In [53]:
# Hand-rolled: pyspark.ml has no cyclical encoder. sin AND cos together, never one alone --
# a lone sin() maps 01:00 and 11:00 to the same coordinate. hour()/minute() are safe here only
# because the session timezone is pinned to UTC in setup; on this box the default is
# Africa/Johannesburg, under which every hour bucket below would be shifted by two.
p3 = (p3
      .withColumn("hour_utc", func.hour("bar_open_time_utc"))
      .withColumn("minute_utc", func.minute("bar_open_time_utc"))
      .withColumn("hod_sin", func.sin(2 * math.pi * func.col("hour_utc") / 24))
      .withColumn("hod_cos", func.cos(2 * math.pi * func.col("hour_utc") / 24))
      .withColumn("moh_sin", func.sin(2 * math.pi * func.col("minute_utc") / 60))
      .withColumn("moh_cos", func.cos(2 * math.pi * func.col("minute_utc") / 60)))

# Sampled at quarter-hours rather than .show(4), which on a 5-second grid would print four
# consecutive rows of the same coordinate and demonstrate nothing.
(p3.filter((func.col("symbol") == "BTCUSDT")
           & (func.col("minute_utc").isin(0, 15, 30, 45))
           & (func.second("bar_open_time_utc") == 0))
   .select("bar_open_time_utc", "hour_utc", "hod_sin", "hod_cos",
           "minute_utc", "moh_sin", "moh_cos")
   .orderBy("bar_us").show(8, truncate=False))

p3_hours = sorted(r[0] for r in p3.select("hour_utc").distinct().collect())
verdict(True, f"Step 6 cyclical: hour-of-day (period 24) and minute-of-hour (period 60) as "
              f"sin/cos pairs; this extract visits {len(p3_hours)}/24 hour positions {p3_hours}, "
              f"so the hour coordinate is near-degenerate on the sample and real on the month")

+-------------------+--------+-------------------+------------------+----------+--------------------+-----------------------+
|bar_open_time_utc  |hour_utc|hod_sin            |hod_cos           |minute_utc|moh_sin             |moh_cos                |
+-------------------+--------+-------------------+------------------+----------+--------------------+-----------------------+
|2025-01-01 00:00:00|0       |0.0                |1.0               |0         |0.0                 |1.0                    |
|2025-01-01 00:15:00|0       |0.0                |1.0               |15        |1.0                 |2.83276944882399E-16   |
|2025-01-01 00:30:00|0       |0.0                |1.0               |30        |5.66553889764798E-16|-1.0                   |
|2025-01-01 00:45:00|0       |0.0                |1.0               |45        |-1.0                |-1.8369701987210297E-16|
|2025-01-01 01:00:00|1       |0.25881904510252074|0.9659258262890683|0         |0.0                 |1.0              

APPLIES  -- Step 6 cyclical: hour-of-day (period 24) and minute-of-hour (period 60) as sin/cos pairs; this extract visits 2/24 hour positions [0, 1], so the hour coordinate is near-degenerate on the sample and real on the month


True

### 7. Feature Engineering — APPLIES

> "Interaction frame preparation."

The lightest Step 7 of the three paths: no cross-products, because the Path 3 engines learn
co-occurrence structure themselves. What Step 7 owes them is the *frame* — a transaction per
instant, holding the tokens that were observed at that instant, as raw strings.

The transaction is one **slot**, not one bar: all three symbols share the bar grid Step 3
built, so grouping them by `bar_us` is a grouping and not a join. (Aligning the symbols on
`event_time` instead is the phantom-row trap the entryway banned at Step 1 — BTC, ETH and SOL
routinely print inside the same microsecond.) The framework never says how to make a basket
out of a time series; this construction is the notebook's, not the document's.

Tokens are namespaced by symbol and kept as strings. A slot where a symbol had no trades
contributes `<SYM>_NO_TRADES` and no price-derived token — the shorter basket *is* the
representation of the gap.

In [54]:
# Decimal division, and the ratio stays decimal: the double cast is deferred to the pyspark.ml
# calls further down, where it is unavoidable. Empty bars have NULL volume, so the ratio is NULL
# and every when() below falls through to NULL -- no token, by construction rather than by a
# special case.
p3 = p3.withColumn("taker_ratio", func.col("taker_buy_qty") / func.col("volume"))

p3 = p3.withColumn(
    "regime_token",
    func.when(func.col("taker_ratio").isNull(), None)
        .when(func.col("taker_ratio") >= P3_TAKER_EXTREME, "TAKER_EXTREME")
        .when(func.col("taker_ratio") >= P3_TAKER_HEAVY, "TAKER_HEAVY")
        .when(func.col("taker_ratio") >= P3_MAKER_HEAVY, "BALANCED")
        .when(func.col("taker_ratio") >= P3_MAKER_EXTREME, "MAKER_HEAVY")
        .otherwise("MAKER_EXTREME"))

# explode-then-filter, NOT a fixed-width struct and NOT a sentinel string: the basket has to be
# genuinely variable-length, so an unobservable token is dropped rather than encoded as "" or
# "UNKNOWN", either of which association mining would dutifully count as an item.
# array_compact() would say this in one call but landed in Spark 3.4; Glue 4.0 is 3.3.0, the
# same constraint that forced expr() over func.pmod in the entryway.
p3_tokens = (p3.select(
    "bar_us", "hour_utc",
    func.explode(func.array(
        func.when(func.col("is_missing_bar") == 1,
                  func.concat_ws("_", func.col("symbol"), func.lit("NO_TRADES"))),
        func.when(func.col("regime_token").isNotNull(),
                  func.concat_ws("_", func.col("symbol"), func.col("regime_token"))),
        func.when(func.col("direction").isNotNull(),
                  func.concat_ws("_", func.col("symbol"), func.concat(func.lit("DIR_"),
                                                                      func.col("direction")))),
    )).alias("token"))
    .filter(func.col("token").isNotNull()))

# No bare "<SYM> was present" token. The grid is dense, so such a token would sit in 100% of
# transactions -- the association-mining twin of all_best_match, adding a constant to every
# support count and dragging every lift toward 1. Symbol identity is carried by the namespace
# prefix on the tokens that DO vary, which is what "preserve the raw token" is protecting.
p3_frame = (p3_tokens.groupBy("bar_us")
            .agg(func.collect_set("token").alias("items"),
                 func.min("hour_utc").alias("hour_utc"))
            .withColumn("items", func.array_union(
                func.col("items"),
                func.array(func.format_string("HOUR_%02d", func.col("hour_utc")))))
            .select("bar_us", "items").cache())

print(f"interaction frame: {p3_frame.count():,} transactions "
      f"(one per slot, {len(P3_ARMS)} arms folded into each)")

# A full basket, and a basket where one arm went quiet -- printed side by side so the shorter
# set is visible as a shorter set.
p3_gap_slots = [r[0] for r in p3.filter(func.col("is_missing_bar") == 1)
                                .select("bar_us").orderBy("bar_us").limit(1).collect()]
p3_frame.filter(func.col("bar_us").isin(p3_gap_slots + [p3_ordered[1]])).show(2, truncate=False)

p3_support = (p3_frame.select(func.explode("items").alias("token"))
              .groupBy("token").count()
              .withColumn("support", func.col("count") / func.lit(p3_frame.count()))
              .orderBy("count"))
print("rarest tokens in the frame:")
p3_support.show(6, truncate=False)

verdict(True, f"Step 7 interaction frame: {p3_frame.count():,} variable-length token sets, "
              f"{p3_support.count()} distinct raw string tokens, no cross-products "
              f"(Apriori/ECLAT/RBM learn the co-occurrence themselves)")

interaction frame: 1,440 transactions (one per slot, 3 arms folded into each)


+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|bar_us          |items                                                                                                                        |
+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|1735689605000000|[BTCUSDT_MAKER_HEAVY, ETHUSDT_MAKER_EXTREME, SOLUSDT_BALANCED, ETHUSDT_DIR_DOWN, SOLUSDT_DIR_DOWN, BTCUSDT_DIR_DOWN, HOUR_00]|
|1735691130000000|[BTCUSDT_MAKER_EXTREME, SOLUSDT_TAKER_HEAVY, ETHUSDT_NO_TRADES, SOLUSDT_DIR_FLAT, BTCUSDT_DIR_FLAT, HOUR_00]                 |
+----------------+-----------------------------------------------------------------------------------------------------------------------------+

rarest tokens in the frame:


+---------------------+-----+---------------------+
|token                |count|support              |
+---------------------+-----+---------------------+
|SOLUSDT_NO_TRADES    |3    |0.0020833333333333333|
|ETHUSDT_NO_TRADES    |5    |0.003472222222222222 |
|SOLUSDT_BALANCED     |117  |0.08125              |
|ETHUSDT_BALANCED     |128  |0.08888888888888889  |
|BTCUSDT_BALANCED     |134  |0.09305555555555556  |
|SOLUSDT_MAKER_EXTREME|220  |0.1527777777777778   |
+---------------------+-----+---------------------+
only showing top 6 rows



APPLIES  -- Step 7 interaction frame: 1,440 variable-length token sets, 28 distinct raw string tokens, no cross-products (Apriori/ECLAT/RBM learn the co-occurrence themselves)


True

### 8. Pruning — BANNED

> "Raw categorical tokens must be preserved for association mining."

> "Apriori and ECLAT need raw categorical token strings … One-hot encoding explodes these
> into thousands of sparse columns — destroying the co-occurrence matrix structure that
> association mining depends on entirely."

Three mechanisms, of which the third is the one this data can demonstrate outright.
ECLAT's support is `|TID(A) ∩ TID(B)| / N`, an intersection of per-item transaction lists;
after one-hot the item is a *column position* in a fixed schema and there is no list left to
intersect. Apriori's candidate generation joins frequent k-itemsets into (k+1)-itemsets, a
lattice walk defined over sets — a fixed-width row makes every basket the same width and the
join has nothing to join.

And then the pruning itself. A one-hot dummy for a token present in a fraction *p* of
transactions has variance *p*(1−*p*) ≈ *p*, so a variance threshold prunes in **ascending
order of token rarity** — it deletes exactly the low-support tail that association mining is
hunting. Below, `VarianceThresholdSelector` is actually fitted, on real one-hot columns, and
the tokens it deletes are named.

In [55]:
verdict(False, "Step 8 prune: one-hot encoding + VarianceThreshold destroys the token identity "
               "and the variable-length set geometry that Apriori, ECLAT and the RBM consume -- "
               "raw categorical tokens are preserved instead", banned=True)

# The demonstration, not an assertion. The frame IS one-hot encoded here and the selector IS
# fitted -- this cell runs the banned operation once, on a copy, so its cost can be measured.
# p3_frame is untouched and everything downstream still reads the raw string sets.
p3_vocab = [r["token"] for r in p3_support.orderBy("token").collect()]
p3_onehot = p3_frame.select(
    "bar_us",
    *[func.array_contains("items", tok).cast("double").alias(tok) for tok in p3_vocab])

p3_assembled = (VectorAssembler(inputCols=p3_vocab, outputCol="features")
                .transform(p3_onehot))

# threshold 0.0049 = p(1-p) at p ~= 0.5%, i.e. "delete anything seen in under ~0.5% of slots".
# Any threshold above zero picks a rarity cut-off; this one is chosen to sit just above the
# NO_TRADES tokens so the deletion is visible on an extract this small.
P3_VARIANCE_THRESHOLD = 0.0049
p3_selector = VarianceThresholdSelector(featuresCol="features", outputCol="selected",
                                        varianceThreshold=P3_VARIANCE_THRESHOLD).fit(p3_assembled)
p3_kept = set(p3_selector.selectedFeatures)
p3_dropped = [tok for i, tok in enumerate(p3_vocab) if i not in p3_kept]

print(f"one-hot: 1 items column -> {len(p3_vocab)} binary columns")
print(f"VarianceThresholdSelector(threshold={P3_VARIANCE_THRESHOLD}) keeps "
      f"{len(p3_kept)}, deletes {len(p3_dropped)}:")
p3_counts = {r["token"]: r["count"] for r in p3_support.collect()}
for tok in p3_dropped:
    n = p3_counts[tok]
    p = n / p3_frame.count()
    print(f"   DELETED  {tok:<22} seen {n:>4} / {p3_frame.count():,}  "
          f"(p={p:.5f}, variance={p * (1 - p):.6f})")

# The zero-variance column the entryway deliberately refused to drop at Step 2, because Step 8
# is path-isolated -- and here is the path where Step 8 never runs, so it survives to the
# engines. On Path 1 it would be the textbook VarianceThreshold casualty.
p3_bm = bars.agg(func.avg(func.col("all_best_match").cast("double")).alias("mean"),
                 func.var_samp(func.col("all_best_match").cast("double")).alias("var")).first()
print(f"\nall_best_match: mean={p3_bm['mean']}, sample variance={p3_bm['var']} "
      f"-- genuinely constant, and it survives here only because Step 8 is banned")

BANNED   -- Step 8 prune: one-hot encoding + VarianceThreshold destroys the token identity and the variable-length set geometry that Apriori, ECLAT and the RBM consume -- raw categorical tokens are preserved instead


one-hot: 1 items column -> 28 binary columns
VarianceThresholdSelector(threshold=0.0049) keeps 26, deletes 2:


   DELETED  ETHUSDT_NO_TRADES      seen    5 / 1,440  (p=0.00347, variance=0.003460)


   DELETED  SOLUSDT_NO_TRADES      seen    3 / 1,440  (p=0.00208, variance=0.002079)

all_best_match: mean=1.0, sample variance=0.0 -- genuinely constant, and it survives here only because Step 8 is banned


### 9. Regularisation — BANNED

> "Elastic Net destroys multi-item relationship detection."

> "a product pairing that appears rarely (low marginal frequency) can have a very high lift
> ratio … Elastic Net would silently delete these rare but high-value itemset relationships
> before the bandit engine ever sees them."

The two criteria optimise different functions and disagree precisely on the tail. Elastic Net
minimises `‖y − Xw‖² + λ(α‖w‖₁ + (1−α)‖w‖²₂)`; a column's ability to reduce squared error
scales with its variance, so a token present in 0.3% of transactions is among the first
coefficients L1 drives to exactly zero regardless of how informative it is *conditionally*.
Lift is `P(B|A) / P(B)` — a conditional ratio with no `P(A)` in the numerator, *maximised* by
rare antecedents. Support enters only as an admission floor, never as a ranking.

There is also a second, structural reason the framework does not state: **Elastic Net is
supervised and Path 3 has no `y`.** The reward is per-arm and per-timestep, and the arms that
were not pulled have no observation at all. To run the banned step below at all, a target has
to be manufactured — which is itself the argument. That inference is the notebook's, not the
document's.

Below: FP-Growth mines the raw frame and is ranked by lift; then Elastic Net is fitted against
a manufactured target and the mined token's coefficient is read back.

In [56]:
verdict(False, "Step 9 regularise: Elastic Net ranks by marginal predictiveness and zeroes the "
               "low-frequency tail; association mining ranks by conditional lift, which is "
               "maximised there -- and Path 3 has no static y to regress against anyway",
        banned=True)

# Spark ships FP-Growth, which the framework lists as MODEL 3, the "industry scale fallback
# when Apriori encounters horizontal bottlenecks". Apriori and ECLAT themselves are [GAP] in
# pyspark.ml. The support/confidence/lift arithmetic is identical; the scan geometry is not.
p3_fp = FPGrowth(itemsCol="items", minSupport=P3_MIN_SUPPORT,
                 minConfidence=P3_MIN_CONFIDENCE).fit(p3_frame)

p3_rules = (p3_fp.associationRules
            .withColumn("n_ante", func.size("antecedent"))
            .orderBy(func.desc("lift"), func.asc("support")))
print(f"FPGrowth(minSupport={P3_MIN_SUPPORT}, minConfidence={P3_MIN_CONFIDENCE}): "
      f"{p3_fp.freqItemsets.count():,} frequent itemsets, {p3_rules.count():,} rules")
p3_rules.select("antecedent", "consequent", "support", "confidence", "lift").show(5,
                                                                                  truncate=False)

# The framework's own floor excludes part of the tail before Elastic Net ever gets a chance to.
p3_below_floor = [tok for tok in p3_vocab
                  if p3_counts[tok] / p3_frame.count() < P3_MIN_SUPPORT]
print(f"tokens below the framework's own min_support={P3_MIN_SUPPORT} floor and therefore "
      f"invisible to FP-Growth: {p3_below_floor}")
print("   (that tail is the RBM's job -- 'sampling its hidden units reconstructs the most "
      "probable missing items' -- and the RBM is [GAP] in pyspark.ml)")

BANNED   -- Step 9 regularise: Elastic Net ranks by marginal predictiveness and zeroes the low-frequency tail; association mining ranks by conditional lift, which is maximised there -- and Path 3 has no static y to regress against anyway


FPGrowth(minSupport=0.003, minConfidence=0.5): 7,040 frequent itemsets, 9,627 rules
+---------------------------------------------------------------------------------------------------------+-----------------------+--------------------+----------+------------------+
|antecedent                                                                                               |consequent             |support             |confidence|lift              |
+---------------------------------------------------------------------------------------------------------+-----------------------+--------------------+----------+------------------+
|[ETHUSDT_DIR_FLAT, ETHUSDT_TAKER_HEAVY, BTCUSDT_DIR_DOWN, SOLUSDT_DIR_DOWN, HOUR_01]                     |[SOLUSDT_MAKER_EXTREME]|0.003472222222222222|1.0       |6.545454545454545 |
|[BTCUSDT_DIR_FLAT, ETHUSDT_TAKER_EXTREME, SOLUSDT_TAKER_EXTREME, ETHUSDT_DIR_UP, SOLUSDT_DIR_UP, HOUR_01]|[BTCUSDT_BALANCED]     |0.003472222222222222|0.5       |5.3731343283582085|
|

tokens below the framework's own min_support=0.003 floor and therefore invisible to FP-Growth: ['SOLUSDT_NO_TRADES']
   (that tail is the RBM's job -- 'sampling its hidden units reconstructs the most probable missing items' -- and the RBM is [GAP] in pyspark.ml)


In [57]:
# The banned step, run once so the deletion is a measurement. y is MANUFACTURED: BTC's own
# strict reward for the slot, with BTC's direction tokens removed from X because they encode
# the label exactly. There is no such y on a real bandit stream, which is the second half of
# the ban's argument.
p3_x_cols = [t for t in p3_vocab if not t.startswith("BTCUSDT_DIR_")]
p3_en_data = (p3_onehot.select("bar_us", *p3_x_cols)
              .join(p3.filter(func.col("symbol") == "BTCUSDT")
                      .select("bar_us", func.col("reward_strict").alias("label")),
                    on="bar_us")
              .filter(func.col("label").isNotNull()))
# .cast("double") here and only here: pyspark.ml refuses anything else, and this is the "last
# step" the decimal rule allows it at.
p3_en_data = (VectorAssembler(inputCols=p3_x_cols, outputCol="features")
              .transform(p3_en_data.withColumn("label", func.col("label").cast("double"))))

# The rarest token FP-Growth can still see, i.e. the one sitting on the boundary the two
# criteria disagree about, against the most common token in the frame.
p3_rare = min((t for t in p3_x_cols
               if p3_counts[t] / p3_frame.count() >= P3_MIN_SUPPORT), key=p3_counts.get)
p3_common = max(p3_x_cols, key=p3_counts.get)
print(f"rare token   {p3_rare:<24} seen {p3_counts[p3_rare]:>4} / {p3_frame.count():,}")
print(f"common token {p3_common:<24} seen {p3_counts[p3_common]:>4} / {p3_frame.count():,}\n")

# standardization=False, and this is NOT a detail. Spark's LinearRegression standardises the
# feature columns before applying the penalty BY DEFAULT, which rescales every indicator to
# unit variance and thereby removes exactly the rarity disadvantage the framework's argument
# rests on. Left at the default, a rare token outlives a common one and the ban looks wrong.
# It is also, on Path 3, a Step 10 scaler smuggled inside a Step 9 estimator.
p3_paths = {}
print(f"{'regParam':>9} {'nonzero':>8} {p3_rare:>20} {p3_common:>10}")
for reg in [0.0, 0.0005, 0.001, 0.002, 0.005, 0.02]:
    # elasticNetParam=1.0 is pure L1 -- the half of Elastic Net that does the zeroing.
    p3_lr = LinearRegression(featuresCol="features", labelCol="label",
                             elasticNetParam=1.0, regParam=reg, maxIter=100,
                             standardization=False).fit(p3_en_data)
    p3_paths[reg] = p3_lr.coefficients.toArray()
    nz = int((p3_paths[reg] != 0.0).sum())
    print(f"{reg:>9} {nz:>8} {p3_paths[reg][p3_x_cols.index(p3_rare)]:>20.6f} "
          f"{p3_paths[reg][p3_x_cols.index(p3_common)]:>10.6f}")

# The lambda at which the rare token is gone but the frame is not: the exact window in which
# Elastic Net would hand the engines a vocabulary with the tail already deleted.
p3_lambda = min(r for r, c in p3_paths.items()
                if c[p3_x_cols.index(p3_rare)] == 0.0
                and int((c != 0.0).sum()) > len(p3_x_cols) // 2)
print(f"\nat regParam={p3_lambda}: {p3_rare} is zero while "
      f"{int((p3_paths[p3_lambda] != 0.0).sum())}/{len(p3_x_cols)} columns survive")

# And the itemset side of the same boundary. The highest-lift rule that lives entirely inside
# the feature set: it fires on 5 of 1,440 slots and multiplies its consequent's probability
# several-fold -- support at the floor, lift at the top, which is the shape Elastic Net cannot
# see. Note the antecedent is a FIVE-token co-occurrence: a linear model selects columns, not
# itemsets, so even with every token retained it has no coefficient that could represent it.
p3_xset = set(p3_x_cols)
p3_top = next(r for r in p3_rules.orderBy(func.desc("lift")).limit(400).collect()
              if p3_xset.issuperset(r["antecedent"]) and p3_xset.issuperset(r["consequent"]))
print(f"\nhighest-lift rule inside the feature set:")
print(f"   {p3_top['antecedent']} -> {p3_top['consequent']}")
print(f"   support {p3_top['support']:.5f} ({round(p3_top['support'] * p3_frame.count())} slots) "
      f"| confidence {p3_top['confidence']:.3f} | lift {p3_top['lift']:.3f}")
for tok in list(p3_top["antecedent"]) + list(p3_top["consequent"]):
    coef = p3_paths[p3_lambda][p3_x_cols.index(tok)]
    print(f"   {tok:<24} L1 coefficient at regParam={p3_lambda}: {coef:>10.6f}"
          f"{'   <- zeroed' if coef == 0.0 else ''}")

rare token   ETHUSDT_NO_TRADES        seen    5 / 1,440
common token HOUR_00                  seen  720 / 1,440

 regParam  nonzero    ETHUSDT_NO_TRADES    HOUR_00


      0.0       25             0.638746   0.005285


   0.0005       20             0.121299   0.004858


    0.001       16             0.000000   0.004285


    0.002       15             0.000000   0.002906


    0.005       10             0.000000   0.000000


     0.02        4             0.000000   0.000000

at regParam=0.001: ETHUSDT_NO_TRADES is zero while 16/25 columns survive

highest-lift rule inside the feature set:
   ['SOLUSDT_DIR_FLAT', 'ETHUSDT_DIR_FLAT', 'ETHUSDT_TAKER_HEAVY', 'BTCUSDT_MAKER_EXTREME'] -> ['SOLUSDT_TAKER_HEAVY']


   support 0.00347 (5 slots) | confidence 1.000 | lift 4.645
   SOLUSDT_DIR_FLAT         L1 coefficient at regParam=0.001:  -0.015547
   ETHUSDT_DIR_FLAT         L1 coefficient at regParam=0.001:  -0.037878
   ETHUSDT_TAKER_HEAVY      L1 coefficient at regParam=0.001:   0.051734
   BTCUSDT_MAKER_EXTREME    L1 coefficient at regParam=0.001:  -0.165221
   SOLUSDT_TAKER_HEAVY      L1 coefficient at regParam=0.001:   0.000000   <- zeroed


### 10. Scaling — BANNED

> "Scaling destroys Bayesian conjugate prior temporal updating."

> "The conjugate update equation `α_new = γα + x` produces valid Beta distribution parameters
> only when α and β are raw non-negative counts. StandardScaler would transform α = 23 and
> β = 89 into standardised z-scores. The posterior arithmetic breaks entirely."

α and β are counts, not features. Three things break at once, and all three are shown below
on this run's own numbers rather than the framework's worked example:

1. **Domain violation.** The Beta density is a distribution only for α > 0 and β > 0.
   Mean-centring makes roughly half the arms negative by construction. `np.random.beta` on a
   negative parameter does not return a wrong number — it raises. Thompson sampling's single
   primitive has nothing to draw from and the engine halts rather than degrading.
2. **The evidence count is annihilated.** α + β is the arm's effective sample size and the
   entire source of the explore/exploit signal, through
   `Var[θ] = αβ / ((α+β)²(α+β+1))`. Standardisation is *defined* to make the values
   unit-variance, so every arm ends up with the same implied sample size. Note this is a
   magnitude property, not a rank property — scaling preserves the ordering of α across arms,
   which is exactly why the failure is easy to miss on inspection.
3. **The update loop stops closing.** `γα + x` works because α and x are in the same unit —
   counts. After scaling the state is in units of σ and the increment is still one impression,
   and in a replay μ and σ move with every new row, so the basis under γ shifts on every step.

The α/β counts themselves are a Spark aggregate — one shuffle over 4,320 rows. Everything
after that is conjugate arithmetic on three numbers per arm and belongs on the driver.

In [58]:
verdict(False, "Step 10 scale: StandardScaler on alpha/beta yields negative shape parameters "
               "(no Beta density), a uniform effective sample size (no posterior width) and a "
               "state whose units no longer match the increment (no conjugacy)", banned=True)

# Spark's half: the counts. Full-information, gamma=1 -- every slot's reward for every arm,
# which is the quantity a replay can see and a live bandit cannot. The gamma-decayed, one-pull-
# per-slot version is the next cell; this one exists to give Step 10 something real to break.
p3_counts_df = (p3.groupBy("symbol").agg(
    func.sum("reward_strict").alias("wins"),
    func.sum((func.col("reward_strict") == 0).cast("int")).alias("losses"),
    func.sum(func.col("reward_strict").isNull().cast("int")).alias("unobserved"))
    .orderBy("symbol"))
p3_counts_df.show()

p3_ab = {r["symbol"]: (P3_PRIOR_A + r["wins"], P3_PRIOR_B + r["losses"])
         for r in p3_counts_df.collect()}

print(f"{'arm':<9} {'alpha':>9} {'beta':>9} {'alpha+beta':>11} {'mean':>8} {'sd':>8}")
for arm in P3_ARMS:
    a, b = p3_ab[arm]
    n = a + b
    print(f"{arm:<9} {a:>9.1f} {b:>9.1f} {n:>11.1f} {a / n:>8.4f} "
          f"{math.sqrt(a * b / (n * n * (n + 1))):>8.5f}")

# Now the banned transform, applied by hand to make the arithmetic auditable (this is exactly
# what StandardScaler(withMean=True, withStd=True) computes, and doing it inline avoids hiding
# the failure inside a fitted model).
p3_raw_params = np.array([[p3_ab[a][0], p3_ab[a][1]] for a in P3_ARMS], dtype=float)
p3_mu, p3_sigma = p3_raw_params.mean(axis=0), p3_raw_params.std(axis=0, ddof=1)
p3_scaled = (p3_raw_params - p3_mu) / p3_sigma

print(f"\nStandardScaler(withMean=True, withStd=True) on the (alpha, beta) columns:")
print(f"   mu = {p3_mu.round(3)}   sigma = {p3_sigma.round(3)}")
for arm, (a, b) in zip(P3_ARMS, p3_scaled):
    n_raw = sum(p3_ab[arm])
    print(f"   {arm:<9} alpha {a:>8.4f}  beta {b:>8.4f}   "
          f"(effective sample size {n_raw:,.1f} -> {a + b:.4f})")

p3_rng = np.random.default_rng(P3_SEED)
for arm, (a, b) in zip(P3_ARMS, p3_scaled):
    try:
        draw = p3_rng.beta(a, b)
        print(f"   np.random.beta({a:.4f}, {b:.4f}) = {draw:.6f}")
    except ValueError as exc:
        print(f"   np.random.beta({a:.4f}, {b:.4f}) -> ValueError: {exc}")

BANNED   -- Step 10 scale: StandardScaler on alpha/beta yields negative shape parameters (no Beta density), a uniform effective sample size (no posterior width) and a state whose units no longer match the increment (no conjugacy)


+-------+----+------+----------+
| symbol|wins|losses|unobserved|
+-------+----+------+----------+
|BTCUSDT| 523|   916|         1|
|ETHUSDT| 575|   854|        11|
|SOLUSDT| 589|   844|         7|
+-------+----+------+----------+



arm           alpha      beta  alpha+beta     mean       sd
BTCUSDT       524.0     917.0      1441.0   0.3636  0.01267
ETHUSDT       576.0     855.0      1431.0   0.4025  0.01296
SOLUSDT       590.0     845.0      1435.0   0.4111  0.01298

StandardScaler(withMean=True, withStd=True) on the (alpha, beta) columns:
   mu = [563.333 872.333]   sigma = [34.775 39.004]
   BTCUSDT   alpha  -1.1311  beta   1.1452   (effective sample size 1,441.0 -> 0.0141)
   ETHUSDT   alpha   0.3642  beta  -0.4444   (effective sample size 1,431.0 -> -0.0802)
   SOLUSDT   alpha   0.7668  beta  -0.7008   (effective sample size 1,435.0 -> 0.0660)
   np.random.beta(-1.1311, 1.1452) -> ValueError: a <= 0
   np.random.beta(0.3642, -0.4444) -> ValueError: b <= 0
   np.random.beta(0.7668, -0.7008) -> ValueError: b <= 0


### The bandit

Steps 4–10 are done and Path 3 hands straight to the engines — **there is no gate system on
this path**. "Gate system bypassed → LIVE"; "No gate system needed — the posterior is valid at
every moment." Paths 1 and 2 have four gates to clear; showing any of them here would be a
misreading of the framework, not extra rigour.

The replay: 1,440 slots in `bar_us` order. At each slot one arm is drawn by Thompson sampling
— sample `θ ~ Beta(α, β)` for each arm, pull the highest roll — and **only that arm's reward
is read**. The other two arms' rewards are present in the file and are deliberately not
looked at, because a live bandit would never have them.

Two departures from the document, both labelled:

* The decay is applied to the **pulled arm only**, matching the framework's equation
  `α_new = γα + x` literally. An arm that is not pulled does not age.
* When the pulled arm's reward is unobserved (an empty bar, or the bar after one), the update
  is `α ← γα`, `β ← γβ` with **no increment** — the decay alone. Imputing `x = 0` would record
  a failure that did not happen and γ would carry that fiction forward for ~33 pulls. This is
  the arithmetic form of the Step 4 override, and the framework never writes it down; it is an
  inference from the stated override.

In [59]:
# Spark collects the reward matrix: 1,440 rows x 3 arms, pivoted. Small by construction, so the
# conjugate loop below is plain Python on the driver -- there is no bandit anywhere in
# pyspark.ml, and a Structured Streaming foreachBatch would only relocate the same 15 lines.
p3_matrix = [tuple(r) for r in
             (p3.groupBy("bar_us").pivot("symbol", P3_ARMS)
                .agg(func.first("reward_strict"))
                .orderBy("bar_us")
                .select("bar_us", *P3_ARMS).collect())]
print(f"replay matrix: {len(p3_matrix):,} slots x {len(P3_ARMS)} arms, "
      f"first slot {p3_matrix[0]}, last slot {p3_matrix[-1]}")


def p3_replay(seed, gamma=P3_GAMMA):
    """One discounted Thompson-sampling pass over the slots in bar_us order.

    Returns (alpha, beta, pulls, observed) per arm. Only the pulled arm is read, and an
    unobserved reward decays without incrementing -- the Step 4 override in arithmetic.
    """
    rng = np.random.default_rng(seed)
    alpha = {a: P3_PRIOR_A for a in P3_ARMS}
    beta = {a: P3_PRIOR_B for a in P3_ARMS}
    pulls = {a: 0 for a in P3_ARMS}
    observed = {a: 0 for a in P3_ARMS}
    for row in p3_matrix:
        rewards = dict(zip(P3_ARMS, row[1:]))
        arm = max(P3_ARMS, key=lambda a: rng.beta(alpha[a], beta[a]))
        pulls[arm] += 1
        x = rewards[arm]
        if x is None:
            alpha[arm] *= gamma           # decay alone: time passed, no evidence arrived
            beta[arm] *= gamma
        else:
            observed[arm] += 1
            alpha[arm] = gamma * alpha[arm] + x
            beta[arm] = gamma * beta[arm] + (1 - x)
    return alpha, beta, pulls, observed


p3_alpha, p3_beta, p3_pulls, p3_observed = p3_replay(P3_SEED)
p3_total_pulls = sum(p3_pulls.values())

print(f"\nseeded replay (seed={P3_SEED}), {p3_total_pulls:,} pulls")
print(f"{'arm':<9} {'pulls':>7} {'budget':>8} {'observed':>9} {'alpha':>8} {'beta':>8} "
      f"{'post.mean':>10} {'post.sd':>9} {'full-info':>10}")
for arm in P3_ARMS:
    a, b = p3_alpha[arm], p3_beta[arm]
    n = a + b
    fa, fb = p3_ab[arm]
    print(f"{arm:<9} {p3_pulls[arm]:>7,} {p3_pulls[arm] / p3_total_pulls:>7.1%} "
          f"{p3_observed[arm]:>9,} {a:>8.3f} {b:>8.3f} {a / n:>10.4f} "
          f"{math.sqrt(a * b / (n * n * (n + 1))):>9.4f} {fa / (fa + fb):>10.4f}")

p3_winner = max(P3_ARMS, key=lambda a: p3_alpha[a] / (p3_alpha[a] + p3_beta[a]))
print(f"\nfinal arm selection (argmax posterior mean): {p3_winner}")
print(f"largest budget share:                        "
      f"{max(P3_ARMS, key=lambda a: p3_pulls[a])}")

replay matrix: 1,440 slots x 3 arms, first slot (1735689600000000, None, None, None), last slot (1735696795000000, 0, 1, 1)

seeded replay (seed=20250101), 1,440 pulls
arm         pulls   budget  observed    alpha     beta  post.mean   post.sd  full-info
BTCUSDT       212   14.7%       211    9.545   23.738     0.2868    0.0772     0.3636
ETHUSDT       693   48.1%       688   13.301   18.122     0.4233    0.0868     0.4025
SOLUSDT       535   37.2%       535    8.709   24.624     0.2613    0.0750     0.4111

final arm selection (argmax posterior mean): ETHUSDT
largest budget share:                        ETHUSDT


In [60]:
# One trajectory of a discounted bandit is a sample, not a result: gamma=0.97 caps the effective
# sample size at 1/(1-gamma) ~= 33 pulls per arm, so the posterior never becomes confident and
# the final argmax is genuinely seed-dependent. Replaying over many seeds is the honest way to
# report it, and it costs nothing here.
p3_reps = [p3_replay(P3_SEED + i) for i in range(P3_SEED_REPLICATES)]
p3_share = {a: np.array([r[2][a] / p3_total_pulls for r in p3_reps]) for a in P3_ARMS}
p3_wins = {a: sum(1 for al, be, _, _ in p3_reps
                  if max(P3_ARMS, key=lambda k: al[k] / (al[k] + be[k])) == a)
           for a in P3_ARMS}

print(f"{P3_SEED_REPLICATES} independent replays of the same 1,440 slots:")
print(f"{'arm':<9} {'budget share mean':>18} {'sd':>8} {'min':>8} {'max':>8} "
      f"{'won the run':>12} {'full-info rate':>15}")
for arm in P3_ARMS:
    s = p3_share[arm]
    fa, fb = p3_ab[arm]
    print(f"{arm:<9} {s.mean():>17.1%} {s.std():>8.3f} {s.min():>7.1%} {s.max():>7.1%} "
          f"{p3_wins[arm] / P3_SEED_REPLICATES:>11.1%} {fa / (fa + fb):>14.4f}")

verdict(True, f"Path 3 engine: {p3_total_pulls:,} pulls replayed, budget concentrates on "
              f"{max(P3_ARMS, key=lambda a: p3_share[a].mean())} "
              f"({p3_share[max(P3_ARMS, key=lambda a: p3_share[a].mean())].mean():.1%} mean "
              f"share over {P3_SEED_REPLICATES} seeds) -- gate system bypassed, no gate is run "
              f"on this path")

200 independent replays of the same 1,440 slots:
arm        budget share mean       sd      min      max  won the run  full-info rate
BTCUSDT               20.3%    0.079    1.2%   38.1%        5.5%         0.3636
ETHUSDT               37.9%    0.084   13.4%   59.5%       60.0%         0.4025
SOLUSDT               41.8%    0.088   14.0%   71.9%       34.5%         0.4111
APPLIES  -- Path 3 engine: 1,440 pulls replayed, budget concentrates on SOLUSDT (41.8% mean share over 200 seeds) -- gate system bypassed, no gate is run on this path


True

### What Path 3 actually learned here

The engine ran, the arithmetic is the framework's, and the answer is not a trading edge.

* The arm ordering under the strict reward is close to an inverse ordering of **flat share**.
  Up and down are near-symmetric on every symbol in this extract, so `up ≈ (1 − flat)/2` and
  the arm that wins is the arm whose 5-second bars least often close exactly where they
  opened. That is a function of tick size and quote density. Swap the reward to `close >= prev`
  and the ranking inverts, because the flat class changes hands.
* The full-information rates are all in the high-30s to low-40s percent, i.e. **below 50%** —
  a strict up-tick is a minority event by construction when flats exist. The bandit is
  choosing between three losing propositions and finding the least-losing one.
* With γ = 0.97 the posterior never gets narrow (α + β saturates near 33, against 1,441 for
  the undecayed count), so the selection stays seed-dependent. The single seeded run and the
  200-seed summary disagree about *how* they disagree: the seeded run's final argmax is
  ETHUSDT, ETHUSDT holds the final argmax in 60% of runs, and yet SOLUSDT takes the largest
  mean budget share. Argmax-of-final-posterior is a snapshot of the last ~33 pulls;
  budget share is the whole trajectory. Reporting either alone as *the* winner is reporting
  noise, and reporting them together is the honest version.
* What the engine does get right is the arm it *avoids*: BTCUSDT has the lowest reward rate
  and takes the smallest budget in every summary. That is the bandit working — on a reward
  that measures flatness.

What the section does establish is the refinery's own claim: the three bans are load-bearing.
The variance filter deleted the rarest tokens on a real fit; the L1 path zeroed the boundary
token on a real fit while that same token carried a mined lift ratio; and standardising the
α/β counts produced negative shape parameters that `np.random.beta` refused outright. Each
was measured, not asserted.